# Harness bench: the skills harness alone, on evaluation parents and their children, on four GPU types

**Why.** Submissions 3–6 scored 40.7–43.5, flat, because all four were in effect direct runs: sub 5 ran the harness only after a direct attempt 1, and sub 6's coverage guard most likely latched in its first hour and sent ~75–90 % of attempt 1s direct (`submissions/SUMMARY.md`). So the harness's real strength has never been measured on its own. From 2026-10-18 the work focuses on harness-only solutions, and this notebook measures that baseline first.

**What runs.** Submission 6's attempt 1, unchanged: Qwen3.8-27B-FP8 at `xhigh`, with two tools:
- `python`, the task preloaded as numpy arrays and the skills toolkit (`summary()` first, object, layout and search helpers);
- `submit`, which runs `transform(grid)` on every training pair. A program that reproduces all of them is accepted, its test output is the answer, and the model is then asked for a different rule that also fits, whose output is the second answer.

Each attempt has 131,072 tokens over all its turns. There is **no direct attempt, no coverage guard and no time cap that cuts attempts**: every attempt ends by its own budget, an answer, or giving up.

**On what.** A frozen set of 24 tasks, 32 test outputs (`bench24_*.json`):
- **8 evaluation parents:** #1–#6 of slice 1 and #11–#12 of slice 2 in sorted-ID order (`0934a4d8`, `135a2760`, `136b0064`, `13e47133`, `142ca369`, `16b78196`, `20270e3b`, `20a9e565`), 12 outputs. None was in the harness's dev or held-out rounds (`0934a4d8` was dropped from held-out24 before that round ran).
- **Their 16 eval children:** two new ARC-AGI-2-level puzzles per parent, hand-crafted and accepted only when independent model families reproduced the key (`data/eval_children/`), 20 outputs. They are for building and validating harnesses, never for training.

**Two questions:**
1. **Parents against children.** How does the harness do on evaluation tasks against new puzzles made from them? If the children behave like their parents, children for the whole evaluation set become a validation split the harness has not been tuned on.
2. **The GPU.** Does the harness behave the same on faster or cheaper GPUs (H100, B200, RTX PRO 6000) as on Kaggle's 4 × L4? If it does, harness-effectiveness work can move off L4. Speed and interactivity are not the question here: the solve rate, tokens, turns and failure modes are.

**How it runs.** On Modal, `agi2_modal_harness/comm/harness_nb.py` starts the server from a per-GPU config, then runs this notebook's own cells: §1–§6, then one round cell of §7. Only paths and the request count change between machines. The notebook is not run on Kaggle: it has no install cells, and the children are local data.

**Trajectory:**
- 2026-10-18 20:30 IST: built from submission 6's harness cells (`docs/reports/1018_2017_harness_bench/build_notebook.py`). The user chose one attempt per output for the 4 × L4 baseline, added parent #6's family (graduated after the set was first named), and asked for the other three GPU types only after the baseline.
- 2026-10-18 20:45 IST: no time cutoff in any round (user: let every puzzle run to its own end and see how long each GPU takes). The round cell's 300-minute stop and the box's 4 h are only safety nets, far beyond round 1's projected ~100 min.
- 2026-10-18 21:37 IST: the RTX PRO 6000 moved ahead of the baseline's end (user). Its server is stock SGLang with the flags that suit sm_120, and none of the extreme compute customizations made for 4 × L4 (G5 GEMMs, tiled weights, int8 sm_reduce, FP8 lm_head, sharded verify). A flag A/B on real xhigh contexts (`1018_2137_rtxpro6000-ab`) chose it: sm_120's own defaults, FlashInfer attention with the CUTLASS FP8 GEMM, are the fastest (815 tok/s at 10 concurrent; Triton attention −73 %), and the generic spec-decode patches (GDN fold, windowed draft KV, block verify) cost nothing in speed and add 30 % to the KV pool, so they stay.
- 2026-10-18 22:14 IST: Modal restarted the 4 × L4 container with 31 of 32 attempts finished; parent `20a9e565` test 1 (33 min, 98.8k tokens, no submit) died with it. The 31 records survived on the cache Volume. Cell §7 `hbr1tail` reruns that one output as a fresh attempt (22:22), so round 1 covers all 32.
- 2026-10-18 22:16 IST: round 2 on one RTX PRO 6000 (`hbr2rtx`): build `.15` sessions, the A/B's flags, 18 requests in flight as on L4.
- 2026-10-18 23:00 IST: round 1 complete, 32 of 32 (the restart's lost output rerun): 24 right, parents 7 of 12, children 17 of 20. The 4 × L4 box released.
- 2026-10-18 23:16 IST: round 2 (RTX PRO 6000) complete in 57 min: 19 right, parents 5 of 12, children 14 of 20, and 12 spent budgets against round 1's 2 (paired: 9 outputs from not spent to spent, none back, p = 0.004). The RTX server makes the model think longer in rare very long turns, so GPU rounds do not measure the harness alone. The H100 and B200 rounds wait for a test of the mechanism (the user's call). The RTX box released.
- 2026-10-18 23:36 IST: H100 next (the user: "start H100", chosen over first testing the RTX's thinking-length mechanism). A flag A/B on one H100 (`1018_2336_h100-ab`): sm_90's own defaults (FA3 attention, DeepGEMM FP8 GEMMs), FlashInfer, and the generic patches on each; round 3 (§7 `hbr3h100`) runs the A/B's choice. Stock: 1,319 / 928 / 223 tok/s at 10 / 6 / 1 concurrent, 1.6–1.7× the RTX. The windowed draft KV needs a FlashInfer draft (`dflash_worker_v2.py:645-653`), so it cannot ride on FA3 alone.
- 2026-10-18 23:4x IST: the user asked whether changed children explain the RTX's lower score. No: both rounds read the same hash-checked frozen copy, all 16 bench children are identical to the live set, none is among the children revised tonight, and the parents (evaluation tasks) dropped too.

## 1. Settings

In [ ]:
import asyncio, glob, json, os, re, time

T0 = time.time()
SLOTS = 18                # requests in flight: 18 on 4 x L4 (the server's decode graphs); set per GPU type
SESSIONS = True           # streaming sessions hold the harness's conversations (section 5)
HARNESS_TOKENS = 131072   # an attempt's budget over all its turns, as submission 6's attempt 1
END_SPENT = True          # a spent budget ends the attempt; its best partial submission is its answer (submission 6)
GATE_HIGH, GATE_ROOM = 0.82, 0.05   # the KV gate: park above HIGH; bring back and admit under HIGH - ROOM
ROUND_MINUTES = 300       # a safety stop only: every attempt is meant to end by its own budget

## 2. The benchmark set

The frozen set: 8 evaluation parents and their 16 children, with the solutions for scoring. The harness sees only each task's training pairs and test input (`Sandbox` and the prompt below); the solutions stay in this process. Outputs run family by family (a parent, then its two children), so a partial round always holds whole families.

In [ ]:
def check(ok, problem):
    if not ok:
        raise RuntimeError(problem)

BENCH = os.path.dirname(glob.glob('/kaggle/input/**/bench24_challenges.json', recursive=True)[0])
challenges = json.load(open(f'{BENCH}/bench24_challenges.json'))
solutions = json.load(open(f'{BENCH}/bench24_solutions.json'))
families = json.load(open(f'{BENCH}/bench24_families.json'))
PARENTS = families['parents']
FAMILY = {c: p for p in PARENTS for c in [p, *families['children'][p]]}   # task -> its parent
KIND = {t: 'parent' if t == p else 'child' for t, p in FAMILY.items()}
OUTPUTS = [(t, i) for p in PARENTS for t in [p, *families['children'][p]] for i in range(len(challenges[t]['test']))]
todo = challenges   # the harness cells take their first task from `todo` for their checks
check(len(challenges) == 24 and len(OUTPUTS) == 32, f'expected 24 tasks and 32 outputs: {len(challenges)}, {len(OUTPUTS)}')
check(set(solutions) == set(challenges) and set(FAMILY) == set(challenges), 'the set files do not match')
print(BENCH, '|', len(PARENTS), 'parents,', len(FAMILY) - len(PARENTS), 'children,', len(OUTPUTS), 'test outputs:',
      sum(KIND[t] == 'parent' for t, _ in OUTPUTS), 'parent,', sum(KIND[t] == 'child' for t, _ in OUTPUTS), 'child')

!nvidia-smi --query-gpu=index,name,memory.total,driver_version --format=csv
!echo "$(nproc) CPUs" && free -g | head -2

## 3. The server

On Modal the runner starts it before these cells, from the GPU type's config (`agi2_modal_harness/workloads/sglang_bench/configs/<gpu>_harness.toml`):
- **4 × L4:** submission 6's server: build `.15` (`v0.5.21-agi2.14` plus the `.14 → .15` diff: session retract, evict and park, the vision budget, the DFlash sampler guards) with submission 4's switches, tensor parallel over the four GPUs, 82 % memory, 18 decode slots, 24 requests, FP8 KV, DFlash2 drafting 8 tokens, park at 16 GB of host RAM per GPU.
- **Other GPUs:** the same build and model on one GPU, without the L4-only kernels and the all-reduce.

These cells wait for it, define the restart the harness's `ServerGuard` uses after a crash, and check the chat template's `xhigh` line.

In [ ]:
import re, subprocess, requests

SERVER = 'http://127.0.0.1:30000'
log_pos = 0

def show_server_log():
    """Print the server log written since the last call; GPUs 1-3 repeat GPU 0's lines, so only GPU 0's are shown."""
    global log_pos
    with open('sglang.log', 'rb') as f:
        f.seek(log_pos)
        text = f.read().decode(errors='replace')
        log_pos = f.tell()
    for line in text.splitlines():
        if not re.search(r' TP[1-3]\]', line):
            print(line, flush=True)

t = time.time()
SERVER_UP = False
while time.time() - t < 1500:
    show_server_log()
    try:
        SERVER_UP = requests.get(f'{SERVER}/health', timeout=5).status_code == 200
    except requests.RequestException:
        pass
    if SERVER_UP or subprocess.run(['pgrep', '-f', 'sglang.launch_server'], capture_output=True).returncode:
        break
    time.sleep(5)
show_server_log()
check(SERVER_UP, 'the server did not come up (its log is above)')
print(f'server up: {SERVER_UP} ({time.time() - t:.0f} s in this cell)')

In [ ]:
def gpu_mem():
    return get_ipython().getoutput('nvidia-smi --query-gpu=memory.used --format=csv,noheader,nounits')


def restart_server(timeout=900):
    """The server died mid-round (a CUDA OOM, a crash): stop what is left of it, start it again with the same
    switches (launch_server.sh keeps the old log) and wait for /health. Every harness conversation then rebuilds its
    session from its own history (ServerGuard, section 9)."""
    global log_pos
    subprocess.run(['pkill', '-f', 'sglang.launch_server'])
    t = time.time()
    while time.time() - t < 120 and max(int(m) for m in gpu_mem()) > 2000:   # MiB: until the old processes let go
        time.sleep(2)
    log_pos = 0   # show_server_log reads the new sglang.log from its start
    subprocess.run(['bash', 'launch_server.sh'], check=True)
    while time.time() - t < timeout:
        try:
            if requests.get(f'{SERVER}/health', timeout=5).status_code == 200:
                return True
        except requests.RequestException:
            pass
        time.sleep(5)
    return False

In [ ]:
from tokenizers import Tokenizer

if SERVER_UP:
    MODEL_DIR = os.path.dirname([p for p in glob.glob('/kaggle/input/models/**/config.json', recursive=True) if 'fp8' in p][0])
    r = requests.post(f'{SERVER}/v1/chat/completions', timeout=300, json={
        'model': 'qwen3.8-27b', 'messages': [{'role': 'user', 'content': 'ping'}], 'max_tokens': 1,
        'chat_template_kwargs': {'reasoning_effort': 'xhigh'}, 'return_input_ids_in_sglext': True}).json()
    rendered = Tokenizer.from_file(f'{MODEL_DIR}/tokenizer.json').decode(r['sglext']['input_ids'], skip_special_tokens=False)
    print(rendered)
    check('Reasoning effort is set to xhigh' in rendered, 'the xhigh effort line is missing')
    PREFIX, SUFFIX = rendered.split('ping', 1)

## 4. The ARC prompt and answer parsing

ARC Prize's prompt and parser, as every submission used them; the harness puts its tools where this prompt asks for the answer format.

In [ ]:
PROMPT = (
    "You are participating in a puzzle solving competition. You are an expert at solving puzzles.\n\n"
    "Below is a list of input and output pairs with a pattern. Your goal is to identify the pattern or "
    "transformation in the training examples that maps the input to the output, then apply that pattern "
    "to the test input to give a final output.\n\n"
    "Respond in the format of the training output examples\n\n"
    "--Training Examples--\n{training_examples}\n--End of Training Examples--\n\n"
    "--Test Input--\n{test_input}\n--End of Test Input--\n\n"
    "Your response:")

def build_prompt(task, test_input, template=PROMPT):
    examples = ''.join(f'--Example {i}-- \n\n INPUT: \n\n{json.dumps(p["input"])}\n\n'
                       f'OUTPUT: \n\n{json.dumps(p["output"])}\n\n' for i, p in enumerate(task['train']))
    return template.format(training_examples=examples, test_input=json.dumps(test_input))

def is_grid(g):
    return isinstance(g, list) and len(g) > 0 and all(isinstance(r, list) for r in g)

def parse_grid(text):
    m = re.search(r'\\boxed\{(.*?)\}', text, re.DOTALL)
    if m:
        try:
            g = json.loads(m.group(1).strip())
            if is_grid(g):
                return g
        except json.JSONDecodeError:
            pass
    end = max(text.rfind(']'), text.rfind('}'))  # the last bracketed JSON value, scanned back to its opening
    if end < 0:
        return None
    close = text[end]
    opening, depth = '[' if close == ']' else '{', 1
    for start in range(end - 1, -1, -1):
        depth += (text[start] == close) - (text[start] == opening)
        if depth == 0:
            try:
                g = json.loads(text[start:end + 1])
            except json.JSONDecodeError:
                return None
            return g if is_grid(g) else None
    return None

def kaggle_grid(g):
    return (is_grid(g) and 1 <= len(g) <= 30 and 1 <= len(g[0]) <= 30
            and all(len(r) == len(g[0]) and all(type(v) is int and 0 <= v <= 9 for v in r) for r in g))

SAMPLING = dict(temperature=1.0, top_p=0.95, top_k=20, min_p=0.0)   # the model card's thinking defaults

## 5. The harness: submission 6's attempt 1, unchanged

The harness of submission 5, with what the probe of 2026-10-17 added:
- **The skills toolkit** (`arc_skills.py`, below): `summary()` (exact per-pair facts), `match`, `objects` with holes and shape keys, `separators`/`panels`, `symmetries`, `period`, fills, `try_all` (many ideas at once), `try_library` (whole-task transforms, exact fits only) and `recipe()` (how to investigate; it names no likely rules). Failing submissions also say, object by object, what is missing, extra, misplaced or recoloured. The model calls `summary()` first in nearly every attempt.
- **Grading from the worker's own copy of the task**, so a python call that rebinds `test` or edits a grid cannot change what `submit` grades (3 of 29 attempts hit this in submission 5's worker).
- **Crash recovery** (`ServerGuard`): a server that dies mid-run is restarted once with the same switches, and every conversation continues from its own history.
- **Session park** behind the KV gate (below).

Three changes since the probe, from its held-out round:
- **A spent budget ends the attempt.** The old last turn asked for an answer with thinking off, after re-reading the whole ~150k-token history; it solved 0 of 25 such cases in our records. The attempt keeps its best partial submission as its answer (in submission 6, a direct attempt followed; not here).
- **A tool call written without its `<tool_call>` opener is still a call** (one held-out attempt was lost when the harness took such a turn as the end).
- **Pictures are off,** and so are the overfit flags: neither helped in the probe.

The sandbox: one worker process per attempt, the task preloaded as numpy arrays, state kept between calls, 60 s per call (then a restart), 4 GB of memory, 12,000 characters of output.

In [ ]:
%%writefile arc_sandbox_worker.py
"""The Python the model's tool calls run in: one worker process per attempt, state kept between calls.

Reads one JSON request per line on stdin, answers one JSON line on stdout:
  {"op": "init", "task": {...}}    load the task: training pairs and test inputs as numpy arrays
  {"op": "exec", "code": "..."}    run code in the shared namespace, return what it printed (and any error)
  {"op": "submit", "code": "..."}  run code, then check `transform` on every training pair and apply it to the tests
"""
import contextlib
import io
import json
import sys
import time
import traceback

import numpy as np

np.set_printoptions(legacy="1.25")   # printed numbers stay plain: (1, 16), not (np.int64(1), np.int64(16))
NS = {}
TASK = {}   # the task as the harness keeps it (set by init). Grading, check() and the skills read this, never the
            # model's namespace, where `test = test_inputs[0]` or an in-place edit would change what gets graded
PICS = []   # pictures asked for by show_image during the current call: lists of grids
SKILLS_ON = False   # set by init: the skills arm (arc_skills.py in the namespace, object-level submit feedback)


PIC_MAX_CELLS, PIC_MAX_PER_ATTEMPT = 1800, 8   # every cell is one visual token; the server encodes pictures together
PIC_COUNT = 0   # pictures this attempt has shown (init resets it)


def show_image(*grids):
    """Show up to 4 grids (each at most 30x30, 1,800 cells together) to yourself as one picture, side by side; it
    comes back with this call's output. An attempt can show 8 pictures."""
    global PIC_COUNT
    arrs = [np.asarray(g) for g in grids[:4]]
    if not arrs or len(PICS) >= 3:
        print("[show_image: nothing shown (no grids, or 3 pictures already in this call)]")
        return
    bad = [str(a.shape) for a in arrs if a.ndim != 2 or not (1 <= a.shape[0] <= 30 and 1 <= a.shape[1] <= 30)]
    if bad:
        print(f"[show_image: nothing shown: each grid must be 1-30 x 1-30 cells, got {', '.join(bad)}. Every cell is "
              "already drawn large, so there is no need to magnify]")
        return
    cells = sum(a.size for a in arrs)
    if cells > PIC_MAX_CELLS:
        print(f"[show_image: nothing shown: {cells} cells; one picture holds at most {PIC_MAX_CELLS}, so show fewer grids]")
        return
    if PIC_COUNT >= PIC_MAX_PER_ATTEMPT:
        print(f"[show_image: nothing shown: this attempt has used its {PIC_MAX_PER_ATTEMPT} pictures]")
        return
    PIC_COUNT += 1
    gs = [a.astype(int).tolist() for a in arrs]
    PICS.append(gs)
    print(f"[picture {len(PICS)}: " + ", ".join(f"{len(g)}x{len(g[0])}" for g in gs) + "]")


def show(grid):
    """Print a grid as digit rows with row and column indices."""
    g = np.asarray(grid)
    print("    " + "".join(str(c % 10) for c in range(g.shape[1])) + f"   ({g.shape[0]}x{g.shape[1]})")
    for r, row in enumerate(g):
        print(f"{r:>3} " + "".join(str(int(v)) for v in row))


def diff(a, b):
    """Cells where two same-shape grids differ, as (row, col, a, b)."""
    a, b = np.asarray(a), np.asarray(b)
    if a.shape != b.shape:
        return f"shapes differ: {a.shape} vs {b.shape}"
    return [(int(r), int(c), int(a[r, c]), int(b[r, c])) for r, c in np.argwhere(a != b)]


def objects(grid, bg=None, diagonal=False, multicolor=False):
    """Connected components: dicts with color (or colors), cells, bbox (r0, c0, r1, c1), size, and shape (the
    object's own cells cropped to its bbox, other cells -1). bg defaults to the most common colour."""
    g = np.asarray(grid)
    if bg is None:
        vals, counts = np.unique(g, return_counts=True)
        bg = int(vals[np.argmax(counts)])
    seen = np.zeros(g.shape, bool)
    steps = [(1, 0), (-1, 0), (0, 1), (0, -1)] + ([(1, 1), (1, -1), (-1, 1), (-1, -1)] if diagonal else [])
    out = []
    for r0 in range(g.shape[0]):
        for c0 in range(g.shape[1]):
            if seen[r0, c0] or g[r0, c0] == bg:
                continue
            stack, cells = [(r0, c0)], []
            seen[r0, c0] = True
            while stack:
                r, c = stack.pop()
                cells.append((r, c))
                for dr, dc in steps:
                    rr, cc = r + dr, c + dc
                    if (0 <= rr < g.shape[0] and 0 <= cc < g.shape[1] and not seen[rr, cc] and g[rr, cc] != bg
                            and (multicolor or g[rr, cc] == g[r0, c0])):
                        seen[rr, cc] = True
                        stack.append((rr, cc))
            rs, cs = [p[0] for p in cells], [p[1] for p in cells]
            box = (min(rs), min(cs), max(rs), max(cs))
            shape = np.full((box[2] - box[0] + 1, box[3] - box[1] + 1), -1, int)
            for r, c in cells:
                shape[r - box[0], c - box[1]] = g[r, c]
            colors = sorted({int(g[r, c]) for r, c in cells})
            out.append({"color": colors[0] if len(colors) == 1 else colors, "cells": sorted(cells), "bbox": box,
                        "size": len(cells), "shape": shape})
    return out


def _grid_text(g):
    return "\n".join("".join(str(int(v)) for v in row) for row in np.asarray(g))


def _diff_text(expected, got, max_cells=900):
    """The expected grid, and the program's output with correct cells shown as '.'."""
    e, g = np.asarray(expected), np.asarray(got)
    if e.shape != g.shape:
        return (f"Wrong output SIZE: expected {e.shape[0]}x{e.shape[1]}, your program returned "
                f"{g.shape[0]}x{g.shape[1] if g.ndim == 2 else '?'}.\nExpected:\n{_grid_text(e)}")
    wrong = int((e != g).sum())
    if e.size > max_cells:
        return f"{wrong} of {e.size} cells are wrong (grid too large to print the diff)."
    marked = "\n".join("".join("." if e[r, c] == g[r, c] else str(int(g[r, c])) for c in range(e.shape[1]))
                       for r in range(e.shape[0]))
    return (f"{wrong} of {e.size} cells are wrong.\nExpected ({e.shape[0]}x{e.shape[1]}):\n{_grid_text(e)}\n"
            f"Your output, with CORRECT cells shown as '.':\n{marked}")


def _as_grid(x):
    a = np.asarray(x)
    if a.ndim != 2 or not (1 <= a.shape[0] <= 30 and 1 <= a.shape[1] <= 30):
        raise ValueError(f"not a 1-30 x 1-30 grid (got shape {a.shape})")
    if a.dtype.kind == "f" and np.isfinite(a).all() and (a == np.round(a)).all():
        a = a.astype(int)  # whole-number floats (np.zeros((h, w)) is float) are unambiguous
    if a.dtype.kind not in "iub":
        raise ValueError(f"cells must be integers (got dtype {a.dtype})")
    if a.min() < 0 or a.max() > 9:
        raise ValueError("cell values must be 0-9")
    return a.astype(int)


def check(fn, verbose=True):
    """Run fn on every training input; report per pair whether it matches exactly and which cells differ."""
    ok = 0
    for i, (x, y) in enumerate(TASK["train"]):
        try:
            got = _as_grid(fn(x.copy()))
        except Exception as e:  # noqa: BLE001 - report any error the model's code raises
            if verbose:
                print(f"Example {i + 1}: ERROR {type(e).__name__}: {e}")
            continue
        if got.shape == y.shape and (got == y).all():
            ok += 1
            if verbose:
                print(f"Example {i + 1}: OK")
        elif verbose:
            print(f"Example {i + 1}: WRONG\n{_diff_text(y, got)}")
    if verbose:
        print(f"{ok} of {len(TASK['train'])} examples reproduced")
    return ok == len(TASK["train"])


D4 = {"rotated 90 degrees": lambda a: np.rot90(a, 1), "rotated 180 degrees": lambda a: np.rot90(a, 2),
      "rotated 270 degrees": lambda a: np.rot90(a, 3), "flipped upside down": np.flipud,
      "flipped left-right": np.fliplr, "transposed": lambda a: a.T, "anti-transposed": lambda a: np.rot90(a, 2).T}


def _rules_broken(train, x, y):
    """Shape and colour rules every training pair obeys that the test output `y` (for test input `x`) breaks."""
    out = []
    pairs = [(np.asarray(i), np.asarray(o)) for i, o in train]
    cols = lambda a: set(np.unique(a).tolist())
    if all(o.shape == i.shape for i, o in pairs) and y.shape != x.shape:
        out.append(f"every training output has its input's size, but yours is {y.shape[0]}x{y.shape[1]} for a "
                   f"{x.shape[0]}x{x.shape[1]} input")
    ratios = {(o.shape[0] / i.shape[0], o.shape[1] / i.shape[1]) for i, o in pairs}
    if len(ratios) == 1 and (y.shape[0] / x.shape[0], y.shape[1] / x.shape[1]) != next(iter(ratios)) and not out:
        r = next(iter(ratios))
        out.append(f"every training output is {r[0]:g}x the input's height and {r[1]:g}x its width, but yours is not")
    if all(cols(o) <= cols(i) for i, o in pairs) and not cols(y) <= cols(x):
        out.append(f"no training output uses a colour its input lacks, but yours adds {sorted(cols(y) - cols(x))}")
    new_ok = set().union(*[cols(o) - cols(i) for i, o in pairs])
    if not (cols(y) - cols(x)) <= new_ok and not all(cols(o) <= cols(i) for i, o in pairs):
        out.append(f"yours adds colours {sorted(cols(y) - cols(x) - new_ok)} that no training output adds")
    if all(cols(o) == cols(i) for i, o in pairs) and cols(y) != cols(x):
        out.append(f"every training output uses exactly its input's colours, but yours uses {sorted(cols(y))} for an "
                   f"input with {sorted(cols(x))}")
    if all(o.shape == i.shape for i, o in pairs) and y.shape == x.shape:
        fr = [float((o != i).mean()) for i, o in pairs]
        f = float((y != x).mean())
        if f < 0.5 * min(fr) - 1e-9 or f > 2 * max(fr) + 1e-9:
            out.append(f"the training outputs change {min(fr):.0%}-{max(fr):.0%} of their input's cells, yours "
                       f"{f:.0%}")
    return out


def _overfit_flags(fn, arg, x_test, y_test, budget_s=15.0):
    """Signs that an accepted program fits the training pairs without having the general rule. Returns notes.
    The traced and transformed runs stop after `budget_s` seconds: the whole submit call must stay inside the tool's
    60 s limit, or the acceptance itself would be lost."""
    t0 = time.time()
    train = TASK["train"]
    notes = _rules_broken(train, x_test, np.asarray(y_test))
    # code only the test input runs (frames of the model's own code: <submit> and earlier <tool> calls)
    seen, cur = {}, [None]

    def ltrace(frame, event, a):
        if event == "line":
            seen[cur[0]].add((frame.f_code.co_filename, frame.f_code.co_name, frame.f_lineno))
        return ltrace

    def gtrace(frame, event, a):
        return ltrace if frame.f_code.co_filename in ("<submit>", "<tool>") else None

    def traced(tag, g):
        cur[0], seen[tag] = tag, set()
        sys.settrace(gtrace)
        try:
            fn(arg(g))
        except Exception:  # noqa: BLE001 - only the lines matter here
            pass
        finally:
            sys.settrace(None)

    for k, (x, _) in enumerate(train):
        traced(k, x)
        if time.time() - t0 > budget_s / 2:   # tracing is slow on a heavy program: skip the rest of the checks
            return notes
    traced("test", x_test)
    only = sorted({ln for f, n, ln in seen["test"] - set().union(*[seen[k] for k in range(len(train))]) if f == "<submit>"})
    if only:
        notes.append(f"lines {only} of your program run on the test input but on no training input, so no example "
                     "checks them")

    # symmetries kept on every training input but not on the test input
    def commutes(g, t):
        try:
            a, b = _as_grid(fn(arg(D4[t](g)))), D4[t](_as_grid(fn(arg(g))))
            return a.shape == b.shape and bool((a == b).all())
        except Exception:  # noqa: BLE001
            return False
    broken = []
    for t in D4:
        if time.time() - t0 > budget_s:
            break
        if all(commutes(x, t) for x, _ in train) and not commutes(x_test, t):
            broken.append(t)
    if broken:
        notes.append("on every training input your program gives the same result whether the grid is " + ", ".join(broken)
                     + " first or not, but not on the test input")
    return notes


def _submit(code, lists=False, overfit=False):
    """Run the code, then judge `transform` on every training pair (feedback for at most 2 failing ones) and the tests.
    `transform` gets each grid as a numpy array, or as a list of lists with `lists` (what the gridprog prompt promises).
    With `overfit`, an accepted program also gets the overfit checks (below)."""
    exec(compile(code, "<submit>", "exec"), NS)  # noqa: S102 - this process exists to run it
    arg = (lambda g: g.tolist()) if lists else (lambda g: g.copy())
    fn = NS.get("transform")
    if not callable(fn):
        return {"n_ok": 0, "n": len(TASK["train"]), "feedback": "Your code does not define transform(grid).",
                "test_outputs": None}
    n_ok, lines, fail_pair = 0, [], None
    for i, (x, y) in enumerate(TASK["train"]):
        try:
            got = _as_grid(fn(arg(x)))
            if got.shape == y.shape and (got == y).all():
                n_ok += 1
                continue
            msg = _diff_text(y, got)
            if SKILLS_ON and got.shape == y.shape:   # the skills arm: also what is wrong, object by object
                import arc_skills
                obj = arc_skills.object_diff(y, got)
                if obj:
                    msg += "\nObject by object:\n" + "\n".join("- " + s for s in obj[:6]) + \
                           (f"\n- ... and {len(obj) - 6} more object differences" if len(obj) > 6 else "")
            if fail_pair is None:   # the first failing example: expected and produced, for a picture
                fail_pair = [i + 1, y.tolist(), got.tolist()]
        except Exception as e:  # noqa: BLE001
            msg = f"Error: {type(e).__name__}: {e}"
        if len(lines) < 2:
            lines.append(f"## Example {i + 1} failed\n{msg}")
    tests, test_err = [], None
    for t in TASK["test"]:
        try:
            tests.append(_as_grid(fn(arg(t))).tolist())
        except Exception as e:  # noqa: BLE001
            tests.append(None)
            test_err = f"{type(e).__name__}: {e}"
    n = len(TASK["train"])
    flags = []
    if n_ok == n and test_err is None:
        fb = f"Accepted: reproduces all {n} examples. Its output on the test input is your answer."
        if overfit:
            try:
                flags = _overfit_flags(fn, arg, TASK["test"][0], tests[0])
            except Exception as e:  # noqa: BLE001 - a failed check must not cost the acceptance
                flags = []
                print(f"[harness: the overfit check failed: {type(e).__name__}]")
    elif n_ok == n:
        fb = f"Reproduces all {n} examples but FAILS on a test input: {test_err} -- fix it and submit again."
    else:
        fb = f"Reproduced {n_ok} of {n} examples.\n" + "\n".join(lines) + "\nFix the rule and submit again."
    return {"n_ok": n_ok, "n": n, "feedback": fb, "test_outputs": tests, "fail_pair": fail_pair, "overfit": flags}


def _reply(obj):
    sys.stdout.write(json.dumps(obj) + "\n")
    sys.stdout.flush()


def _err_text(e):
    """The model's own frames only (filenames <tool>/<submit>); a compile error has none, so just its message."""
    tb = e.__traceback__
    while tb is not None and not tb.tb_frame.f_code.co_filename.startswith("<"):
        tb = tb.tb_next
    if tb is None:
        return "".join(traceback.format_exception_only(type(e), e))
    return "".join(traceback.format_exception(type(e), e, tb))


for line in sys.stdin:
    req = json.loads(line)
    if req["op"] == "init":
        t = req["task"]
        tr = [(np.array(p["input"], dtype=int), np.array(p["output"], dtype=int)) for p in t["train"]]
        TASK.clear()
        TASK.update(train=tr, test=[np.array(p["input"], dtype=int) for p in t["test"]])
        PIC_COUNT = 0
        mine = [(x.copy(), y.copy()) for x, y in tr]   # the model's own copies, free to rebind or edit
        NS.clear()
        NS.update(np=np, show=show, diff=diff, objects=objects, check=check, show_image=show_image, train=mine,
                  train_inputs=[x for x, _ in mine], train_outputs=[y for _, y in mine],
                  test_inputs=[x.copy() for x in TASK["test"]])
        NS["test"] = NS["test_inputs"]
        SKILLS_ON = bool(req.get("skills"))
        if SKILLS_ON:   # the skills module (arc_skills.py beside this file), its functions in the namespace; it
            import arc_skills   # reads the task from copies of TASK, like the grader
            arc_skills.NS = {"train": [(x.copy(), y.copy()) for x, y in tr], "test_inputs": [x.copy() for x in TASK["test"]]}
            NS.update({k: getattr(arc_skills, k) for k in arc_skills.SKILL_NAMES})
        _reply({"ok": True})
    elif req["op"] in ("exec", "submit"):
        PICS.clear()
        buf = io.StringIO()
        with contextlib.redirect_stdout(buf), contextlib.redirect_stderr(buf):
            try:
                if req["op"] == "exec":
                    exec(compile(req["code"], "<tool>", "exec"), NS)  # noqa: S102
                    res = {"error": None}
                else:
                    res = _submit(req["code"], req.get("lists", False), req.get("overfit", False))
            except Exception as e:  # noqa: BLE001
                res = {"error": _err_text(e)}
        res["out"] = buf.getvalue()
        res["pictures"] = list(PICS)
        _reply(res)

In [ ]:
%%writefile arc_skills.py
"""ARC skills: tested helpers preloaded in the harness sandbox. The prompt shows only SKILLS_INDEX; `help(name)` prints a
skill's full description with an example. Every function takes and returns numpy int arrays (lists also accepted)."""
from collections import Counter

import numpy as np

# ----------------------------------------------------------------------------------------------------------- basics


def _g(x):
    return np.asarray(x, dtype=int)


def bg(g):
    """The background colour: the most common colour of g."""
    v, c = np.unique(_g(g), return_counts=True)
    return int(v[np.argmax(c)])


D4_NAMES = ("identity", "rot90", "rot180", "rot270", "flip_lr", "flip_ud", "transpose", "anti_transpose")


def d4(g, name):
    """g rotated, flipped or transposed. name: rot90 (counter-clockwise), rot180, rot270, flip_lr (mirror left-right),
    flip_ud (upside down), transpose (main diagonal), anti_transpose (other diagonal), identity."""
    g = _g(g)
    return {"identity": g, "rot90": np.rot90(g, 1), "rot180": np.rot90(g, 2), "rot270": np.rot90(g, 3),
            "flip_lr": np.fliplr(g), "flip_ud": np.flipud(g), "transpose": g.T,
            "anti_transpose": np.rot90(g, 2).T}[name].copy()


def symmetries(g, ignore=None):
    """The D4 names (other than identity) that leave g unchanged. With `ignore` (a colour), cells of that colour
    match anything, e.g. a damaged symmetric pattern whose damage is painted in one colour."""
    g = _g(g)
    out = []
    for n in D4_NAMES[1:]:
        t = d4(g, n)
        if t.shape != g.shape:
            continue
        eq = t == g
        if ignore is not None:
            eq |= (t == ignore) | (g == ignore)
        if eq.all():
            out.append(n)
    return out


def period(g, ignore=None):
    """The smallest tile (rows, cols) that repeats to make g: g[r, c] == g[r % rows, c % cols] everywhere. With `ignore`
    (a colour), cells of that colour match anything. Returns (rows, cols); (H, W) means no repetition."""
    g = _g(g)
    H, W = g.shape
    known = np.ones_like(g, bool) if ignore is None else g != ignore
    best = (H, W)
    for pr in range(1, H + 1):
        for pc in range(1, W + 1):
            if pr * pc >= best[0] * best[1]:
                break
            ok = True
            for r0 in range(pr):
                for c0 in range(pc):
                    vals = g[r0::pr, c0::pc][known[r0::pr, c0::pc]]
                    if len(vals) and (vals != vals[0]).any():
                        ok = False
                        break
                if not ok:
                    break
            if ok:
                best = (pr, pc)
                break
    return best


def fill_by_period(g, ignore):
    """Repaint the cells of colour `ignore` from the smallest repeating tile of the other cells (period with ignore).
    Returns the repaired grid, or g unchanged when some tile cell is never seen."""
    g = _g(g).copy()
    pr, pc = period(g, ignore)
    known = g != ignore
    for r0 in range(pr):
        for c0 in range(pc):
            vals = g[r0::pr, c0::pc][known[r0::pr, c0::pc]]
            if len(vals) == 0:
                return _g(g)
            g[r0::pr, c0::pc] = vals[0]
    return g


def fill_by_symmetry(g, ignore, which=None):
    """Repaint the cells of colour `ignore` from their mirror images under the symmetries `which` (default: those
    symmetries(g, ignore) finds). Cells no mirror covers keep `ignore`."""
    g = _g(g).copy()
    which = which or symmetries(g, ignore)
    for _ in range(3):
        for n in which:
            t = d4(g, n)
            if t.shape != g.shape:
                continue
            m = (g == ignore) & (t != ignore)
            g[m] = t[m]
    return g


# ----------------------------------------------------------------------------------------------------------- objects


def objects(g, bg=None, diagonal=False, multicolor=False):
    """Connected components of non-background cells. Each is a dict: color (or sorted colors with multicolor), cells
    [(r, c)], bbox (r0, c0, r1, c1) inclusive, size, shape (the object cropped to its bbox, other cells -1), holes
    (number of enclosed background regions) and key (a shape signature: same key = same shape and colours, any
    position). bg defaults to the most common colour."""
    g = _g(g)
    if bg is None:
        bg = globals()["bg"](g)
    H, W = g.shape
    seen = np.zeros(g.shape, bool)
    steps = [(1, 0), (-1, 0), (0, 1), (0, -1)] + ([(1, 1), (1, -1), (-1, 1), (-1, -1)] if diagonal else [])
    out = []
    for r0 in range(H):
        for c0 in range(W):
            if seen[r0, c0] or g[r0, c0] == bg:
                continue
            stack, cells = [(r0, c0)], []
            seen[r0, c0] = True
            while stack:
                r, c = stack.pop()
                cells.append((r, c))
                for dr, dc in steps:
                    rr, cc = r + dr, c + dc
                    if (0 <= rr < H and 0 <= cc < W and not seen[rr, cc] and g[rr, cc] != bg
                            and (multicolor or g[rr, cc] == g[r0, c0])):
                        seen[rr, cc] = True
                        stack.append((rr, cc))
            rs, cs = [p[0] for p in cells], [p[1] for p in cells]
            box = (min(rs), min(cs), max(rs), max(cs))
            shape = np.full((box[2] - box[0] + 1, box[3] - box[1] + 1), -1, int)
            for r, c in cells:
                shape[r - box[0], c - box[1]] = g[r, c]
            colors = sorted({int(g[r, c]) for r, c in cells})
            out.append({"color": colors[0] if len(colors) == 1 else colors, "cells": sorted(cells), "bbox": box,
                        "size": len(cells), "shape": shape, "holes": _holes(shape),
                        "key": (shape.shape, tuple(shape.flatten().tolist()))})
    return out


def _holes(shape):
    """Background regions inside an object's bbox (cells -1) that do not touch the bbox border."""
    m = shape == -1
    H, W = m.shape
    seen = np.zeros_like(m)
    n = 0
    for r0 in range(H):
        for c0 in range(W):
            if not m[r0, c0] or seen[r0, c0]:
                continue
            stack, border = [(r0, c0)], False
            seen[r0, c0] = True
            while stack:
                r, c = stack.pop()
                border |= r in (0, H - 1) or c in (0, W - 1)
                for dr, dc in ((1, 0), (-1, 0), (0, 1), (0, -1)):
                    rr, cc = r + dr, c + dc
                    if 0 <= rr < H and 0 <= cc < W and m[rr, cc] and not seen[rr, cc]:
                        seen[rr, cc] = True
                        stack.append((rr, cc))
            n += not border
    return n


def match(a, b, bg=None):
    """Pair the objects of grid a with those of grid b (same-size or not). Returns lines describing each: 'moved by
    (dr, dc)', 'rotated/flipped (name)' (and moved), 'recoloured c1->c2', objects only in a ('gone') or only in b
    ('new'), then 'same' (same place); more than 8 'same' objects become one line. A quick way to see what a training
    pair does to its objects."""
    a, b = _g(a), _g(b)
    oa = objects(a, bg)
    ob = objects(b, bg)
    used, lines, same = set(), [], []
    for i, x in enumerate(oa):
        best, how = None, None
        for j, y in enumerate(ob):
            if j in used:
                continue
            if x["key"] == y["key"]:
                dr, dc = y["bbox"][0] - x["bbox"][0], y["bbox"][1] - x["bbox"][1]
                cand = ("same" if (dr, dc) == (0, 0) else f"moved by ({dr}, {dc})")
                if best is None or (dr, dc) == (0, 0):
                    best, how = j, cand
            elif best is None and x["shape"].shape == y["shape"].shape and ((x["shape"] == -1) == (y["shape"] == -1)).all() \
                    and x["bbox"] == y["bbox"]:
                best, how = j, f"recoloured {x['color']}->{y['color']}"
            elif best is None and x["size"] == y["size"] and x["size"] > 1:
                turned = [n for n in D4_NAMES[1:] if d4(x["shape"], n).shape == y["shape"].shape
                          and (d4(x["shape"], n) == y["shape"]).all()]
                if turned:
                    best, how = j, f"{turned[0]} and moved to bbox {y['bbox']}"
        if best is None:
            lines.append(f"gone: colour {x['color']} size {x['size']} at bbox {x['bbox']}")
        else:
            used.add(best)
            (same if how == "same" else lines).append(f"{how}: colour {x['color']} size {x['size']} at bbox {x['bbox']}")
    lines.sort(key=lambda s: s.startswith("gone"))   # what changed first; the 12,000-character cut keeps the head
    for j, y in enumerate(ob):
        if j not in used:
            lines.append(f"new: colour {y['color']} size {y['size']} at bbox {y['bbox']}")
    if len(same) > 8:
        counts = Counter(int(s.split("colour ")[1].split()[0]) for s in same)
        same = [f"same place: {len(same)} objects (" + ", ".join(f"colour {c} x{n}" for c, n in sorted(counts.items())) + ")"]
    return lines + same


# ----------------------------------------------------------------------------------------------------------- layout


def separators(g):
    """Full rows and full columns of one non-background colour: (rows, cols, colour) for the most common such colour,
    or ([], [], None). Lines of a colour found nowhere else usually border panels; a full line that is part of a
    larger shape usually does not."""
    g = _g(g)
    H, W = g.shape
    b = bg(g)
    rows = {r: int(g[r, 0]) for r in range(H) if (g[r] == g[r, 0]).all() and g[r, 0] != b}
    cols = {c: int(g[0, c]) for c in range(W) if (g[:, c] == g[0, c]).all() and g[0, c] != b}
    if not rows and not cols:
        return [], [], None
    colour = Counter(list(rows.values()) + list(cols.values())).most_common(1)[0][0]
    return (sorted(r for r, v in rows.items() if v == colour), sorted(c for c, v in cols.items() if v == colour),
            colour)


def panels(g):
    """Split g by its separator lines (see separators) into panels: a list of (r0, c0, panel grid), row-major. Without
    separators, [(0, 0, g)]."""
    g = _g(g)
    rs, cs, colour = separators(g)
    if colour is None:
        return [(0, 0, g.copy())]
    out = []
    rb = [-1] + rs + [g.shape[0]]
    cb = [-1] + cs + [g.shape[1]]
    for i in range(len(rb) - 1):
        for j in range(len(cb) - 1):
            r0, r1, c0, c1 = rb[i] + 1, rb[i + 1], cb[j] + 1, cb[j + 1]
            if r1 > r0 and c1 > c0:
                out.append((r0, c0, g[r0:r1, c0:c1].copy()))
    return out


def bbox(g, colours=None, bg=None):
    """(r0, c0, r1, c1), inclusive, of the cells whose colour is in `colours` (default: every non-background cell);
    None if there are none."""
    g = _g(g)
    m = (g != (globals()["bg"](g) if bg is None else bg)) if colours is None else np.isin(g, list(colours))
    if not m.any():
        return None
    rs, cs = np.where(m)
    return int(rs.min()), int(cs.min()), int(rs.max()), int(cs.max())


def crop(g, box):
    """g[r0..r1, c0..c1] for an inclusive box (r0, c0, r1, c1)."""
    r0, c0, r1, c1 = box
    return _g(g)[r0:r1 + 1, c0:c1 + 1].copy()


def find(g, sub):
    """Every (r, c) where grid `sub` appears exactly inside g."""
    g, sub = _g(g), _g(sub)
    H, W = g.shape
    h, w = sub.shape
    return [(r, c) for r in range(H - h + 1) for c in range(W - w + 1) if (g[r:r + h, c:c + w] == sub).all()]


# ----------------------------------------------------------------------------------------------------------- painting


def scale(g, k):
    """Each cell becomes a k x k block (k may be a pair (kr, kc))."""
    kr, kc = (k, k) if np.isscalar(k) else k
    return np.kron(_g(g), np.ones((kr, kc), int))


def downscale(g, k):
    """The inverse of scale: one cell per k x k block (k may be a pair); None if a block is not one colour."""
    g = _g(g)
    kr, kc = (k, k) if np.isscalar(k) else k
    H, W = g.shape
    if H % kr or W % kc:
        return None
    blocks = g.reshape(H // kr, kr, W // kc, kc)
    first = blocks[:, :1, :, :1]
    if (blocks != first).any():
        return None
    return blocks[:, 0, :, 0].copy()


def tile(g, rows, cols):
    """g repeated rows x cols times."""
    return np.tile(_g(g), (rows, cols))


def recolor(g, mapping):
    """Every colour a in `mapping` (a dict a -> b) repainted b; other colours kept."""
    g = _g(g)
    out = g.copy()
    for a, b in mapping.items():
        out[g == a] = b
    return out


def flood(g, r, c, colour, diagonal=False):
    """Flood-fill the region of g[r, c]'s colour that contains (r, c) with `colour`."""
    g = _g(g).copy()
    old = g[r, c]
    if old == colour:
        return g
    stack = [(r, c)]
    steps = [(1, 0), (-1, 0), (0, 1), (0, -1)] + ([(1, 1), (1, -1), (-1, 1), (-1, -1)] if diagonal else [])
    while stack:
        y, x = stack.pop()
        if 0 <= y < g.shape[0] and 0 <= x < g.shape[1] and g[y, x] == old:
            g[y, x] = colour
            stack.extend((y + dy, x + dx) for dy, dx in steps)
    return g


def fill_holes(g, colour, bg=None):
    """Paint every background region that does not touch the grid's border (an enclosed hole) with `colour`."""
    g = _g(g).copy()
    b = globals()["bg"](g) if bg is None else bg
    outside = np.zeros(g.shape, bool)
    H, W = g.shape
    stack = [(r, c) for r in range(H) for c in (0, W - 1)] + [(r, c) for r in (0, H - 1) for c in range(W)]
    while stack:
        r, c = stack.pop()
        if 0 <= r < H and 0 <= c < W and not outside[r, c] and g[r, c] == b:
            outside[r, c] = True
            stack.extend(((r + 1, c), (r - 1, c), (r, c + 1), (r, c - 1)))
    g[(g == b) & ~outside] = colour
    return g


def gravity(g, direction, bg=None):
    """Let every non-background cell fall as far as it can: direction 'down', 'up', 'left' or 'right' (cells keep their
    order along the line)."""
    g = _g(g)
    b = globals()["bg"](g) if bg is None else bg
    k = {"down": 0, "left": 1, "up": 2, "right": 3}[direction]   # np.rot90 turns counter-clockwise: left edge -> bottom
    t = np.rot90(g, k).copy()   # make it fall 'down'
    for c in range(t.shape[1]):
        col = t[:, c]
        cells = col[col != b]
        t[:, c] = np.concatenate([np.full(len(col) - len(cells), b), cells])
    return np.rot90(t, -k).copy()


def draw_line(g, r0, c0, r1, c1, colour):
    """A straight line (horizontal, vertical or 45-degree diagonal) from (r0, c0) to (r1, c1), inclusive."""
    g = _g(g).copy()
    n = max(abs(r1 - r0), abs(c1 - c0))
    dr = (r1 > r0) - (r1 < r0)
    dc = (c1 > c0) - (c1 < c0)
    for i in range(n + 1):
        r, c = r0 + dr * i, c0 + dc * i
        if 0 <= r < g.shape[0] and 0 <= c < g.shape[1]:
            g[r, c] = colour
    return g


# ----------------------------------------------------------------------------------------------------------- summary


def _relations(x, y):
    """How output y relates to input x, as short phrases (exact checks only)."""
    out = []
    if x.shape == y.shape:
        if (x == y).all():
            return ["output = input"]
        for n in D4_NAMES[1:]:
            if d4(x, n).shape == y.shape and (d4(x, n) == y).all():
                out.append(f"output = {n}(input)")
        diff = x != y
        out.append(f"same size; {int(diff.sum())} of {x.size} cells change")
        trans = Counter((int(a), int(b)) for a, b in zip(x[diff], y[diff]))
        out.append("changes: " + ", ".join(f"{a}->{b} x{n}" for (a, b), n in trans.most_common(8)))
    else:
        for n in D4_NAMES:
            t = d4(x, n)
            if t.shape == y.shape and (t == y).all():
                out.append(f"output = {n}(input)")
        if y.shape[0] <= x.shape[0] and y.shape[1] <= x.shape[1]:
            pos = find(x, y)
            if pos:
                out.append(f"output is the input's sub-grid at {pos[:3]}" + (" (and more)" if len(pos) > 3 else ""))
        for k in range(2, 6):
            if y.shape == (x.shape[0] * k, x.shape[1] * k) and (scale(x, k) == y).all():
                out.append(f"output = scale(input, {k})")
            if x.shape == (y.shape[0] * k, y.shape[1] * k) and downscale(x, k) is not None and (downscale(x, k) == y).all():
                out.append(f"output = downscale(input, {k})")
        if y.shape[0] % x.shape[0] == 0 and y.shape[1] % x.shape[1] == 0 and y.shape != x.shape:
            tr, tc = y.shape[0] // x.shape[0], y.shape[1] // x.shape[1]
            blocks = [[y[i * x.shape[0]:(i + 1) * x.shape[0], j * x.shape[1]:(j + 1) * x.shape[1]] for j in range(tc)]
                      for i in range(tr)]
            names = []
            for row in blocks:
                for bl in row:
                    hit = [n for n in D4_NAMES if d4(x, n).shape == bl.shape and (d4(x, n) == bl).all()]
                    names.append(hit[0] if hit else "?")
            if "?" not in names:
                out.append(f"output = {tr}x{tc} tiles of the input: " + ", ".join(names))
        out.append(f"size {x.shape[0]}x{x.shape[1]} -> {y.shape[0]}x{y.shape[1]}")
    return out


def _grid_facts(g, name):
    rs, cs, colour = separators(g)
    objs = objects(g)
    sym = symmetries(g)
    pr, pc = period(g)
    facts = [f"{name}: {g.shape[0]}x{g.shape[1]}, background {bg(g)}, colours "
             + ", ".join(f"{int(v)}:{int(n)}" for v, n in zip(*np.unique(g, return_counts=True)))]
    if colour is not None:   # the lines and how much of their colour they hold; whether they border panels is not a fact
        on = np.zeros(g.shape, bool)
        on[rs, :] = True
        on[:, cs] = True
        n_on, n_all = int((on & (g == colour)).sum()), int((g == colour).sum())
        facts.append(f"  full lines of colour {colour}: rows {rs} cols {cs} ("
                     + (f"all {n_all} of its cells" if n_on == n_all else f"{n_on} of its {n_all} cells")
                     + f"; panels(g) cuts along them into {len(panels(g))} panels)")
    facts.append(f"  {len(objs)} objects (4-connected, one colour), sizes " + str(sorted((o['size'] for o in objs),
                                                                                             reverse=True)[:12]))
    if sym:
        facts.append(f"  symmetric under {sym}")
    rows_rep, cols_rep = pr <= g.shape[0] // 2, pc <= g.shape[1] // 2   # at least two repeats that way
    if rows_rep and cols_rep:
        facts.append(f"  repeats as a tile of {pr}x{pc}")
    elif rows_rep:
        facts.append(f"  rows repeat every {pr}")
    elif cols_rep:
        facts.append(f"  columns repeat every {pc}")
    return facts


def summary(train=None, test=None):
    """Facts about every training pair and the test input(s), all computed exactly: sizes, colour counts, separators
    and panels, objects, symmetries, periods, and how each output relates to its input (identity, flip/rotation,
    sub-grid, scale, tiling, which colours change into which). Use it first to see the task, then check ideas in code."""
    train = NS["train"] if train is None else train
    test = NS["test_inputs"] if test is None else test
    lines = []
    for i, (x, y) in enumerate(train):
        x, y = _g(x), _g(y)
        lines.append(f"Example {i + 1}:")
        lines += ["  " + s for s in _grid_facts(x, "input")]
        lines += ["  " + s for s in _grid_facts(y, "output")]
        lines += ["  " + s for s in _relations(x, y)]
    sizes = {(_g(y).shape) for _, y in train}
    if len(sizes) == 1:
        lines.append(f"All training outputs are {next(iter(sizes))[0]}x{next(iter(sizes))[1]}.")
    if all(_g(x).shape == _g(y).shape for x, y in train):
        lines.append("Every training output has its input's size.")
    for i, t in enumerate(test):
        lines += _grid_facts(_g(t), f"Test input {i + 1}" if len(test) > 1 else "Test input")
    print("\n".join(lines))


# ----------------------------------------------------------------------------------------------------------- search


def try_all(fns):
    """Run each candidate transform (a function grid -> grid, or a dict name -> function) on every training input.
    Prints, per candidate, how many training examples it reproduces exactly and the first error; returns the names
    of the candidates that reproduce all of them. Lets you test many hypotheses in one call."""
    train = NS["train"]
    items = list(fns.items()) if isinstance(fns, dict) else [(getattr(f, "__name__", str(i)), f) for i, f in enumerate(fns)]
    good = []
    for name, f in items:
        ok, err = 0, None
        for x, y in train:
            try:
                got = _g(f(_g(x).copy()))
                ok += got.shape == _g(y).shape and bool((got == _g(y)).all())
            except Exception as e:  # noqa: BLE001
                err = err or f"{type(e).__name__}: {e}"
        print(f"{name}: {ok}/{len(train)}" + (f"  (error: {err})" if err else ""))
        if ok == len(train):
            good.append(name)
    return good


def _largest_object(g):
    """The largest multicolour object, cropped to its bbox, its outside cells in the background colour."""
    objs = objects(g, multicolor=True)
    if not objs:
        return None
    s = max(objs, key=lambda o: o["size"])["shape"]
    return np.where(s == -1, bg(g), s)


def _library():
    """Whole-task transforms with no free parameters, or parameters read from the training pairs."""
    lib = {f"d4 {n}": (lambda n: lambda g: d4(g, n))(n) for n in D4_NAMES[1:]}
    for k in (2, 3, 4):
        lib[f"scale {k}"] = (lambda k: lambda g: scale(g, k))(k)
        lib[f"downscale {k}"] = (lambda k: lambda g: downscale(g, k))(k)
    for r, c in ((1, 2), (2, 1), (2, 2), (3, 3)):
        lib[f"tile {r}x{c}"] = (lambda r, c: lambda g: tile(g, r, c))(r, c)
    lib["mirror right"] = lambda g: np.hstack([_g(g), np.fliplr(_g(g))])
    lib["mirror down"] = lambda g: np.vstack([_g(g), np.flipud(_g(g))])
    lib["mirror 2x2"] = lambda g: np.vstack([np.hstack([_g(g), np.fliplr(_g(g))]),
                                             np.hstack([np.flipud(_g(g)), np.rot90(_g(g), 2)])])
    lib["crop to content"] = lambda g: crop(g, bbox(g))
    lib["largest object"] = _largest_object
    for d in ("down", "up", "left", "right"):
        lib[f"gravity {d}"] = (lambda d: lambda g: gravity(g, d))(d)
    return lib


def try_library():
    """Try a bank of common whole-task transforms (flips and rotations, scaling, tiling, mirroring, cropping to the
    content or the largest object, gravity, filling enclosed holes, colour maps, period and symmetry repair) on every
    training pair. Prints the ones that reproduce all of them, with their output on the test input. Costs no thinking;
    worth one call at the start."""
    fits, n = _library_fits(NS["train"], NS["test_inputs"])
    if not fits:
        print(f"none of {n} library transforms reproduces every training example")
        return []
    for name, outs, err in fits:
        if err:
            print(f"{name}: reproduces every training example but fails on the test input: {err}")
        else:
            print(f"{name}: reproduces every training example; its test output(s): {outs}")
    return [name for name, _, _ in fits]


def _library_fits(train, tests):
    """[(name, test outputs, error)] for the library transforms that reproduce every training pair, and the number
    tried."""
    lib = _library()
    # transforms whose parameters come from the training pairs
    pairs = [(_g(x), _g(y)) for x, y in train]
    cmap = {}
    if all(x.shape == y.shape for x, y in pairs):
        ok = True
        for x, y in pairs:
            for a, b in zip(x.flatten(), y.flatten()):
                if cmap.setdefault(int(a), int(b)) != b:
                    ok = False
        if ok and any(a != b for a, b in cmap.items()):
            lib["colour map " + str(cmap)] = lambda g, m=dict(cmap): recolor(g, m)
        new = set().union(*[set(np.unique(y)) - set(np.unique(x)) for x, y in pairs])
        for colour in new:
            lib[f"fill enclosed holes with {colour}"] = (lambda c: lambda g: fill_holes(g, c))(int(colour))
        gone = set().union(*[set(np.unique(x)) - set(np.unique(y)) for x, y in pairs])
        for colour in gone:
            lib[f"repair colour {colour} by period"] = (lambda c: lambda g: fill_by_period(g, c))(int(colour))
            lib[f"repair colour {colour} by symmetry"] = (lambda c: lambda g: fill_by_symmetry(g, c))(int(colour))
    fits = []
    for name, f in lib.items():
        try:
            if not all((lambda got: got is not None and _g(got).shape == y.shape and (_g(got) == y).all())(f(x.copy()))
                       for x, y in pairs):
                continue
        except Exception:  # noqa: BLE001
            continue
        try:
            fits.append((name, [_g(f(_g(t).copy())).tolist() for t in tests], None))
        except Exception as e:  # noqa: BLE001
            fits.append((name, None, f"{type(e).__name__}: {e}"))
    return fits, len(lib)


# ----------------------------------------------------------------------------------------------------------- feedback


def object_diff(expected, got, bg=None):
    """What a wrong output gets wrong, object by object (same-size grids): objects of the expected output that yours
    misses, has in another place or another colour, and objects yours has that the expected output does not. Returns
    lines; [] when the grids are equal."""
    e, o = _g(expected), _g(got)
    if e.shape != o.shape:
        return [f"wrong size: expected {e.shape[0]}x{e.shape[1]}, yours {o.shape[0]}x{o.shape[1]}"]
    if (e == o).all():
        return []
    b = globals()["bg"](e) if bg is None else bg
    oe, og = objects(e, b), objects(o, b)
    free = list(range(len(og)))
    lines = []
    for x in oe:
        same = [j for j in free if og[j]["key"] == x["key"] and og[j]["bbox"] == x["bbox"]]
        if same:
            free.remove(same[0])
            continue
        moved = sorted((j for j in free if og[j]["key"] == x["key"]),   # the nearest copy of the same shape
                       key=lambda j: abs(og[j]["bbox"][0] - x["bbox"][0]) + abs(og[j]["bbox"][1] - x["bbox"][1]))
        recol = [j for j in free if og[j]["bbox"] == x["bbox"] and og[j]["size"] == x["size"]]
        if moved:
            free.remove(moved[0])
            lines.append(f"misplaced: colour {x['color']} size {x['size']} belongs at bbox {x['bbox']}, yours is at "
                         f"{og[moved[0]]['bbox']}")
        elif recol:
            free.remove(recol[0])
            lines.append(f"wrong colour: the object at bbox {x['bbox']} should be {x['color']}, yours is "
                         f"{og[recol[0]]['color']}")
        else:
            lines.append(f"missing: colour {x['color']} size {x['size']} at bbox {x['bbox']}")
    lines += [f"extra: colour {og[j]['color']} size {og[j]['size']} at bbox {og[j]['bbox']}" for j in free]
    return lines


# ----------------------------------------------------------------------------------------------------------- recipes

# How to investigate with the tools. They name no likely rules: the training examples decide the rule.
RECIPES = {
    "testing ideas": (
        "Write each idea about the rule as a small function and run them together with try_all([...]). Keep only "
        "those that reproduce every training example. If several do and their outputs on the test input differ, the "
        "examples leave the rule open: look for what in the examples decides between them."),
    "objects": (
        "Make a table: for every training pair, each input object (objects(x)) with what happened to it in the "
        "output (match(x, y)) and its measurable properties (size, colour, bbox, holes, key, position). A rule must "
        "explain every row of every pair; test it with try_all before submitting."),
    "panels": (
        "When separators(g) finds lines whose colour appears nowhere else (summary() then says 'all N of its cells'), "
        "split each grid with panels(g) and treat the panels as grids of their own: compare them with each other and "
        "with the output using exact checks (np.array_equal, colour counts, diff, symmetries). Lines whose colour also "
        "appears elsewhere may be part of a shape. Check every observation in code on all pairs before relying on it."),
    "hidden pattern": (
        "When symmetries(g, ignore=c) or period(g, ignore=c) finds more structure with one colour c ignored than "
        "without it, fill_by_symmetry(g, c) and fill_by_period(g, c) repaint c from that structure. Compare the "
        "result with each training output before treating it as the rule."),
    "output size": (
        "When output sizes differ from the inputs or from each other, list each pair's input size, output size and "
        "candidate quantities (bboxes and counts from objects(), panel sizes, colour counts) side by side, and find "
        "a quantity that matches in every pair. A wrong size can never be right, so settle it first."),
    "a failing submission": (
        "submit lists the failing examples with a cell diff and, object by object, what is missing, extra, misplaced "
        "or recoloured. Read which objects are wrong before changing the program; one wrong object usually points to "
        "one wrong step."),
}


def recipe(name=None):
    """How-to guides for investigating a task with the tools (they suggest no rule): recipe() lists them,
    recipe('name') prints one."""
    if name is None:
        print("Recipes: " + "; ".join(RECIPES) + ".\nrecipe('name') prints one.")
        return
    key = name.lower().strip()
    hits = [k for k in RECIPES if key in k]
    if not hits:
        print(f"no recipe {name!r}; recipe() lists them")
        return
    print(f"{hits[0]}:\n{RECIPES[hits[0]]}")


# ----------------------------------------------------------------------------------------------------------- help

SKILL_NAMES = ("summary", "try_library", "try_all", "objects", "match", "separators", "panels", "symmetries", "period",
               "fill_by_period", "fill_by_symmetry", "bg", "bbox", "crop", "find", "d4", "scale", "downscale", "tile",
               "recolor", "flood", "fill_holes", "gravity", "draw_line", "object_diff", "recipe", "help")

SKILLS_INDEX = """Skills (preloaded in `python`; each one is tested; call help('name') for its full description and an example):
- summary()                exact facts about every training pair and the test input: sizes, colours, separators and
                           panels, objects, symmetries, periods, and how each output relates to its input. Start here.
- try_library()            tries ~40 common whole-task transforms on the training pairs and shows any that fit.
- try_all(fns)             runs several of your candidate transforms on every training pair at once.
- objects(g, ...)          connected components: colour, cells, bbox, size, shape, holes, key (shape signature).
- match(a, b)              which objects of a stay, move, change colour, appear or disappear in b.
- separators(g), panels(g) full lines of one colour, and the panels they cut g into.
- symmetries(g, ignore)    which flips/rotations leave g unchanged; period(g, ignore): the smallest repeating tile.
- fill_by_period(g, c), fill_by_symmetry(g, c)   repaint the cells of colour c from the pattern.
- bg(g), bbox(g, colours), crop(g, box), find(g, sub)
- d4(g, name), scale(g, k), downscale(g, k), tile(g, r, c), recolor(g, {a: b}), flood(g, r, c, colour),
  fill_holes(g, colour), gravity(g, direction), draw_line(g, r0, c0, r1, c1, colour)
- object_diff(expected, got) what a wrong output gets wrong, object by object (submit uses it for failing examples).
- recipe()                 how-to guides for investigating with these tools (testing ideas, objects, panels, hidden
                           patterns, output size, a failing submission); recipe('name') prints one."""

_EXAMPLES = {
    "summary": "summary()   # prints e.g. 'Example 1: input: 10x10, background 0, colours 0:88, 4:12 ... output = flip_lr(input)'",
    "try_library": "found = try_library()   # e.g. ['mirror right'] and its test output",
    "try_all": ("def a(g): return np.fliplr(g)\ndef b(g): return np.flipud(g)\ntry_all([a, b])   # 'a: 3/3', 'b: 1/3' -> ['a']"),
    "objects": "for o in objects(train_inputs[0]): print(o['color'], o['size'], o['bbox'], o['holes'])",
    "match": "print('\\n'.join(match(train_inputs[0], train_outputs[0])))",
    "panels": "for r0, c0, p in panels(train_inputs[0]): show(p)",
    "symmetries": "symmetries(train_outputs[0])   # e.g. ['flip_lr', 'rot180']",
    "period": "period(test_inputs[0], ignore=0)   # e.g. (3, 4)",
}


def help(name=None):  # noqa: A001 - the model calls help('name')
    """help() lists the skills; help('name') prints one skill's full description and an example."""
    if name is None:
        print(SKILLS_INDEX)
        return
    f = globals().get(name)
    if not callable(f):
        print(f"no skill named {name!r}; help() lists them")
        return
    import inspect
    print(f"{name}{inspect.signature(f)}\n{inspect.getdoc(f)}")
    if name in _EXAMPLES:
        print("Example:\n" + _EXAMPLES[name])

The parent side, and a check of the sandbox on its own, before any model is involved: a grid shown with indices, a wrong program's feedback, and a runaway loop, plus one `show_image` picture.

In [ ]:
import asyncio, sys

WORKER = os.path.abspath('arc_sandbox_worker.py')
TOOL_TIMEOUT = 60.0     # seconds per tool call
TOOL_OUT = 12000        # characters of tool output the model sees


def _limits():  # in the worker process: cap its memory so a runaway array cannot touch the server
    import resource
    resource.setrlimit(resource.RLIMIT_AS, (4 << 30, 4 << 30))


def clip(text, limit=TOOL_OUT):
    """Keep the head (2/3) and the tail (1/3): the tail holds the conclusion or the error."""
    if len(text) <= limit:
        return text
    head, tail = text[: limit * 2 // 3], text[-limit // 3:]
    return f'{head}\n... [{len(text) - len(head) - len(tail)} characters cut] ...\n{tail}'


class Sandbox:
    """One worker process per attempt; its Python state persists between the model's calls."""

    def __init__(self, task, timeout=TOOL_TIMEOUT, skills=False):
        self.task, self.timeout, self.proc, self.calls, self.timeouts = task, timeout, None, 0, 0
        self.pics = []   # the last call's show_image pictures (lists of grids)
        self.skills = skills   # the skills module's functions in the namespace (arc_skills.py)

    async def start(self):
        self.proc = await asyncio.create_subprocess_exec(
            sys.executable, '-u', WORKER, stdin=asyncio.subprocess.PIPE, stdout=asyncio.subprocess.PIPE,
            stderr=asyncio.subprocess.DEVNULL, preexec_fn=_limits,
            limit=64 << 20)   # a reply is one JSON line, clipped only here (the probe: 64 KiB killed an attempt)
        return await self._ask({'op': 'init', 'task': self.task, 'skills': self.skills})

    async def _ask(self, req):
        self.proc.stdin.write((json.dumps(req) + '\n').encode())
        await self.proc.stdin.drain()
        line = await asyncio.wait_for(self.proc.stdout.readline(), self.timeout)
        if not line:
            raise RuntimeError('sandbox worker exited')
        return json.loads(line)

    async def _call(self, op, code, **extra):
        self.calls += 1
        try:
            return await self._ask({'op': op, 'code': code, **extra})
        except (asyncio.TimeoutError, RuntimeError) as e:
            self.timeouts += 1
            await self.close()
            await self.start()
            why = f'timed out after {self.timeout:.0f} s' if isinstance(e, asyncio.TimeoutError) else 'crashed'
            return {'error': f'[the code {why}; the Python state was reset: earlier variables and functions are '
                             f'gone, the task data is reloaded]', 'out': ''}

    async def run(self, code):
        """A `python` call: the text the model sees."""
        r = await self._call('exec', code)
        self.pics = r.get('pictures') or []
        out = r.get('out', '') + (r.get('error') or '')
        return clip(out) if out.strip() else '[no output]'

    async def submit(self, code, overfit=False):
        """A `submit` call: (text the model sees, result with n_ok / n / test_outputs, or None on an error). With
        `overfit`, an accepted program's result also lists the overfit signs (`overfit`)."""
        r = await self._call('submit', code, overfit=overfit)
        self.pics = r.get('pictures') or []
        if r.get('error'):
            return clip(r.get('out', '') + r['error']), None
        text = (r['out'].strip() + '\n\n' if r['out'].strip() else '') + r['feedback']
        return clip(text), r

    async def close(self):
        if self.proc and self.proc.returncode is None:
            self.proc.kill()
            await self.proc.wait()


# the sandbox on its own, before any model: the first task of this run, a wrong program, a long print and an endless
# loop (a 5 s limit here, so the check is quick)
async def _sandbox_selftest():
    task = next(iter(todo.values()))
    sb = Sandbox({'train': task['train'], 'test': task['test'][:1]}, timeout=5)
    await sb.start()
    print(await sb.run('show(train_inputs[0]); print(len(objects(train_inputs[0])), "objects")'))
    print(await sb.run('show_image(train_inputs[0], train_outputs[0])'), '->', len(sb.pics), 'picture(s)')
    pictured = len(sb.pics) == 1 and len(sb.pics[0]) == 2
    text, res = await sb.submit('def transform(grid):\n    return grid')
    print(text[:600], '| first failing example for a picture:', res['fail_pair'][0] if res and res['fail_pair'] else None)
    n = len(await sb.run('print("x" * 200000)'))
    print(n, 'characters back from a 200,000-character print')
    looped = await sb.run('while True: pass')
    print(looped)
    await sb.close()
    return n <= TOOL_OUT + 100 and 'timed out' in looped and pictured

SANDBOX_OK = await _sandbox_selftest()
check(SANDBOX_OK, 'the sandbox self-test failed')

**Timing the tools.** A thin wrapper around the sandbox's calls adds up each attempt's seconds in `python` and `submit` (CPU work, the same on every GPU), so a round's wall time splits into tool time and model time. It changes nothing the model sees.

In [ ]:
TOOL_S = {}   # the asyncio task running an attempt -> seconds its sandbox spent in tool calls (run_batch reads it)

if not getattr(Sandbox, 'timed', False):
    class Sandbox(Sandbox):
        timed = True

        async def _call(self, op, code, **extra):
            t = time.time()
            try:
                return await super()._call(op, code, **extra)
            finally:
                k = asyncio.current_task()
                TOOL_S[k] = TOOL_S.get(k, 0.0) + time.time() - t

**The harness's prompt, tools and notes.**
- **The prompt** is attempt 1's task text, with the tools described where attempt 1 says "Respond in the format of the training output examples". The cell checks that the task text is the same.
- **Notes the harness adds** (wording adapted from a public ARC-AGI-2 REPL agent):
  - after six `python` calls without a submission: submitting is free;
  - after the first accepted program, when its output equals attempt 1's: look for a different rule;
  - when the budget runs out: give the grid now.
- **Parsing:** the model's tool calls are Qwen3-Coder XML, and they are parsed here. A turn whose text compresses below 1 % is a repetition loop.
- **Pictures (with `vision`):** a line before the examples names the colours; the tools text adds `show_image` and the picture of a failing example.

In [ ]:
import zlib
from collections import Counter

# ARC Prize's task text, as attempt 1 sends it, with the tools in place of "Respond in the format ..."
HEAD = ("You are participating in a puzzle solving competition. You are an expert at solving puzzles.\n\n"
        "Below is a list of input and output pairs with a pattern. Your goal is to identify the pattern or "
        "transformation in the training examples that maps the input to the output, then apply that pattern "
        "to the test input to give a final output.\n\n")
EXAMPLES = ("--Training Examples--\n{training_examples}\n--End of Training Examples--\n\n"
            "--Test Input--\n{test_input}\n--End of Test Input--\n\n")
check(HEAD + "Respond in the format of the training output examples\n\n" + EXAMPLES + "Your response:" == PROMPT,
      "the harness's task text differs from attempt 1's")

import numpy as np

IMG = '<|vision_start|><|image_pad|><|vision_end|>'   # a picture's placeholder (this notebook sends no pictures)


def task_parts(task, test_input, head, tail, images=False):
    """The ARC task text, as build_prompt writes it, between `head` and `tail`; pictures are off, so none."""
    ex = ''.join(f'--Example {i}-- \n\n INPUT: \n\n{json.dumps(p["input"])}\n\nOUTPUT: \n\n{json.dumps(p["output"])}\n\n'
                 for i, p in enumerate(task['train']))
    text = (head + '--Training Examples--\n' + ex + '\n--End of Training Examples--\n\n'
            '--Test Input--\n' + json.dumps(test_input) + '\n--End of Test Input--\n\n' + tail)
    return text, []


import contextlib, importlib, io
import arc_skills   # the skills module written above: its index goes into the skills arm's prompt
importlib.reload(arc_skills)   # as section 9 last wrote it (a kernel keeps its first import otherwise)


def repl_tail(vision, skills=False):
    """The tools part of the harness prompt; with `vision`, also show_image and the pictures in submit's reply; with
    `skills`, the skills index (arc_skills.py) instead of the three helpers' one-line descriptions."""
    helpers = ("Helpers: `show(g)` prints a grid with row "
               "and column indices, `diff(a, b)` lists the cells where two same-shape grids differ as (row, col, value in a, "
               "value in b), and `objects(g, bg=None, diagonal=False, "
               "multicolor=False)` returns connected components (color, cells, bbox, size, shape). ")
    if skills:
        helpers = ("`show(g)` prints a grid with row and column indices and `diff(a, b)` lists the cells where two "
                   "same-shape grids differ as (row, col, value in a, value in b).\n" + arc_skills.SKILLS_INDEX + "\n"
                   "Calling `summary()` first gives you exact facts about the task to reason from; `help('name')` shows a "
                   "skill's full description and an example. ")
    return (
        "You have two tools.\n"
        "- `python` runs Python. The task is preloaded as numpy int arrays: `train_inputs`, `train_outputs`, `train` (the "
        "list of (input, output) pairs) and `test_inputs` (the test input above). " + helpers
        + ("`show_image(*grids)` shows you up to 4 grids side by side as a picture, which comes back with the call's "
           "output: use it to look at a grid, at your program's output, or at a comparison. " if vision else "")
        + "Variables and functions persist between calls.\n"
        "- `submit` takes the full source of `def transform(grid)`, runs it on every training input (a numpy int array, "
        "like `train_inputs[i]`), and tells you which "
        "examples it reproduces, with a cell-level diff for the ones it does not"
        + (" and a picture of the first one (expected output left, yours right)" if vision else "")
        + ". When it reproduces all of them it is "
        "accepted, and its output on the test input is your final answer.\n\n"
        "Think as long as you need about what the rule could be. For anything that must be exact (reading, counting or "
        "indexing cells, checking a hypothesis against every training example), use `python`, where the grids are already "
        "loaded: exact work done by eye is where mistakes happen. Submit as soon as you have a candidate: a failing "
        "submission shows exactly which examples and cells are wrong. If you cannot find a program that reproduces every "
        "example, finish with your best output grid for the test input in the format of the training output examples.")


REPL_PROMPT = HEAD + EXAMPLES + repl_tail(False)   # submission 5's harness prompt, unchanged

# The facts arm (repl_m): summary()'s exact facts in the prompt, after the test input; sub 5's tools otherwise
FACTS_NOTE = ('Exact facts about these grids, computed by a program (sizes, colours, objects, symmetries, and how each '
              'output relates to its input):\n{facts}\n\n')


def facts_text(task, test_input):
    """summary()'s facts about the task's training pairs and this test input, as the text it prints."""
    buf = io.StringIO()
    with contextlib.redirect_stdout(buf):
        arc_skills.summary(train=[(np.array(p['input']), np.array(p['output'])) for p in task['train']],
                           test=[np.array(test_input)])
    return buf.getvalue().strip()

TOOLS = [
    {'type': 'function', 'function': {
        'name': 'python',
        'description': ('Run Python code in the task sandbox (numpy and the task grids are preloaded; state persists '
                        'between calls). Print what you need; the output is cut at about 12,000 characters.'),
        'parameters': {'type': 'object', 'properties': {'code': {'type': 'string', 'description': 'Python code'}},
                       'required': ['code']}}},
    {'type': 'function', 'function': {
        'name': 'submit',
        'description': ('Submit a complete program defining transform(grid) -> grid, where grid is a 2-D numpy int '
                        'array. It is run on every training '
                        'input; you are told which examples it reproduces. When it reproduces all of them it is '
                        'accepted and its output on the test input is your answer.'),
        'parameters': {'type': 'object', 'properties': {
            'code': {'type': 'string', 'description': 'Full Python source defining transform(grid).'}},
            'required': ['code']}}},
]

# Notes the harness adds in the conversation (wording from a public ARC-AGI-2 REPL agent, adapted)
THINKCAP_HINT = ('[harness] Your last turn hit the per-turn thinking limit and was cut off before any tool call. Stop '
                 'analysing. In your next reply, call `python` to test your current best hypothesis in code, or '
                 '`submit` a transform implementing it.')
NUDGE = ('\n\n[harness] {n} python calls since your last submission. Submitting is free and returns a cell-level diff '
         'against every example, so a candidate transform is worth submitting even before it is finished.')
REVIEW_NOTE = ('\n\n[harness] Accepted: this program reproduces every training example, and its output on the test input (below) is your first answer. You also get a second answer. Training examples often leave the rule open: look for a different rule that also fits every training example but would change the test output (another reading of what is kept, counted, copied or recoloured, another tie-break, another order). If you find one, submit its program: if it reproduces every training example and its test output differs, it becomes your second answer and the attempt ends. If you are confident no other rule fits, submit the same program again to finish.\nTest input: {ih}x{iw}, colours {icol}. Your program\'s output on it: {oh}x{ow}, colours {ocol}:\n{grid}')
COMMIT = ('[harness] The budget for this puzzle is used up. Give your final answer now: only the output grid for the '
          'test input, in the format of the training output examples.')


def review_note(test_input, grid):
    """The accepted program's test output, shown back as a grid with indices and colour counts."""
    colours = lambda g: dict(sorted(Counter(v for r in g for v in r).items()))
    rows = '\n'.join(f'{r:>3} ' + ''.join(map(str, row)) for r, row in enumerate(grid))
    head = '    ' + ''.join(str(c % 10) for c in range(len(grid[0])))
    return REVIEW_NOTE.format(ih=len(test_input), iw=len(test_input[0]), icol=colours(test_input), oh=len(grid),
                              ow=len(grid[0]), ocol=colours(grid), grid=head + '\n' + rows)


# the opener is optional: one held-out attempt wrote a call as <function=...> alone and was taken as finished
CALL_RE = re.compile(r'(?:<tool_call>\s*)?<function=([\w.-]+)>(.*?)</function>(?:\s*</tool_call>)?', re.DOTALL)
PARAM_RE = re.compile(r'<parameter=([\w.-]+)>\n?(.*?)\n?</parameter>', re.DOTALL)


def xml_tool_calls(content):
    """The model's Qwen3-Coder XML tool calls, and the text around them."""
    calls = [{'id': f'xml_{i}', 'type': 'function',
              'function': {'name': m.group(1), 'arguments': json.dumps(dict(PARAM_RE.findall(m.group(2))))}}
             for i, m in enumerate(CALL_RE.finditer(content))]
    return calls, CALL_RE.sub('', content).strip()


def degenerate(text):
    """A repetition loop: zlib squeezes it below 1 % (prose is ~20 %, an ASCII board ~1.6 %)."""
    b = text.encode()
    return len(b) >= 500 and len(zlib.compress(b)) / len(b) < 0.01


FLAG_NOTE = ('\n\n[harness] This program reproduces every training example, but there are signs it may not have the '
             'general rule:\n{notes}\nLook at the training examples and the test input again with these in mind. If '
             'you find a program that fits every example and also explains them, submit it. If you are confident this '
             'program is right, submit it again.')
PICTURE_NOTE = '\nThe test input and your program\'s output on it as a picture (input left, output right): ' + IMG
FAIL_NOTE = '\nExample {n} as a picture (expected output left, your program\'s output right): ' + IMG

# with pictures off, the harness prompt is submission 5's to the character
first_task = next(iter(todo.values()))
text_prompt, no_pics = task_parts(first_task, first_task['test'][0]['input'], HEAD, repl_tail(False), False)
check(text_prompt == build_prompt(first_task, first_task['test'][0]['input'], REPL_PROMPT) and not no_pics,
      'the text arm differs from submission 5\'s harness prompt')
print(len(text_prompt), 'characters in a harness prompt;',
      len(task_parts(first_task, first_task['test'][0]['input'], HEAD, repl_tail(False, True))[0]),
      'with the skills index')

Pictures stay off: in the probe's E7 they did not help the skills harness (7 of 13 against 10). The attempt below names `picture` only on its `vision` path, which this notebook never takes.

In [ ]:
def picture(*grids, **kw):
    raise RuntimeError('pictures are off in this notebook')

**A harness conversation is an SGLang streaming session.** The server keeps its KV cache and recurrent state between turns, so a turn sends, and prefills, only its new tokens: the tool replies, then the next assistant header. They are written exactly as the chat template writes them, and this cell checks that against the server's own rendering of turn 1.

A turn cut at the budget is continued in a new session, with the whole history prefilled again. A session trims a cut turn's overshoot from the KV cache but not from the GDN state, so a continuation would diverge (the probe's check U2).

With pictures, the history holds one placeholder per picture (the server expands it into the picture's visual tokens), and each request carries the pictures whose placeholders it sends.

In [ ]:
CONTEXT = 262144   # the server's context length; a conversation finishes before it outgrows it
MAX_TURNS = 200    # only a guard against runaway loops; the budget and the context are the real limits
TPREFIX = TSUFFIX = None   # turn 1 of a harness conversation, rendered once by the server (set_repl_template)
# Every /generate request of this notebook holds one of these: more than SLOTS requests in flight crash this server
# (the DFLASH draft sampler's buffers are sized to the 18-request CUDA graphs; stock 0.5.21, dflash_worker_v2.py:844).
INFLIGHT = asyncio.Semaphore(SLOTS)
import aiohttp, random


def healthy():
    try:
        return requests.get(f'{SERVER}/health', timeout=5).status_code == 200
    except requests.RequestException:
        return False


class ServerGuard:
    """Recovers from a server that dies mid-round (E5 lost 7 attempts to a vision-tower OOM). The first conversation
    whose request fails at the connection restarts it (restart_server, section 5); the others wait. At most
    `max_restarts` per kernel. `epoch` counts restarts: a session opened before the last one is gone, so its
    conversation rebuilds from the history it keeps here."""

    def __init__(self, max_restarts=3):
        self.epoch, self.restarts, self.max_restarts = 0, 0, max_restarts
        self.lock, self.calm_until = asyncio.Lock(), 0.0

    async def recover(self, epoch, why):
        async with self.lock:
            if self.epoch != epoch or await asyncio.to_thread(healthy):   # restarted meanwhile, or only a dropped link
                return
            if self.restarts >= self.max_restarts:
                raise RuntimeError(f'the server died again ({why}) after {self.restarts} restarts')
            self.restarts += 1
            t = time.time()
            print(f"{time.strftime('%H:%M:%S')} [guard] the server is down ({why}): restart {self.restarts}", flush=True)
            if not await asyncio.to_thread(restart_server):
                raise RuntimeError('the server did not come back after a restart')
            print(f"{time.strftime('%H:%M:%S')} [guard] server back after {time.time() - t:.0f} s", flush=True)
            self.epoch += 1
            self.calm_until = time.time() + 60   # rebuilds with pictures spread over a minute: one ViT batch each


GUARD = ServerGuard()


def ids(text):
    return TOK.encode(text, add_special_tokens=False).ids


def set_repl_template():
    """The chat template's turn 1 with the tools: the system turn (effort line + tool list) and the assistant header
    around the user's prompt, rendered by the server itself. True if local tokenization gives the server's ids."""
    global TPREFIX, TSUFFIX
    r = requests.post(f'{SERVER}/v1/chat/completions', timeout=300, json={
        'model': 'qwen3.8-27b', 'messages': [{'role': 'user', 'content': 'ping'}], 'tools': TOOLS, 'max_tokens': 1,
        'chat_template_kwargs': {'reasoning_effort': 'xhigh', 'preserve_thinking': True},
        'return_input_ids_in_sglext': True}).json()
    TPREFIX, TSUFFIX = TOK.decode(r['sglext']['input_ids'], skip_special_tokens=False).split('ping', 1)
    ok = ids(TPREFIX + 'ping' + TSUFFIX) == r['sglext']['input_ids'] and 'Reasoning effort is set to xhigh' in TPREFIX
    check(ok, 'the harness template: local tokenization differs from the server, or the xhigh line is missing')
    return ok


class SessionConv:
    """A harness conversation as an SGLang streaming session: the server keeps its KV and recurrent state between
    turns, so a turn sends, and prefills, only its new tokens (the tool replies or a note, then the next assistant
    header), written exactly as the chat template writes them. The history is the prompt plus exactly the ids the
    model wrote.

    Pictures: the history holds one placeholder per picture (the server expands it), and `pics` every picture in
    placeholder order. A request carries the pictures whose placeholders it sends: the new ones on a normal turn, all of
    them when a new session gets the whole history."""

    def __init__(self, http, prompt, pics=()):
        self.http, self.sid, self.hist, self.queue = http, None, [], []
        self.first = ids(TPREFIX + prompt.strip() + TSUFFIX)
        self.close_with, self.out_at, self.rebuild = '', 0, False
        self.parks, self.parked_s = 0, 0.0   # how often the KV gate (below) parked it, and for how long
        self.pics, self.sent = [uri for uri, _ in pics], 0   # every picture, and how many the session already holds
        self.pic_tokens = sum(n for _, n in pics)            # visual tokens the placeholders stand for (KV gate)
        self.epoch = GUARD.epoch                             # the server its session lives on (ServerGuard)

    async def _post(self, path, body):
        async with self.http.post(f'{SERVER}{path}', json=body) as r:
            return await (r.json() if path != '/close_session' else r.text())

    def _pending(self, thinking):
        """What the template writes after the model's last turn: its end if it was cut, the queued tool replies (one
        user turn) and notes, then the next assistant header."""
        s, in_tools = self.close_with, False
        for kind, text in self.queue:
            if kind == 'tool':   # a picture's placeholder is part of its note's text (PICTURE_NOTE, FAIL_NOTE, IMG)
                s += ('' if in_tools else '\n<|im_start|>user') + f'\n<tool_response>\n{text.strip()}\n</tool_response>'
                in_tools = True
            else:
                s += ('<|im_end|>' if in_tools else '') + f'\n<|im_start|>user\n{text.strip()}<|im_end|>'
                in_tools = False
        s += ('<|im_end|>' if in_tools else '') + '\n<|im_start|>assistant\n<think>\n' + ('' if thinking else '\n</think>\n\n')
        self.queue, self.close_with = [], ''
        return ids(s)

    async def turn(self, max_tokens, thinking=True):
        if self.sid is not None and self.epoch != GUARD.epoch:   # its session died with the server (ServerGuard)
            self.sid, self.rebuild = None, True
        if GATE is not None:
            await GATE.before_turn(self)   # may park it first (host RAM, or closed and rebuilt from its history later)
        send = self.first if not self.hist else self._pending(thinking)
        for left in (2, 1, 0):   # retries: after a dead server's restart, or over a pooled connection it had closed
            try:
                if self.sid is not None and self.epoch != GUARD.epoch:   # restarted while it waited (gate, a slot)
                    self.sid, self.rebuild = None, True
                if self.rebuild:   # cut, a removed loop, closed by the gate, a restarted server: a new session gets
                    if self.sid is not None:   # the whole history again, once
                        await self.close()
                    send, self.hist, self.rebuild = self.hist + send, [], False
                    if self.pics and time.time() < GUARD.calm_until:   # after a restart: one picture batch at a time
                        await asyncio.sleep(random.uniform(0, GUARD.calm_until - time.time()))
                if self.sid is None:
                    self.sid = await self._post('/open_session', {'capacity_of_str_len': CONTEXT, 'streaming': True})
                    if not isinstance(self.sid, str):
                        raise RuntimeError(f'open_session failed: {str(self.sid)[:200]}')
                    self.sent, self.epoch = 0, GUARD.epoch   # a new session holds no pictures yet: all of them go now
                body = {'input_ids': send, 'session_params': {'id': self.sid},
                        'sampling_params': {**SAMPLING, 'max_new_tokens': max_tokens, 'no_stop_trim': True,
                                            'skip_special_tokens': False, 'spaces_between_special_tokens': False}}
                if len(self.pics) > self.sent:
                    body['image_data'] = self.pics[self.sent:]
                async with INFLIGHT:
                    data = await self._post('/generate', body)
                if left and isinstance(data, dict) and 'session' in str(data.get('error', '')):   # the server lost it
                    self.sid, self.rebuild = None, True
                    continue
                break
            except aiohttp.ClientConnectionError as e:   # the server went away (or the link dropped) mid-turn
                if not left:
                    raise
                await GUARD.recover(self.epoch, type(e).__name__)
                if self.epoch != GUARD.epoch:             # restarted: its session is gone, rebuild from the history
                    self.sid, self.rebuild = None, True   # kept here (a live server gets the same request again)
        if not isinstance(data, dict) or 'meta_info' not in data:
            raise RuntimeError(f'server error: {str(data)[:300]}')
        mi = data['meta_info']
        fin = mi.get('finish_reason') or {}
        if fin.get('type') == 'abort':   # the server dropped the turn (and the session's KV with it)
            raise RuntimeError(f"turn aborted: {str(fin.get('message'))[:300]}")
        self.hist += send
        self.sent = len(self.pics)
        self.out_at = len(self.hist)
        self.hist += data['output_ids']
        stopped = fin.get('type') == 'stop'
        text = data['text']
        for eos in ('<|im_end|>', '<|endoftext|>'):
            text = text.removesuffix(eos)
        reasoning, content = text.split('</think>', 1) if '</think>' in text else (text, '')
        if not stopped:   # cut at max_tokens: the next turn first closes what is still open, in a new session (above)
            self.close_with = ('' if '</think>' in text else '\n</think>\n\n') + '<|im_end|>'
            self.rebuild = True
        bare = '<function=' in content and '<tool_call>' not in content   # a call written without its opener
        calls, content = xml_tool_calls(content) if '<function=' in content else ([], content)
        return dict(content=content.strip(), reasoning=reasoning.strip(), calls=calls,
                    finish='length' if not stopped else 'tool_calls' if calls else 'stop',
                    prompt=mi.get('prompt_tokens', 0), tokens=mi.get('completion_tokens', 0),
                    cached=mi.get('cached_tokens') or 0, bare=bare and bool(calls))

    def drop_loop(self):
        self.hist = self.hist[:self.out_at] + ids('\n</think>\n\n[a repetition loop, removed by the harness]<|im_end|>')
        self.close_with, self.rebuild = '', True

    def tool(self, call, out, pics=()):
        """A tool reply; `pics` are (uri, tokens) whose placeholders `out` already contains, in order."""
        self.queue.append(('tool', out))
        self.pics += [uri for uri, _ in pics]
        self.pic_tokens += sum(n for _, n in pics)

    def user(self, text):
        self.queue.append(('user', text))

    async def park(self):
        """Move this idle session's KV, recurrent state and draft ring to host RAM (session park, section 4); its next
        turn, to the same session id, copies them back. False if the server refuses (host budget) or lacks it."""
        try:
            res = await self._post('/park_session', {'session_id': self.sid})
        except Exception:  # noqa: BLE001 - a refusal or an old server: the caller closes as before
            return False
        return isinstance(res, dict) and bool(res.get('success'))

    async def close(self):
        if self.sid is not None:
            await self._post('/close_session', {'session_id': self.sid})
            self.sid = None


TOK = Tokenizer.from_file(f'{MODEL_DIR}/tokenizer.json')
HARNESS_OK = set_repl_template() and SANDBOX_OK
print('turn 1 template:', len(ids(TPREFIX)), '+', len(ids(TSUFFIX)), 'tokens around the prompt')


**Keeping the KV pool from filling.** A session holds its whole conversation in the KV pool, also while it waits on a tool. Harness conversations grow to 80–250k tokens, so only ~6–10 of them fit the ~1.0M-token pool at once. In the probe, 18 conversations filled the pool in 24 min and stock SGLang could not recover (round 3). Build `.15`'s session fixes let the server survive that; this gate keeps it from happening.

**How the gate works:**
- Every 2 s it asks the server how full the pool is (`/v1/loads`, counted by the scheduler when asked).
- **Parking:** above `GATE_HIGH` (82 %), a conversation that has just finished a turn is parked: its session moves to host RAM (session park), which frees the pool at once; if the server refuses (host budget), the session is closed instead.
- **Coming back:** a parked conversation returns as soon as its history fits under `GATE_HIGH - GATE_ROOM` (77 %) again, the oldest first; a restore from host RAM takes ~0.2 s, a closed session rebuilds from its exact history (10 s apart).
- **New attempts:** a harness attempt starts only when nothing parked is waiting and its first turn fits under the same mark.

In the probe's rounds 4–6, the gate parked 8–30 conversations per round; all of them came back, and the server never stalled.

In [ ]:
import aiohttp


class KVGate:
    """Keeps session histories from filling the KV pool (above). `usage` is the pool's use as the server's scheduler
    counts it when asked (`/v1/loads`; the `token_usage` gauge in /metrics is the max over the KV pool and the mamba
    slots, refreshed only with a log line), plus what was parked or brought back since that answer.

    Sizes, not fixed marks, decide who comes back (the probe's round 4: with fixed marks at 60 % and 65 %, parked
    conversations and the queue waited while the pool sat at 64-79 % and the batch drained from 18 to 5). A parked
    conversation returns as soon as its history fits under `high - room`, the oldest first; a new attempt starts when
    nothing parked is waiting and its first turn (about `first` of the pool) fits there too. Each admission is counted
    at once (round 4: ten queued attempts started in one poll and the server spent a minute only prefilling)."""

    def __init__(self, high=0.82, room=0.05, first=0.02, every=2.0, gap=10.0):
        self.high, self.room, self.first, self.every, self.gap = high, room, first, every, gap
        self.usage, self.pool, self.retracted = 0.0, None, 0
        self.waiting, self.parks, self.next_back = [], 0, 0.0   # waiting: parked conversations, oldest first
        self.host_parks = 0   # parks the server kept in host RAM (the rest were closed, to be re-read)

    @property
    def parked(self):
        return len(self.waiting)

    async def poll(self, http):
        while True:
            try:
                async with http.get(f'{SERVER}/v1/loads') as r:
                    load = (await r.json())['loads'][0]
                self.pool, self.retracted = load['max_total_num_tokens'], load['queues']['retracted']
                self.usage = load['num_used_tokens'] / self.pool
            except (aiohttp.ClientError, ValueError, KeyError, IndexError, TypeError):
                pass
            await asyncio.sleep(self.every)

    def fits(self, share):
        return self.usage + share <= self.high - self.room

    async def admit_attempt(self):
        while self.waiting or not self.fits(self.first):
            await asyncio.sleep(self.every)
        self.usage += self.first   # counted now, so workers waiting on the same poll do not all start at once

    async def before_turn(self, conv):
        if conv.sid is None or not self.pool or self.usage < self.high:
            return
        hosted = await conv.park()   # host RAM (session park): the next turn, same session id, restores it in ~0.2 s
        if hosted:
            self.host_parks += 1
        else:                   # refused or unavailable: close, and the next turn re-reads the whole history
            with contextlib.suppress(aiohttp.ClientConnectionError):   # a dead server took the session with it
                await conv.close()  # between turns nothing is in flight, so the server frees the whole session at once
            conv.sid, conv.rebuild = None, True
        share = (len(conv.hist) + conv.pic_tokens) / self.pool   # a picture's placeholder stands for its visual tokens
        self.usage -= share  # until the next answer from the server: the session's KV leaves the pool either way
        conv.parks, t = conv.parks + 1, time.time()
        me = object()
        self.waiting.append(me)
        self.parks += 1
        try:
            while self.waiting[0] is not me or not self.fits(share) or (not hosted and time.time() < self.next_back):
                await asyncio.sleep(self.every)
            self.usage += share   # its history comes back
            if not hosted:   # a re-read is a long prefill, so the next re-read waits `gap`; E6's restores took 0.2 s
                self.next_back = time.time() + self.gap
        finally:
            self.waiting.remove(me)
            conv.parked_s += time.time() - t


GATE = KVGate(high=GATE_HIGH, room=GATE_ROOM) if SESSIONS else None
print('KV gate:', {k: v for k, v in vars(GATE).items() if k in ('high', 'room', 'first', 'gap')} if GATE else 'off')

**Park check.** A session parked in host RAM between two turns must answer its next turn exactly as a fresh request of the whole conversation (greedy). The park study proved it on 4 × L4; this repeats it on whatever GPU this run has, before any attempt starts.

In [ ]:
GREEDY = dict(temperature=0.0, max_new_tokens=48)
USER, ASSISTANT_OFF = '<|im_start|>user\n', '<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n'


async def gen(http, prompt_ids, sid=None, **sampling):
    """One /generate request from token ids, in a session if `sid`."""
    body = {'input_ids': prompt_ids, 'sampling_params': {**SAMPLING, 'no_stop_trim': True, 'skip_special_tokens': False,
                                                         'spaces_between_special_tokens': False, **sampling}}
    if sid:
        body['session_params'] = {'id': sid}
    async with INFLIGHT, http.post(f'{SERVER}/generate', json=body) as r:
        data = await r.json()
    if not isinstance(data, dict) or 'meta_info' not in data:
        raise RuntimeError(f'server error: {str(data)[:300]}')
    return data


async def park_check():
    task = challenges[OUTPUTS[0][0]]
    text1 = USER + task_parts(task, task['test'][0]['input'], HEAD, 'Describe the rule in one sentence.')[0] + ASSISTANT_OFF
    async with aiohttp.ClientSession() as http:
        sid = await (await http.post(f'{SERVER}/open_session', json={'capacity_of_str_len': CONTEXT,
                                                                       'streaming': True})).json()
        s1 = await gen(http, ids(text1), sid=sid, **GREEDY)
        parked = await (await http.post(f'{SERVER}/park_session', json={'session_id': sid})).json()
        close = '' if s1['meta_info']['finish_reason'].get('type') == 'stop' else '<|im_end|>'
        turn2 = close + '\n' + USER + 'How many training examples are there? One number.' + ASSISTANT_OFF
        s2 = await gen(http, ids(turn2), sid=sid, **dict(GREEDY, max_new_tokens=16))
        await http.post(f'{SERVER}/close_session', json={'session_id': sid})
        fresh2 = await gen(http, ids(text1) + s1['output_ids'] + ids(turn2), **dict(GREEDY, max_new_tokens=16))
    same = s2['output_ids'] == fresh2['output_ids']
    print(f"park: {parked}; turn 2 cached {s2['meta_info'].get('cached_tokens')} of {s2['meta_info']['prompt_tokens']:,} "
          f"tokens; after the park == fresh: {same} ({s2['text'][:40]!r} vs {fresh2['text'][:40]!r})")
    return same and isinstance(parked, dict) and bool(parked.get('success'))


PARK_OK = await park_check()
if os.environ.get('SGLANG_ENABLE_STREAMING_SESSION_FOLDED_CLAIM') == '1':
    # Submission 10's exact-turn server (the night of 2026-10-22): a session's state is built in decode, a fresh request's
    # in prefill, so a greedy near-tie may differ between them (1022_0252: '3' after the park, '4' fresh, for "how many
    # training examples", which the prompt numbers 0-3). Park exactness there is judged against a prefill-built floor
    # (memory exact-turn-boundaries), not this fresh read: the result is recorded, the round goes on.
    print(f'park check under exact turns: {"same as" if PARK_OK else "differs from"} the fresh read (recorded, not a gate)')
else:
    check(PARK_OK, 'session park is not exact on this server: no round runs')

**One harness attempt** (`repl_attempt`):
- **No per-turn limit.** The model works with `python` and `submit`, its reasoning kept between turns. The limits are the attempt's budget (131,072 tokens, as attempt 1 gets) and the 262k context.
- **A nudge:** after six `python` calls without a submission, the tool reply adds that submitting is free.
- **Acceptance:** a program that reproduces every training example is accepted, and its output on the test input is recorded at once.
  - The model is then asked for a different rule that also fits every training example and would change the test output, because training examples often leave the rule open. If it finds one, that rule's output is the harness's second answer (3 right second answers on the 33 dev outputs, E4 and E8).
- **Other endings:**
  - a turn with no tool call: its grid is the answer;
  - the budget running out first: the attempt ends (`END_SPENT`) and keeps its best partial submission as its answer.
- **Loops:** a repetition loop is cut out of the history.

**What the probe measured:**
- In round 4 (33 evaluation outputs, one harness attempt each), 24 programs were accepted and 13 of them were right.
- 13 outputs were solved in that round, 5 of them outputs that direct answering never solved in 4–6 attempts.
- A harness attempt took a median 32–43 min and generated ~76k tokens.

In [ ]:
def spent(g):
    return {k: g[k] for k in ('prompt', 'tokens', 'cached')}


async def repl_attempt(http, task, test_input, budget, a1=None, live=None, on_accept=None, nudge_every=6,
                       vision=False, overfit=False, skills=False, facts=False):
    """One harness attempt: the tool loop on this attempt's sandbox, in a streaming session (above). `a1` is attempt
    1's grid: an accepted output equal to it (or no attempt 1 grid at all) gets the note asking for a different rule.
    `on_accept(grid)` gets every accepted output at once; `live` names a file that gets one JSON line per turn (its
    tool calls, their code and what they returned). With `vision`, the model also sees pictures: every grid of the
    task in the prompt, show_image's pictures, the first failing example of a submission, and an accepted program's
    test output. With `overfit`, the first accepted program that shows overfit signs (sandbox `_overfit_flags`) is not
    taken yet: the model hears the signs and goes on; it stays the fallback answer. With `skills`, the sandbox has
    arc_skills.py and the prompt its index; with `facts`, the prompt carries summary()'s facts (FACTS_NOTE)."""
    left, ctx, t0 = budget, 0, time.time()   # ctx: tokens the conversation holds (prompt + output, plus tool replies)

    def log(**kw):
        if live:
            with open(live, 'a') as f:
                f.write(json.dumps(dict(kw, s=round(time.time() - t0))) + '\n')

    sb = Sandbox({'train': task['train'], 'test': [{'input': test_input}]}, skills=skills)
    await sb.start()
    tail = repl_tail(vision, skills)
    if facts:   # summary()'s facts, after the test input and before the tools part
        tail = FACTS_NOTE.format(facts=facts_text(task, test_input)) + tail
    prompt, pics = task_parts(task, test_input, HEAD, tail, vision)
    conv = SessionConv(http, prompt, pics)
    rec = dict(requests=[], turns=0, texts=[], python_calls=0, submits=[], cut_turns=0, loops=0, committed=False,
               calls_in_reasoning=0, pictures=len(pics), shown=0,   # shown: show_image pictures the model asked for
               flags=None, flagged_test=None, bare_calls=0)   # the overfit signs of the first flagged acceptance, and its output
    since_submit, accepted, best, final, pending = 0, None, None, None, None   # pending: accepted, asked for another rule
    try:
        while rec['turns'] < MAX_TURNS and left > 512:
            room = CONTEXT - ctx - 1024          # how far the conversation can still grow
            if room < 4096:
                rec['context_full'] = True
                log(turn=rec['turns'] + 1, note=f'context nearly full ({ctx:,} tokens): finishing')
                break
            g = await conv.turn(min(left, room))
            rec['turns'] += 1
            rec['requests'].append(spent(g))
            left -= g['tokens']
            ctx = g['prompt'] + g['tokens']
            calls, content = g['calls'], g['content']
            if not calls and not content.strip() and '<function=' in g['reasoning']:
                calls, _ = xml_tool_calls(g['reasoning'])   # written inside <think>: still a call
                rec['calls_in_reasoning'] += bool(calls)
            rec['bare_calls'] += bool(g.get('bare'))
            rec['texts'].append(dict(g, content=content, calls=calls))
            turn = dict(turn=rec['turns'], tokens=g['tokens'], prompt=g['prompt'], finish=g['finish'],
                        think_chars=len(g['reasoning']), content=content[:1500])
            if degenerate(g['reasoning'] + content):
                rec['loops'] += 1
                log(**turn, note='repetition loop, removed')
                conv.drop_loop()
                conv.user(THINKCAP_HINT)
                continue
            if calls:
                seen = []
                for c in calls:
                    name = c['function']['name']
                    try:
                        code = json.loads(c['function']['arguments'] or '{}').get('code', '')
                    except json.JSONDecodeError:
                        code = ''
                    shown = []                       # pictures for this reply: (uri, visual tokens)
                    if name == 'python':
                        rec['python_calls'] += 1
                        since_submit += 1
                        out = await sb.run(code)
                        if vision and sb.pics:
                            shown = [picture(*grids) for grids in sb.pics]
                            out += ''.join(f'\n[picture {k + 1}] {IMG}' for k in range(len(shown)))
                            rec['shown'] += len(shown)
                        if since_submit % nudge_every == 0:
                            out += NUDGE.format(n=since_submit)
                    elif name == 'submit':
                        since_submit = 0
                        out, res = await sb.submit(code, overfit=overfit and rec['flags'] is None)
                        sub = dict(n_ok=res['n_ok'], n=res['n'], test=(res['test_outputs'] or [None])[0]) if res \
                            else dict(n_ok=-1, n=len(task['train']), test=None)
                        rec['submits'].append(sub)
                        if res and res['n_ok'] == res['n'] and kaggle_grid(sub['test']):
                            if on_accept:
                                on_accept(sub['test'])
                            if res.get('overfit') and rec['flags'] is None:   # once per attempt: not taken yet
                                rec['flags'], rec['flagged_test'] = res['overfit'], sub['test']
                                out += FLAG_NOTE.format(notes='\n'.join(f'- {n}' for n in res['overfit']))
                                best = sub   # the fallback answer if nothing better comes
                                if vision:
                                    out += PICTURE_NOTE
                                    shown = [picture(test_input, sub['test'])]
                            elif pending is None and (a1 is None or sub['test'] == a1):   # attempt 2 is still free
                                pending, rec['first_accept'] = sub, sub['test']
                                out += review_note(test_input, sub['test'])
                                if vision:
                                    out += PICTURE_NOTE
                                    shown = [picture(test_input, sub['test'])]
                            else:
                                accepted = sub
                        else:
                            if res and kaggle_grid(sub['test']) and (best is None or sub['n_ok'] > best['n_ok']):
                                best = sub
                            if vision and res and res.get('fail_pair'):
                                n_ex, want, got = res['fail_pair']
                                out += FAIL_NOTE.format(n=n_ex)
                                shown = [picture(want, got)]
                    else:
                        out = f'[harness] There is no tool {name!r}; the tools are `python` and `submit`.'
                    conv.tool(c, out, shown)
                    ctx += len(out) + 16 + sum(n for _, n in shown)   # grids print about one token per character
                    seen.append(dict(name=name, code=code[:3000], out=out[:2500]))
                log(**turn, calls=seen)
                if accepted:
                    break
                continue
            log(**turn, note='cut at the budget' if g['finish'] == 'length' else 'ended without a tool call')
            if g['finish'] == 'length':
                rec['cut_turns'] += 1
                conv.user(THINKCAP_HINT)
                continue
            final = parse_grid(content) if content.strip() else None
            break
        rec['spent'] = left <= 512 or bool(rec.get('context_full'))   # an answer-only turn would re-read it all
        if accepted is None and final is None and pending is None and not (END_SPENT and rec['spent']):
            conv.user(COMMIT)
            try:
                g = await conv.turn(max(256, min(8192, CONTEXT - ctx - 512)), thinking=False)
            except RuntimeError as e:          # no room left even for the answer: keep the best submission
                g = dict(content='', reasoning='', calls=[], finish=f'error: {e}'[:200], prompt=0, tokens=0, cached=0)
            rec['requests'].append(spent(g))
            rec['texts'].append(g)
            final, rec['committed'] = parse_grid(g['content']), True
            log(turn='commit', tokens=g['tokens'], prompt=g['prompt'], finish=g['finish'], content=g['content'][:1500])
    finally:
        rec['sandbox_calls'], rec['sandbox_timeouts'] = sb.calls, sb.timeouts
        rec['parks'], rec['parked_s'] = conv.parks, round(conv.parked_s)
        await sb.close()
        try:
            await conv.close()
        except Exception:  # noqa: BLE001 - a dropped connection must not hide the attempt's own outcome
            pass
    final = final if final is not None and kaggle_grid(final) else None
    p = pending['test'] if pending else None
    if accepted:
        if p is not None and p != accepted['test']:   # a different rule after the note: the second answer
            return dict(rec, answer=p, second=accepted['test'], verified=True, source='program', alternative=True)
        return dict(rec, answer=accepted['test'], verified=True, source='program')
    if pending:   # accepted once, then the model ended without submitting again
        return dict(rec, answer=p, second=final if final and final != p else None, verified=True, source='program')
    if final:
        return dict(rec, answer=final, second=best['test'] if best and best['test'] != final else None,
                    verified=False, source='grid')
    return dict(rec, answer=best['test'] if best else None, verified=False, source='best-submit' if best else None)

## 6. A round

`run_batch` runs one harness attempt per job, `SLOTS` requests in flight, a new attempt starting only when the KV gate has room for it. Each finished attempt writes one record (`OUT/<round>/records.jsonl`), its full text (`attempts/`) and its turns as they happen (`live/`). Every 5 minutes it prints a status line: progress, the KV gate, and the server's latest decode line.

A record holds, besides what the attempt returns:
- **the score:** the answer right (`solved`), the second answer right, either (`pair_solved`, what one harness attempt fields as attempt 1 and 2), the first accepted program right, any accepted program right (`any_accepted_right`, the ceiling of choosing among them);
- **the cost:** tokens generated and newly prefilled, seconds of wall time and of tool calls (`tool_s`), parks;
- **the path:** turns, `python` calls, submits, accepted programs, cut turns, loops, a spent budget, bare tool calls.

`summarize` prints parents and children side by side, family by family.

In [ ]:
import aiohttp

OUT = '/kaggle/working/bench'
TIMEOUT = aiohttp.ClientTimeout(total=None, sock_connect=60, sock_read=None)   # a harness turn is silent until it ends


def decode_line():
    """The server's latest decode line (GPU 0)."""
    with open('sglang.log', 'rb') as f:
        f.seek(max(0, os.path.getsize('sglang.log') - 200_000))
        lines = [l for l in f.read().decode(errors='replace').splitlines() if 'TP0' in l and 'Decode batch' in l]
    return lines[-1].split('] ', 1)[-1] if lines else 'no decode line yet'


def live_summary(path):
    """What an attempt has done so far, from its live log: turns, tokens, the submits' scores, its last step."""
    try:
        turns = [json.loads(l) for l in open(path)]
    except (FileNotFoundError, json.JSONDecodeError):
        return 0, 0, 'in its first turn'
    scores = [c['out'].split('\n', 1)[0][:40] for t in turns for c in t.get('calls', []) if c['name'] == 'submit']
    last = turns[-1]
    step = ', '.join(c['name'] for c in last['calls']) if last.get('calls') else last.get('note', last.get('turn'))
    return len(turns), sum(t.get('tokens', 0) for t in turns), f"last: {step}; submits: {scores or 'none'}"


async def run_batch(name, jobs, minutes, slots=SLOTS):
    """Run (task, test, sample) jobs, `slots` at a time, one harness attempt each, for at most `minutes` (a safety
    stop: attempts still running then are cancelled and recorded as unfinished)."""
    out = f'{OUT}/{name}'
    for d in ('attempts', 'live'):
        os.makedirs(f'{out}/{d}', exist_ok=True)
    queue, done, running, t_round = list(jobs), [], {}, time.time()
    stop = t_round + minutes * 60
    print(f'{name}: {len(jobs)} attempts, {slots} requests in flight, safety stop at '
          f'{time.strftime("%H:%M", time.gmtime(stop + 19800))} IST', flush=True)

    def write(rec, texts=None):
        done.append(rec)
        with open(f'{out}/records.jsonl', 'a') as f:
            f.write(json.dumps(rec) + '\n')
        if texts is not None:
            with open(f"{out}/attempts/{rec['task']}_{rec['test']}_s{rec['sample']}.json", 'w') as f:
                json.dump(dict(rec, texts=texts), f)

    async def status(every=300):
        while True:
            await asyncio.sleep(every)
            fin = [r for r in done if not r.get('unfinished')]
            kinds = ' '.join(f"{k} {sum(r['solved'] for r in fin if r['kind'] == k)}"
                             f"(+{sum(r['second_solved'] and not r['solved'] for r in fin if r['kind'] == k)})"
                             f"/{sum(r['kind'] == k for r in fin)}" for k in ('parent', 'child'))
            head = (f"{time.strftime('%H:%M:%S')} {name}: {len(done)}/{len(jobs)} done ({kinds} solved), "
                    f"{len(running)} running, {len(queue)} waiting, {(time.time() - t_round) / 60:.0f} min in"
                    + (f" | KV {GATE.usage:.0%}, parked {GATE.parked}, parks {GATE.parks} ({GATE.host_parks} to host RAM)"
                       if GATE else '') + f" | server: {decode_line()}")
            print(head, flush=True)
            rows = []
            for aid, t in sorted(running.items(), key=lambda kv: kv[1]):
                n, tok, what = live_summary(f'{out}/live/{aid}.jsonl')
                rows.append(f'  {aid}: {(time.time() - t) / 60:.0f} min, {n} turns, {tok:,} tokens, {what}')
            with open(f'{out}/status.txt', 'w') as f:
                f.write(head + '\n' + '\n'.join(rows) + '\n')

    async def worker(http):
        while queue and time.time() < stop:
            if GATE is not None:
                await GATE.admit_attempt()   # a new attempt only while the KV pool has room (KVGate)
                if not queue or time.time() >= stop:
                    break
            tid, i, k = queue.pop(0)
            aid, task, t = f'{tid}_{i}_s{k}', challenges[tid], time.time()
            base = dict(arm='skills', task=tid, test=i, sample=k, kind=KIND[tid], family=FAMILY[tid],
                        start_s=round(t - t_round))
            me = asyncio.current_task()
            TOOL_S.pop(me, None)
            running[aid] = t
            try:
                rec = await repl_attempt(http, task, task['test'][i]['input'], HARNESS_TOKENS,
                                         live=f'{out}/live/{aid}.jsonl', skills=True)
            except asyncio.CancelledError:
                n, tok, _ = live_summary(f'{out}/live/{aid}.jsonl')
                write(dict(base, unfinished=True, turns=n, generated=tok, solved=False, second_solved=False,
                           pair_solved=False, seconds=round(time.time() - t), tool_s=round(TOOL_S.pop(me, 0.0), 1)))
                raise
            except Exception as e:  # noqa: BLE001 - one failed attempt must not stop the round
                rec = dict(answer=None, second=None, requests=[], submits=[], turns=0, texts=[],
                           error=f'{type(e).__name__}: {e}'[:300])
            finally:
                running.pop(aid, None)
            sol = solutions[tid][i]
            acc = [s['test'] for s in rec.get('submits', []) if s['n_ok'] == s['n'] and kaggle_grid(s['test'])]
            rec.update(base, seconds=round(time.time() - t), end_s=round(time.time() - t_round),
                       tool_s=round(TOOL_S.pop(me, 0.0), 1),
                       solved=rec.get('answer') == sol, second_solved=rec.get('second') == sol,
                       pair_solved=sol in (rec.get('answer'), rec.get('second')),
                       first_accept_solved=rec['first_accept'] == sol if rec.get('first_accept') else None,
                       accepted=len(acc), accepted_distinct=len({json.dumps(g) for g in acc}),
                       any_accepted_right=sol in acc,
                       generated=sum(q['tokens'] for q in rec['requests']),
                       prefilled=sum(q['prompt'] for q in rec['requests']),
                       prefill_new=sum(q['prompt'] - q['cached'] for q in rec['requests']))
            write(rec, rec.pop('texts', None))
            print(time.strftime('%H:%M:%S'), json.dumps({k: rec.get(k) for k in (
                'arm', 'task', 'test', 'kind', 'solved', 'second_solved', 'verified', 'source', 'any_accepted_right',
                'turns', 'python_calls', 'generated', 'prefill_new', 'spent', 'parks', 'tool_s', 'seconds',
                'error')}), flush=True)

    st = asyncio.create_task(status())
    async with aiohttp.ClientSession(timeout=TIMEOUT) as http:
        gate = asyncio.create_task(GATE.poll(http)) if GATE is not None else None
        workers = [asyncio.create_task(worker(http)) for _ in range(slots)]
        try:
            await asyncio.wait(workers, timeout=max(1, stop - time.time()))
        finally:   # the safety stop, or an interrupt of this cell: no worker may outlive it
            st.cancel()
            if gate is not None:
                gate.cancel()
            for w in workers:
                w.cancel()
            await asyncio.gather(*workers, return_exceptions=True)
    print(f"{name}: {len(done)} records in {(time.time() - t_round) / 60:.0f} min, "
          f"{sum(bool(r.get('unfinished')) for r in done)} cancelled at the safety stop; {len(queue)} never started")
    return done


def summarize(recs):
    """Parents and children: solved by the answer, by the pair (answer or second), accepted programs and how many are
    right, the ceiling over accepted programs, how attempts ended, tokens and time; then one row per family."""
    for kind in ('parent', 'child', None):
        rs = [r for r in recs if kind is None or r['kind'] == kind]
        fin = [r for r in rs if not r.get('unfinished')]
        if not rs:
            continue
        n = max(1, len(fin))
        ver = [r for r in fin if r.get('verified')]
        print(f"{kind or 'all':6s} {len(rs)} outputs: solved {sum(r['solved'] for r in rs)}, pair {sum(r['pair_solved'] for r in rs)}"
              f" | accepted {len(ver)}, right {sum(r['solved'] for r in ver)}, ceiling {sum(bool(r.get('any_accepted_right')) for r in fin)}"
              f" | spent {sum(bool(r.get('spent')) for r in fin)}, no answer {sum(r.get('answer') is None for r in fin)},"
              f" unfinished {len(rs) - len(fin)}, errors {sum(bool(r.get('error')) for r in rs)}"
              f" | per attempt: {sum(r.get('generated', 0) for r in fin) / n:,.0f} generated,"
              f" {sum(r.get('prefill_new', 0) for r in fin) / n:,.0f} prefilled, {sum(r.get('turns') or 0 for r in fin) / n:.1f} turns,"
              f" median {sorted(r['seconds'] for r in fin)[len(fin) // 2] if fin else 0} s, tools {sum(r.get('tool_s', 0) for r in fin) / n:.0f} s")
    mark = lambda r: ('u' if r.get('unfinished') else 'S' if r['solved'] else 's' if r['second_solved']
                      else 'v' if r.get('verified') else '.')
    print('\nS solved, s second answer right, v accepted but wrong, . wrong or no answer, u unfinished')
    for p in PARENTS:
        cells = []
        for t in [p, *families['children'][p]]:
            cells.append(t + ' ' + ''.join(mark(r) for r in sorted((r for r in recs if r['task'] == t),
                                                                     key=lambda r: (r['test'], r['sample']))))
        print('  ' + ' | '.join(cells))

## 7. Rounds

One cell per GPU type. Each states its prediction before it runs and its result after.

### Round 1: 4 × L4, the baseline (written 2026-10-18 20:35 IST, before it runs)

One attempt per output, 32 attempts; submission 6's server and harness exactly.

**Prediction:**
- **Pace:** like E9 (29 attempts in ~100 min at 7–11 resident): ~32 min per attempt, the round ~1.8–2.2 h.
- **Score:** the harness solved 62 % of held-out22 by its answer (18 of 29), 50 % of the other dev outputs (E8). Here: **parents 5–8 of 12, children 9–14 of 20, 14–22 of 32 by the pair.**
- **Children against parents:** the first child pair took Claude solvers ~0.4× its parent's effort, so children may be a little easier; a gap within ±15 points would make them a usable stand-in.

In [ ]:
r1 = await run_batch('r1_l4x4', [(t, i, 0) for t, i in OUTPUTS], minutes=ROUND_MINUTES)
summarize(r1)

**Round 1's lost attempt (22:20 IST).** Modal restarted the 4 × L4 container at ~22:14 IST with 31 of 32 attempts finished. The one running, parent `20a9e565` test 1 (33 min, 21 turns, 98.8k tokens, no submit yet), was lost with the container; the other 31 records survived on the cache Volume. This cell runs that output once more, as a fresh attempt on the same server, so round 1 covers all 32 outputs. Prediction: like round 1's other late parents, a wrong late acceptance or a spent budget is the likely ending; a solve would be a surprise.

In [ ]:
r1b = await run_batch('r1_l4x4_tail', [('20a9e565', 1, 0)], minutes=ROUND_MINUTES)
summarize(r1b)

**Round 1's result (4 × L4, written 2026-10-18 23:00 IST).** With the rerun of the lost output (cell `hbr1tail`), all 32: **24 right by the pair; parents 7 of 12 (58 %), children 17 of 20 (85 %)**; ARC's per-task score 56.2 % against 84.4 %. 23 of 30 first acceptances right; 2 spent budgets; median 64.5k tokens and 35 min per attempt; the round 98 min. Against the prediction: parents as expected (5–8), children above it (9–14), the gap 27 points against the predicted ±15, directional on one sample (Fisher p = 0.12). Why the parents failed is in `docs/harness_flaws/` (cases `R1_*`): every studied failure held a candidate 45–75k tokens before it submitted or ran out.

### Round 2: one RTX PRO 6000 (written 2026-10-18 22:15 IST, before it runs)

One attempt per output, 32 attempts, the same harness and cells as round 1. The server: build `.15` with its session fixes, FlashInfer attention and the CUTLASS FP8 GEMM (sm_120's own default), and the same generic spec-decode patches as round 1 (GDN fold, windowed draft KV, block verify); none of the L4-only kernels. Requests in flight as on L4 (18, 24 held). Chosen by the sm_120 A/B `1018_2137_rtxpro6000-ab`: 809 / 597 / 129 tok/s at 10 / 6 / 1 concurrent on real xhigh contexts, KV pool 1.41M tokens (L4 × 4: 1.01M).

**Prediction:**
- **Pace:** decode ~1.4–1.7× round 1's at the same load, and fewer parks (a 40 % larger pool): the round ends in ~60–80 min against round 1's ~100.
- **Score:** the same harness on the same set, so the same solve rate within noise: round 1's outcome ±3 outputs. A gap beyond that would point at the GPU (numerics or a time-dependent path), not at the harness.

In [ ]:
r2 = await run_batch('r2_rtxpro6000', [(t, i, 0) for t, i in OUTPUTS], minutes=ROUND_MINUTES)
summarize(r2)

**Round 2's result (one RTX PRO 6000, written 2026-10-18 23:30 IST).** **19 of 32 right; parents 5 of 12, children 14 of 20**; 57 min for the round. The difference from round 1 is spent budgets: **12 against 2**. Output by output, 9 went from not spent to spent and none the other way (p = 0.004), and the RTX used more tokens on 22 of 30 (median ×1.23); its acceptances were more often right (19 of 21 against 23 of 30). Same cells, data and sampling; no tool timeouts, retracts or loops. The bulk of the turn-length distribution is unchanged and the tail is heavier (25 turns of 30k+ tokens, 42.5 % of all tokens, against 14 and 28.5 %). So a GPU round measures the harness together with the server's numerics: harness effectiveness measured here does not transfer to Kaggle's 4 × L4 as is. Next: test the mechanism (replayed long turns on both servers; the FP8 lm_head switched on here) before the H100 and B200 rounds.

### Round 3: one H100 (written 2026-10-18 23:51 IST, before it runs)

One attempt per output, 32 attempts, the same harness and cells as rounds 1 and 2 (the user, 23:3x: "start H100"). The server: build `.15` with its session fixes and the flags the H100 A/B (`1018_2336_h100-ab`) chooses among SGLang's own sm_90 defaults (FA3 attention, DeepGEMM FP8 GEMMs), FlashInfer, and our generic spec-decode patches; none of the L4-only kernels. Requests in flight as on L4 and the RTX (18, 24 held).

**Why this round decides something beyond the GPU:** round 2 (RTX PRO 6000) spent 12 budgets against round 1's 2 on the same outputs, through longer silent turns. Both single-GPU servers run TP1 with stock FP8 GEMMs and a BF16 lm_head; only L4 runs TP4 with the lossy int8 all-reduce, the FP8 lm_head and the G5 GEMMs. The H100 shares the RTX's "clean" path with entirely different kernels (FA3 and DeepGEMM against FlashInfer and CUTLASS).

**Prediction:**
- **Pace:** decode ~1.6× the RTX's at the same load (the A/B's stock arm: 1,319 / 928 / 223 tok/s at 10 / 6 / 1), with a smaller KV pool (0.75–1.0M against the RTX's 1.41M), so more parks: the round ends in ~40–50 min.
- **Behaviour:** like the RTX, not like L4: 7–12 spent budgets, a median of ~85–95k generated tokens per attempt, 17–22 right by the pair. If instead it spends ≤ 4 budgets and scores ~24, the RTX's sm_120 path is the odd one, not L4's lossy numerics.

In [ ]:
r3 = await run_batch('r3_h100', [(t, i, 0) for t, i in OUTPUTS], minutes=ROUND_MINUTES)
summarize(r3)

### Round 4: one Colab G4, round 2 again (written 2026-10-21 17:42 IST, before it runs)

One attempt per output, 32 attempts, the same harness, cells and frozen set as rounds 1–3, and round 2's server:
`rtxpro6000_harness.toml` on one RTX PRO 6000 Blackwell (Colab's G4, run `1021_1742_colab-g4-harness-r4`). The build is
`.16`, which adds only guards to round 2's `.15`. Requests in flight as before (18, 24 held).

Phase 2 of the Colab G4 plan (`1021_1651`) showed this box is round 2's server:
- greedy outputs identical to Modal's RTX PRO 6000 on both probe prompts;
- KV pool within 0.02 %;
- decode within the acceptance noise (802.6 / 562.4 / 124.6 against 809.4 / 596.7 / 128.5 tok/s).

So this round asks whether round 2 reproduces: is its heavy tail of long thinking turns (12 spent budgets against round 1's
2) a property of this server, or a draw?

**Prediction:**
- **Pace:** the round ends in ~50–65 min (round 2: 57).
- **Score and endings:** round 2's within noise, 19 ± 4 right. Spent budgets high as in round 2, ~8–14. If they fall to
  round 1's level (≤ 4), round 2's gap was the sampling draw, not the GPU.
- **Paired against round 2, output by output:** no systematic token difference (sign test p > 0.1).

In [ ]:
r4 = await run_batch('r4_colab_g4', [(t, i, 0) for t, i in OUTPUTS], minutes=ROUND_MINUTES)
summarize(r4)

**Round 4's result (one Colab G4, written 2026-10-21 18:47 IST).** **21 of 32 right; parents 6 of 12, children 15 of
20**; 44 min for the round (run `1021_1742_colab-g4-harness-r4`). Spent budgets **8**, against round 2's 12 and
round 1's 2.

Against the prediction: the score is inside 19 ± 4 and the spent budgets at the low edge of 8–14, but the round was
faster than 50–65 min, and the paired token test failed: output by output, round 4 used fewer tokens than round 2 on 19
of 25 (sign test p = 0.015, median ×0.86). Against round 1 (4 × L4) it used more on 19 of 30 (p = 0.20, ×1.12), and 6
outputs went from not spent to spent with none the other way (p = 0.03; round 2 against round 1: 9 and none).

So round 2's direction reproduces on a second RTX PRO 6000 server: this server class ends more attempts on a spent
budget than 4 × L4. Its size in round 2 was partly a high draw; round 4 lies between rounds 1 and 2. A G4 round measures
the harness on round 2's server class: comparisons between harness variants on G4 hold, and absolute scores read a few
outputs below 4 × L4's. The v1 children were again easier than their parents (15 of 20 against 6 of 12), as in rounds
1 and 2.

## Round 5: the v2 eval children of parents #21–#30, on one Colab G4

**Why.** Rounds 1–4 measured the harness on the first frozen set, whose 16 children came from the v1 process. Those
children turned out easier for the harness than their parents (round 1: children 17/20 against parents 7/12; round 2:
14/20 against 5/12). The v2 process (2026-10-21) designs each child one notch harder than its parent and accepts it only
when Sonnet's thinking on it is at least 0.8 of the parent's. **Do the v2 children behave like evaluation tasks for our
harness?** If they do, the user gives the go-ahead to make children for the whole evaluation set.

**What runs.** Rounds 1–4's harness, unchanged, on a second frozen set, `v2b1_*.json` beside `bench24_*.json`:
- **10 evaluation parents,** #21–#30 in sorted-id order, 14 outputs. Five were used while the harness was built:
  `38007db0` was in its dev set (PILOT12); `31f7f899` and `3a25b0d8` were in HELDOUT12, and `2c181942` and `2d0172a1`
  in the heldout24 cohort. Those five are reported apart.
- **Their 20 v2 children** (batch 1 of the v2 process), 28 outputs.
- **Two attempts per output** (samples 0 and 1; all of sample 0 is queued first): 84 attempts, 18 in flight.

The next cell replaces the set's names (`challenges`, `solutions`, `families`, `PARENTS`, `FAMILY`, `KIND`, `OUTPUTS`);
`run_batch` and `summarize` read them when they run. On Kaggle, `v2b1_*.json` must sit in the same input as `bench24`.

**Prediction (18:45 IST, before the round):**
- parents: 50 % of attempts right by the pair (35–65 %); the five seen during the harness's tuning above the other five;
- children: 55 % (40–70 %). A child rate more than ~20 points above the parents' (v1's gap) means the v2 children are
  still easier than evaluation tasks for this harness; within ±15 points means they behave like them;
- generated tokens per attempt, child against its parent: median ratio ~1.0 (0.7–1.4);
- both samples agree on 70–80 % of outputs;
- wall time ~2–2.5 h on one G4 (~20 compute units).

In [ ]:
# Round 5's set: evaluation parents #21-#30 and their 20 v2 eval children (v2b1_*.json, beside bench24_*.json)
challenges = json.load(open(f'{BENCH}/v2b1_challenges.json'))
solutions = json.load(open(f'{BENCH}/v2b1_solutions.json'))
families = json.load(open(f'{BENCH}/v2b1_families.json'))
PARENTS = families['parents']
FAMILY = {c: p for p in PARENTS for c in [p, *families['children'][p]]}   # task -> its parent
KIND = {t: 'parent' if t == p else 'child' for t, p in FAMILY.items()}
OUTPUTS = [(t, i) for p in PARENTS for t in [p, *families['children'][p]] for i in range(len(challenges[t]['test']))]
check(len(challenges) == 30 and len(OUTPUTS) == 42, f'expected 30 tasks and 42 outputs: {len(challenges)}, {len(OUTPUTS)}')
check(set(solutions) == set(challenges) and set(FAMILY) == set(challenges), 'the set files do not match')
print(BENCH, '| v2b1:', len(PARENTS), 'parents,', len(FAMILY) - len(PARENTS), 'children,', len(OUTPUTS), 'test outputs:',
      sum(KIND[t] == 'parent' for t, _ in OUTPUTS), 'parent,', sum(KIND[t] == 'child' for t, _ in OUTPUTS), 'child')

# A targeted subset (2026-10-23; the user: "runs on targeted worthy tasks / puzzles instead of all, so we can find the faults
# more quickly"): NIGHT_TASKS=<task ids, comma-separated> keeps only those tasks' outputs; unset, the whole set runs.
_only = [t for t in os.environ.get('NIGHT_TASKS', '').split(',') if t]
if _only:
    check(set(_only) <= set(challenges), f'NIGHT_TASKS not in the set: {sorted(set(_only) - set(challenges))}')
    OUTPUTS = [(t, i) for t, i in OUTPUTS if t in _only]
    print('NIGHT_TASKS: a targeted subset of', len(_only), 'tasks,', len(OUTPUTS), 'outputs:',
          sum(KIND[t] == 'parent' for t, _ in OUTPUTS), 'parent,', sum(KIND[t] == 'child' for t, _ in OUTPUTS), 'child')


In [ ]:
# The 10-22 combined round's set: evaluation parents #31-#40 and their 20 v2 eval children (v2b2_*.json, beside bench24_*)
challenges = json.load(open(f'{BENCH}/v2b2_challenges.json'))
solutions = json.load(open(f'{BENCH}/v2b2_solutions.json'))
families = json.load(open(f'{BENCH}/v2b2_families.json'))
PARENTS = families['parents']
FAMILY = {c: p for p in PARENTS for c in [p, *families['children'][p]]}   # task -> its parent
KIND = {t: 'parent' if t == p else 'child' for t, p in FAMILY.items()}
OUTPUTS = [(t, i) for p in PARENTS for t in [p, *families['children'][p]] for i in range(len(challenges[t]['test']))]
check(len(challenges) == 30 and len(OUTPUTS) == 48, f'expected 30 tasks and 48 outputs: {len(challenges)}, {len(OUTPUTS)}')
check(set(solutions) == set(challenges) and set(FAMILY) == set(challenges), 'the set files do not match')
check(PARENTS == ['3e6067c3', '409aa875', '446ef5d2', '45a5af55', '4a21e3da', '4c3d4a41', '4c416de3', '4c7dc4dd',
                  '4e34c42c', '53fb4810'], f'not parents #31-#40: {PARENTS}')
print(BENCH, '| v2b2:', len(PARENTS), 'parents,', len(FAMILY) - len(PARENTS), 'children,', len(OUTPUTS), 'test outputs:',
      sum(KIND[t] == 'parent' for t, _ in OUTPUTS), 'parent,', sum(KIND[t] == 'child' for t, _ in OUTPUTS), 'child')

In [ ]:
# Exp 1's set (2026-10-24): evaluation parents #41-#50 and their 20 v2 eval children (v2b3_*.json, beside bench24_*)
challenges = json.load(open(f'{BENCH}/v2b3_challenges.json'))
solutions = json.load(open(f'{BENCH}/v2b3_solutions.json'))
families = json.load(open(f'{BENCH}/v2b3_families.json'))
PARENTS = families['parents']
FAMILY = {c: p for p in PARENTS for c in [p, *families['children'][p]]}   # task -> its parent
KIND = {t: 'parent' if t == p else 'child' for t, p in FAMILY.items()}
OUTPUTS = [(t, i) for p in PARENTS for t in [p, *families['children'][p]] for i in range(len(challenges[t]['test']))]
check(len(challenges) == 30 and len(OUTPUTS) == 42, f'expected 30 tasks and 42 outputs: {len(challenges)}, {len(OUTPUTS)}')
check(set(solutions) == set(challenges) and set(FAMILY) == set(challenges), 'the set files do not match')
check(PARENTS == ['5545f144', '581f7754', '58490d8a', '58f5dbd5', '5961cc34', '5dbc8537', '62593bfd', '64efde09',
                  '65b59efc', '67e490f4'], f'not parents #41-#50: {PARENTS}')
print(BENCH, '| v2b3:', len(PARENTS), 'parents,', len(FAMILY) - len(PARENTS), 'children,', len(OUTPUTS), 'test outputs:',
      sum(KIND[t] == 'parent' for t, _ in OUTPUTS), 'parent,', sum(KIND[t] == 'child' for t, _ in OUTPUTS), 'child')
_only = [t for t in os.environ.get('NIGHT_TASKS', '').split(',') if t]   # as hbv2b1data: a targeted subset
if _only:
    check(set(_only) <= set(challenges), f'NIGHT_TASKS not in the set: {sorted(set(_only) - set(challenges))}')
    OUTPUTS = [(t, i) for t, i in OUTPUTS if t in _only]
    print('NIGHT_TASKS: a targeted subset of', len(_only), 'tasks,', len(OUTPUTS), 'outputs')

In [ ]:
# The fresh calibration set (2026-10-31; runs/plans/1029_fresh61/README.md): evaluation parents #61-#80 and their 40 v2
# eval children (fresh61_*.json, beside bench24_*), 60 tasks / 78 outputs: the set of Flash-Next's fresh runs (p1, p2, t1,
# t2, qg61 p1/p2). Here it runs the 27B (side session, the user 10-31: "2 normal runs on two colabs on the same set with
# 27b with its best performance stack").
challenges = json.load(open(f'{BENCH}/fresh61_challenges.json'))
solutions = json.load(open(f'{BENCH}/fresh61_solutions.json'))
families = json.load(open(f'{BENCH}/fresh61_families.json'))
PARENTS = families['parents']
FAMILY = {c: p for p in PARENTS for c in [p, *families['children'][p]]}   # task -> its parent
KIND = {t: 'parent' if t == p else 'child' for t, p in FAMILY.items()}
OUTPUTS = [(t, i) for p in PARENTS for t in [p, *families['children'][p]] for i in range(len(challenges[t]['test']))]
check(len(challenges) == 60 and len(OUTPUTS) == 78, f'expected 60 tasks and 78 outputs: {len(challenges)}, {len(OUTPUTS)}')
check(set(solutions) == set(challenges) and set(FAMILY) == set(challenges), 'the set files do not match')
check(PARENTS == ['7b80bb43', '7c66cb00', '7ed72f31', '800d221b', '80a900e0', '8698868d', '88bcf3b4', '88e364bc', '89565ca0', '898e7135', '8b7bacbf', '8b9c3697', '8e5c0c38', '8f215267', '8f3a5a89', '9385bd28', '97d7923e', '981571dc', '9aaea919', '9bbf930d'], f'not parents #61-#80: {PARENTS}')
print(BENCH, '| fresh61:', len(PARENTS), 'parents,', len(FAMILY) - len(PARENTS), 'children,', len(OUTPUTS), 'test outputs:',
      sum(KIND[t] == 'parent' for t, _ in OUTPUTS), 'parent,', sum(KIND[t] == 'child' for t, _ in OUTPUTS), 'child')
_only = [t for t in os.environ.get('NIGHT_TASKS', '').split(',') if t]   # as hbv2b1data: a targeted subset
if _only:
    check(set(_only) <= set(challenges), f'NIGHT_TASKS not in the set: {sorted(set(_only) - set(challenges))}')
    OUTPUTS = [(t, i) for t, i in OUTPUTS if t in _only]
    print('NIGHT_TASKS: a targeted subset of', len(_only), 'tasks,', len(OUTPUTS), 'outputs')


In [ ]:
# The decode-3x campaign's hard calibration set (2026-10-25): 27 tasks of evaluation parents #21-#50 and their v2 eval
# children that Flash-Next often got wrong or ran near the 131k budget on (docs/reports/1025_1016_fn_decode_3x/hard_set.py:
# attempt_1 wrong in >= 1/3 of its rounds, or median first-attempt tokens >= 96k, or a spent budget). Longest first
# (fnhard27_families.json 'tasks', the LPT order), so a round ends on short attempts and every speed run starts the same queue.
challenges = json.load(open(f'{BENCH}/fnhard27_challenges.json'))
solutions = json.load(open(f'{BENCH}/fnhard27_solutions.json'))
families = json.load(open(f'{BENCH}/fnhard27_families.json'))
PARENTS = families['parents']
FAMILY = dict(families['family'])   # task -> its parent (the parent itself may be outside the set)
KIND = dict(families['kind'])
OUTPUTS = [(t, i) for t in families['tasks'] for i in range(len(challenges[t]['test']))]
check(len(challenges) == 27 and len(OUTPUTS) == 44, f'expected 27 tasks and 44 outputs: {len(challenges)}, {len(OUTPUTS)}')
check(set(solutions) == set(challenges) == set(FAMILY) == set(KIND) == set(families['tasks']), 'the set files do not match')
check(len(PARENTS) == 8 and all(KIND[p] == 'parent' and p in challenges for p in PARENTS), f'expected 8 parents: {PARENTS}')
check(all(len(solutions[t]) == len(challenges[t]['test']) for t in challenges), 'a solution count differs from the tests')
print(BENCH, '| fnhard27:', len(PARENTS), 'parents,', len(challenges) - len(PARENTS), 'children,', len(OUTPUTS),
      'test outputs:', sum(KIND[t] == 'parent' for t, _ in OUTPUTS), 'parent,', sum(KIND[t] == 'child' for t, _ in OUTPUTS),
      'child; first in queue', families['tasks'][:3])
_only = [t for t in os.environ.get('NIGHT_TASKS', '').split(',') if t]   # as hbv2b1data: a targeted subset
if _only:
    check(set(_only) <= set(challenges), f'NIGHT_TASKS not in the set: {sorted(set(_only) - set(challenges))}')
    OUTPUTS = [(t, i) for t, i in OUTPUTS if t in _only]
    print('NIGHT_TASKS: a targeted subset of', len(_only), 'tasks,', len(OUTPUTS), 'outputs')

In [ ]:
# Draft-model training data (2026-10-25, the decode-3x campaign): ARC-AGI-2 TRAINING tasks only, a seeded random sample
# (docs/reports/1025_1016_fn_decode_3x/make_fntrain_set.py). Weights are trained only on public training tasks, never on
# evaluation tasks or our eval children, so this cell refuses any id from the evaluation set. Every task is a 'parent'.
challenges = json.load(open(f'{BENCH}/fntrain_challenges.json'))
solutions = json.load(open(f'{BENCH}/fntrain_solutions.json'))
families = json.load(open(f'{BENCH}/fntrain_families.json'))
PARENTS = families['parents']
FAMILY = dict(families['family'])
KIND = dict(families['kind'])
OUTPUTS = [(t, i) for t in families['tasks'] for i in range(len(challenges[t]['test']))]
check(set(solutions) == set(challenges) == set(FAMILY) == set(KIND) == set(families['tasks']), 'the set files do not match')
check(all(k == 'parent' for k in KIND.values()), 'fntrain holds training tasks only')
_ev = glob.glob('data/original_competition_data/arc-agi_evaluation_challenges.json')   # the repo copy; never runs on Kaggle
if _ev:
    check(not (set(challenges) & set(json.load(open(_ev[0])))), 'an evaluation task is in the training set')
check(all(len(solutions[t]) == len(challenges[t]['test']) for t in challenges), 'a solution count differs from the tests')
print(BENCH, '| fntrain:', len(challenges), 'training tasks,', len(OUTPUTS), 'test outputs; eval overlap checked:',
      bool(_ev), '; first in queue', families['tasks'][:3])
_only = [t for t in os.environ.get('NIGHT_TASKS', '').split(',') if t]
if _only:
    check(set(_only) <= set(challenges), f'NIGHT_TASKS not in the set: {sorted(set(_only) - set(challenges))}')
    OUTPUTS = [(t, i) for t, i in OUTPUTS if t in _only]
    print('NIGHT_TASKS: a targeted subset of', len(_only), 'tasks,', len(OUTPUTS), 'outputs')

In [ ]:
r5 = await run_batch('r5_v2b1_g4', [(t, i, k) for k in (0, 1) for t, i in OUTPUTS], minutes=ROUND_MINUTES)
summarize(r5)

**Round 5's result (the v2 children of parents #21–#30 on one Colab G4, written 2026-10-21 21:30 IST).** Two attempts
per output, 84 attempts, 123 min. Run `1021_1828_colab-g4-harness-r5-v2b1`.

- **Parents** 16 of 28 attempts right (57 %), **children 30 of 56 (54 %)**: −3 points, inside the ±15 rule. v1's
  children were 27 points easier than their parents; the gap is gone.
- Spent budgets: parents 21 %, children 32 %. Token ratio child/parent: 1.32 over families whose parent finished
  under budget.
- A deep reading of all 84 attempts (kept with the children, as the validation split) found every key consistent with
  its rule. 30 of the 41 failures are the harness's own reasoning. Six children have a rival reading that fits every
  demo; they go back to the eval-children process.

As a validation split the v2 children behave like evaluation tasks. Aggregate only here: per-child results stay in the
children's tree (`data/eval_children/design/_quality/`).

## Round 6: the first frozen set on NVIDIA's NVFP4 checkpoint, one Colab G4 (written 2026-10-21 20:18 IST, before it runs)

**Why.** Colab's G4 can serve NVFP4, which 4 × L4 cannot. Run `1021_1952` measured NVIDIA's NVFP4 checkpoint
(`nvidia/Qwen3.8-27B-NVFP4`: MLP and `lm_head` NVFP4 W4A4, attention and GDN projections FP8) at 1.15× the FP8
checkpoint's decode rate under this harness's load, with a 19 % larger KV pool. **Does the harness behave the same on
it as on the FP8 checkpoint we run on Kaggle?** If it does, G4 rounds on NVFP4 measure our harness faster. If not, they
measure a different model.

**What runs.** Round 4's design exactly: the first frozen set, one attempt per output, 18 in flight, no time cutoff.
The only change is the model (config `rtxpro6000_nvfp4_harness.toml`, which differs from round 4's
`rtxpro6000_harness.toml` in the slug and the model alias only). Round 4 (FP8 on a G4: 21/32, 8 spent, 44 min) is the
pair.

**Prediction:**
- **Pace:** ~38–45 min (round 4: 44).
- **Score and endings:** round 4's within noise, 21 ± 4 right; spent budgets 5–11.
- **Paired with round 4, output by output:** no systematic token difference (sign test p > 0.1); more tokens per
  output at most ~10 % (NVFP4 loop reports).
- **Within noise vs not:** a gap of 5 or more right, or 12 or more spent, would say NVFP4 changes the harness's
  behaviour.

In [ ]:
r6 = await run_batch('r6_g4_nvfp4', [(t, i, 0) for t, i in OUTPUTS], minutes=ROUND_MINUTES)
summarize(r6)

**Round 6's result (NVIDIA's NVFP4 on one Colab G4, written 2026-10-21 21:30 IST).** **21 of 32 right; parents 6 of
12, children 15 of 20**, exactly round 4's on FP8. 8 spent budgets (round 4: 8), 43 min for the round (44). Run
`1021_2018_colab-g4-harness-r6-nvfp4`.

Paired with round 4 output by output: 16 solved in both, and **5 flips each way**, the noise level of these rounds.
Round 6 used more tokens on 17 outputs and fewer on 10 (sign test p = 0.25, median ×1.04). Every prediction held.

Decode at equal load (15–18 running, binned by KV tokens): **1.10–1.19× FP8's tok/s**, step 6–8 ms shorter. The 19 %
larger KV pool kept 18 requests running to the end. The round's wall did not shrink: the last attempts are spent
budgets that run to 131k tokens at a per-stream rate.

So the harness behaves the same on NVIDIA's NVFP4 as on FP8, as far as one 32-output round can tell. NVFP4 rounds suit
qualitative harness work; absolute scores for Kaggle still come from FP8. The fidelity gate's 5.8× KL (`1021_2005`) is
far below what 32 outputs can resolve.

## Round 7: the first frozen set on RedHat's NVFP4 v2, one Colab G4 (written 2026-10-21 21:06 IST, before it runs)

**Why.** The fidelity gate (`1021_2005`) found RedHat's v2 about twice as close to BF16 as NVIDIA's NVFP4 on our long
traces (KL 3.3× FP8's against 5.8×; top-1 −1.2 to −1.6 points against −2.3 to −3.0). It has a BF16 `lm_head`, FP8
per-token-dynamic attention/GDN and FP8 MLP in layers 56–63. **How fast is it under this harness's load, and does the
harness behave on it as on FP8 (round 4) and NVIDIA's NVFP4 (round 6)?**

**What runs.** Rounds 4 and 6's design exactly: the first frozen set, one attempt per output, 18 in flight. The only
change is the model (config `rtxpro6000_nvfp4_redhat_harness.toml`, which differs from round 6's in the slug and the
model alias only).

**Prediction:**
- **Speed:** decode ~1.07–1.10× FP8's: NVFP4 on 56 of 64 MLPs, BF16 head, FP8 projections. That is below round 6's
  NVIDIA build (1.15×). The KV pool lands between FP8's and NVIDIA's.
- **Score and endings:** round 4's within noise, 21 ± 4 right; spent budgets 5–11.
- **Paired with round 4:** no systematic token difference (sign test p > 0.1).
- **Against round 6:** no difference beyond noise. One round each cannot separate builds whose KL differs by 1.8×.

In [ ]:
r7 = await run_batch('r7_g4_nvfp4_redhat', [(t, i, 0) for t, i in OUTPUTS], minutes=ROUND_MINUTES)
summarize(r7)

## Round 8: the first frozen set on RadixArk's NVFP4 with a BF16 `lm_head`, one Colab G4 (written 2026-10-21 22:15 IST, before it runs)

**Why.** The user chose `RadixArk/Qwen3.8-27B-NVFP4-BF16-LMHead` (@009632f) for the G4 wing.
- **Provenance:** it is the SGLang team's own export, the one their cookbook measures every recipe on.
- **Format:** NVIDIA's ModelOpt map, so it takes SGLang's tuned path: FP4 autotune and the fused SiLU+FP4 quant.
- **Head:** a dense BF16 `lm_head`, as in our Kaggle FP8 checkpoint, so the DFlash2 selector projects through the same
  kind of head as in production.
- **Fidelity, independent measurement** (Telperion, vLLM): top-1 agreement 91.06 % against the FP4-head sibling's
  90.23 %. "Certain" flips are unchanged (0.71 % against 0.70 %).

**Does the harness behave on it as on FP8 (round 4) and on NVIDIA's NVFP4 (round 6)?**

**What runs.** Rounds 4, 6 and 7's design exactly: the first frozen set, one attempt per output, 18 in flight. The only
change is the model: config `rtxpro6000_nvfp4_radixark_bf16head_harness.toml`, round 6's config with the model alias
swapped.

**Prediction:**
- **Speed:** the decode step ~2.5 ms above round 6's NVIDIA build, since the BF16 head costs 3.85 ms per step against
  NVFP4's 1.31 (`1021_1952`). So ~1.08–1.10× FP8's at equal KV. The KV pool is ~1.6M tokens, below NVIDIA's 1.68M (the
  dense head is ~3.2 GB at runtime). Accept length equal to NVIDIA's or up to +0.1.
- **Score and endings:** round 4's within noise, 21 ± 4 right; spent budgets 5–11.
- **Paired with rounds 4 and 6:** no systematic token difference (sign test p > 0.1). Flips at the noise level of these
  rounds, ~5 each way.

In [ ]:
r8 = await run_batch('r8_g4_radixark_bf16head', [(t, i, 0) for t, i in OUTPUTS], minutes=ROUND_MINUTES)
summarize(r8)

**Round 8's result (RadixArk's NVFP4 with a BF16 `lm_head`, one Colab G4, written 2026-10-21 23:37 IST).** **19 of 32
right; parents 4 of 12, children 15 of 20**, with 8 spent budgets and 48 min for the round. Run
`1021_2245_colab-g4-harness-r8-radixark-bf16head`.

- **Against rounds 4 (FP8) and 6 (NVIDIA NVFP4)**, both 21/32 with 8 spent: the children are 15/20 in all three, the
  parents 4 against 6.
- **Paired, output by output:** 5 and 3 flips against round 4, 8 and 6 against round 6; tokens p = 0.57 and 0.71,
  ×1.07 and ×1.06.
- **Decode at equal KV:** 1.03–1.11× FP8's tok/s, a little below NVIDIA's (the BF16 head costs ~2.5 ms per step).
- **KV gate:** the smaller pool (1.63M tokens) let the gate park 5 sessions late in the round.

Every prediction held. The harness behaves on this build as on FP8 and NVIDIA's within one round's resolution. With
fidelity gate F2 (the most faithful of the tuned-path NVFP4 builds) and the step profile (`1021_2210`), it carries G4
rounds.

## Vision stage A: host-embed check and the perception bench (written 2026-10-22 01:21 IST, before it runs)

The vision plan is `docs/reports/1022_0010_vision_harness_plan/README.md`. Stage A asks whether Qwen3.8 reads ARC grids
better from larger or annotated pictures, before any harness round spends budget on pictures. The renderer is
`arc_eval/runtime/arc_vision.py`. Its renderings: 32 / 64 / 96 px per cell, rulers with a 5-cell major grid, and digits
in the cells.

**Server:** the G4 harness config with RadixArk's BF16-head NVFP4 checkpoint
(`rtxpro6000_nvfp4_radixark_bf16head_harness.toml`), build `v0.5.21-agi2.19`. That is `.16` plus code gated off by
default. The one flag turned on is host embed (`SGLANG_AGI2_HOST_EMBED=1`), folded in from submission 11's run: the
embedding table moves to pinned host RAM, exact by design, +7.7 % pool on 4 × L4. The cells find the plan folder from
the package (`VIS_PLAN`), because the runner works in its own directory.

**`hbvisgreedy`** runs four text and three picture prompts greedily, and records the KV pool. It runs once with host
embed off and once on, on the same box. Prediction:
- identical completions;
- the pool grows by ~2.5 GB of embedding table, about +78k tokens at 32 KiB per FP8 token on this GPU.

**`hbvisA`** is the perception bench (`perception_bench.py`): 60 grids from the evaluation tasks' training pairs.
- Probes: dimensions, an exact copy, 10 point lookups, cell and object counts, and three structure questions.
- Conditions: the picture alone, the digits alone, and the digits with the picture.
- Seven renderings, thinking off, plus a thinking pass on dimensions and counts for every third grid. 4,620 requests,
  all scored by code.

**Prediction:**
- **Picture alone:** 64–96 px reads dimensions and lookups better than 32 px (10-17's), mostly at 11–30 cells a side.
  Rulers help counting more than size does; digits in cells make copies near-exact up to 20 a side.
- **Digits alone** answer dimensions and lookups near-perfectly, and miss on counts and structure.
- **Digits with a picture:** dimensions and lookups as the digits alone (no harm). Counts and structure +0.05 to
  +0.15, if pictures help at all.

**Decision rule (the plan, section 3):**
- the stage-B rendering is the one with the best digits-with-picture accuracy on lookups, counts and structure per 1k
  visual tokens;
- if no rendering beats digits alone there, pictures stay out of cell-level work, and stage B keeps only the
  scene-description and check arms.

In [ ]:
import hashlib, sys
from pathlib import Path

import agi2_modal_harness
from agi2_modal_harness.workloads.arc_eval.runtime import arc_vision as av

# The plan folder (perception_bench.py, items_a.jsonl), absolute: the runner works in its own directory, not the repo.
VIS_PLAN = Path(os.environ.get('VIS_PLAN') or
                Path(agi2_modal_harness.__file__).resolve().parents[1] / 'docs/reports/1022_0010_vision_harness_plan')
sys.path.insert(0, str(VIS_PLAN))
VIS_LABEL = os.environ.get('VIS_LABEL', 'run')   # e.g. host-embed-off / host-embed-on: names this run's file
VIS_OUT = Path(os.environ.get('VIS_OUT', '.'))
VIS_OUT.mkdir(parents=True, exist_ok=True)


async def greedy_probe():
    """Fixed greedy requests, text and pictures, plus the KV pool: two boots of the same build that differ only in one
    server flag must give identical completions (the host-embed check of the vision plan, stage A)."""
    import aiohttp
    tasks = list(challenges.items())[:4]
    reqs = []
    for tid, t in tasks:   # text: ARC's own prompt, thinking on, 256 tokens
        reqs.append((f'text/{tid}', [{'role': 'user', 'content': build_prompt(t, t['test'][0]['input'])}]))
    for tid, t in tasks[:3]:   # pictures: the first training pair at 64 px with rulers
        p = t['train'][0]
        uri, n = av.picture(p['input'], p['output'], style=av.STYLES['c64r'])
        reqs.append((f'image/{tid}', [{'role': 'user', 'content': [
            {'type': 'text', 'text': 'Describe exactly what this picture shows: the left grid, the right grid, and '
                                     'how they differ.'},
            {'type': 'image_url', 'image_url': {'url': uri}}]}]))
    out = {'label': VIS_LABEL, 'results': []}
    async with aiohttp.ClientSession() as http:
        async with http.get(f'{SERVER}/get_server_info') as r:
            info = await r.json()
        out['max_total_num_tokens'] = info.get('max_total_num_tokens')
        out['host_embed'] = os.environ.get('SGLANG_AGI2_HOST_EMBED')
        out['kv_v4'] = os.environ.get('SGLANG_AGI2_KV_V4')
        for key, msgs in reqs:
            body = {'model': 'qwen3.8-27b', 'messages': msgs, 'temperature': 0.0, 'max_tokens': 256}
            async with http.post(f'{SERVER}/v1/chat/completions', json=body) as r:
                js = await r.json()
            text = js['choices'][0]['message'].get('content') or ''
            reasoning = js['choices'][0]['message'].get('reasoning_content') or ''
            out['results'].append({'key': key, 'sha256': hashlib.sha256((reasoning + '\x00' + text).encode()).hexdigest(),
                                   'prompt_tokens': js.get('usage', {}).get('prompt_tokens'), 'head': (reasoning + text)[:300]})
    path = VIS_OUT / f'greedy_{VIS_LABEL}.json'
    path.write_text(json.dumps(out, indent=1))
    print(f"[greedy] {VIS_LABEL}: pool {out['max_total_num_tokens']:,} tokens, host embed {out['host_embed']}, "
          f"kv-v4 {out['kv_v4']}, {len(out['results'])} completions -> {path}")
    for r in out['results']:
        print(f"  {r['key']:16s} {r['sha256'][:16]}  prompt {r['prompt_tokens']}")


await greedy_probe()

In [ ]:
import importlib
import perception_bench as pb   # VIS_PLAN (the cell above) is on sys.path
importlib.reload(pb)

VIS_ITEMS = Path(os.environ.get('VIS_ITEMS') or VIS_PLAN / 'items_a.jsonl')
VIS_RESULTS = VIS_OUT / 'perception_results.jsonl'
# Stage A: 60 grids x (dims, copy, lookup, count, struct) x (picture, digits, digits + picture) x 7 renderings, plus a
# thinking pass on dims and count for every third grid; at most SLOTS requests in flight (the DFlash sampler's limit)
await pb.run(VIS_ITEMS, VIS_RESULTS, SERVER, think=True, inflight=SLOTS, model='qwen3.8-27b')
pb.score(VIS_RESULTS, VIS_ITEMS)

## The night of 2026-10-22: vision arms on parents #21–#30 and their children

The user, 2026-10-22 02:33 IST: "for the next 8 hours ... keep trying things with vision focus ... on harness and try to
improve the 55ish score both on parents and children of 21-30 ... study the failure points and thus improve the harness
(should be generic not overfitted)". The set is round 5's `v2b1` (10 parents, 14 outputs; 20 v2 children, 28 outputs).
Failures are studied on the parents only; the children are the validation split and are read in aggregate.

### Control N0 (written 2026-10-22 02:47 IST, before it runs)

Round 5's harness unchanged, on tonight's G4 stack: RadixArk's BF16-head NVFP4 checkpoint, build `v0.5.21-agi2.19`,
host embed on. Two attempts per output (84 attempts), as round 5. It is the night's baseline; every arm also runs its own
control beside it, and the controls are pooled.

**Prediction:** parents 50–60 % of attempts right by the pair, children 45–60 % (round 5 on FP8: 57 % and 54 %);
spent budgets ~25–30 % of attempts, almost all with no submission; ~2 h.

In [ ]:
n0 = await run_batch('n0_v2b1_ctrl', [(t, i, k) for k in (0, 1) for t, i in OUTPUTS], minutes=ROUND_MINUTES)
summarize(n0)

### The night's harness: submission 10's, with vision and fresh-context arms (written 2026-10-22 03:20 IST)

Round 5's harness above is submission 6's. The submissions since added budget notes, a final answer turn that works,
tolerant parsing, submit by reference and one conversation per task (its accepted program answers its other tests), and
they dropped pictures. Arms measured on submission 6's harness could re-measure what is already shipped, so the night's
baseline is **submission 10's harness**, loaded from its own notebook's cells (`nightsub10`), and every arm is that
harness with switches (`nightarms`): its `repl_attempt`, patched by exact replacements, and its solve per (arm, sample).

The arms come from two places:
- **Stage A** (`1022_0128`): ruled pictures read well (32 px with rulers: sizes 1.00, lookups 0.84 alone; beside the
  digits they help counting), unruled ones hurt. `v1` puts ruled pictures in the prompt, in `show_image`, in a failing
  submit (cells that differ outlined) and at acceptance.
- **Claude Code's harness patterns** (the user, 03:00: improve our competition harness from Claude Code's design): one
  long context drifts and favours its own answers ("goal drift", "self-preferential bias"); fresh contexts and
  independent verification counter both. `r1` asks for notes at half the budget with no accepted program and goes on in
  a fresh conversation; `f1` checks the first accepted output in a fresh conversation that sees only the task, the
  output and their pictures, and passes an objection to the solver.

Failures are studied on parents only; children are read in aggregate.

In [ ]:
# The night's baseline harness: submission 10's, run from its own notebook's cells, verbatim, on this notebook's server.
# Submission 10 is round 5's harness plus everything sub 7-10 added (budget notes at 50/75/90 %, a final answer turn
# that works, tolerant parsing, submit by reference, the second-answer and malformed-call notes, a task's accepted
# program answering its other test inputs). Its settings are set here as its settings cell sets them.
import ast as _ast
from pathlib import Path
import agi2_modal_harness as _pkg

REPO = Path(_pkg.__file__).resolve().parents[1]   # the runner works in its own directory, not the repo
SUB10_NB = REPO / 'notebooks/1022-sub10-qwen38-xhigh-kv4-all-levers/1022-sub10-qwen38-xhigh-kv4-all-levers.ipynb'
SUB10 = {c['id']: ''.join(c['source']) for c in json.load(open(SUB10_NB, encoding='utf-8'))['cells']}
# FN_BUDGET (2026-10-25, the decode-3x campaign's 1.5x-budget baseline): an attempt's budget in generated tokens over
# all its turns. Unset: 131072, as every round before it. The budget notes (BUDGET_AT) are fractions, so they scale.
HARNESS_TOKENS = harness_tokens = int(os.environ.get('FN_BUDGET') or 131072)
if HARNESS_TOKENS != 131072:
    print('FN_BUDGET: an attempt may generate', HARNESS_TOKENS, 'tokens (131072 before)', flush=True)
BUDGET_AT = (0.5, 0.75, 0.9)
GATE_HIGH, GATE_ROOM = 0.90, 0.05
SHARE = EXACT_TURNS = SESSIONS = True
# Exact turn boundaries need the server's own flags (submission 10's 41aad7bc sets them; round.sh passes them here)
check(all(os.environ.get(k) == '1' for k in ('SGLANG_ENABLE_DFLASH_LENGTH_CAP', 'SGLANG_ENABLE_DFLASH_STOP_CAP',
                                             'SGLANG_ENABLE_STREAMING_SESSION_FOLDED_CLAIM')),
      'submission 10 needs the exact-turn server flags (pass them to round.sh)')
for _cid in ('49f20e6b', '74bf130f', 'cae4f7b8', 'aceb4d00', 'a5509211', 'e8365c62', 'ca2b5fbb', '7a7b1aaa'):
    _src = SUB10[_cid]
    if _src.startswith('%%writefile '):
        _head, _body = _src.split('\n', 1)
        Path(_head.split(None, 1)[1].strip()).write_text(_body + '\n', encoding='utf-8')
        print(f'submission 10 {_cid}: wrote {_head.split(None, 1)[1].strip()} ({len(_body):,} characters)')
        continue
    _r = eval(compile(_src, f'<submission 10 cell {_cid}>', 'exec', flags=_ast.PyCF_ALLOW_TOP_LEVEL_AWAIT), globals())
    if asyncio.iscoroutine(_r):
        await _r
    print(f'submission 10 {_cid}: ran ({len(_src):,} characters)')
check(HARNESS_OK, 'submission 10 harness checks failed (the turn-1 template or the sandbox self-test)')
print('submission 10 harness loaded; KV gate high', GATE.high if GATE else None, '| budget notes at', BUDGET_AT)

In [ ]:
# The night's arms on top of submission 10's harness (cell above). The attempt is submission 10's repl_attempt, patched
# by exact replacements (each must match once), so every arm differs from it only where an arm switch says so; with no
# switches (arm 's10') it is submission 10's attempt. Switches:
#   pics        pictures in the prompt: each training pair (input | output) and the test input, in a style of stage A
#   show        the `show_image(*grids)` tool in python, its pictures drawn in this style
#   fail_pic    a failing submit also shows its first failing example (expected | yours, differing cells outlined)
#   accept_pic  an accepted program's test output is shown as a picture (test input | output) with the review note
#   restart_at  at this fraction of the budget with no accepted program: the model writes notes, and the attempt goes on
#               in a fresh conversation (the task, its notes, a "look again" note; `restart_pics`: pictures in it)
#   verify      at the first acceptance, a fresh conversation (no history: the task and the program's test output,
#               pictured) is asked whether the output follows the examples' rule; its objection, if any, goes back to the
#               solver with the review note (an adversarial check without the solver's own bias)
#   tail        (in night_round, 10-22 midday) end-of-run second attempts: once no first attempt is waiting, a task whose
#               first pass left an output unaccepted or accepted late (>= TAIL_LATE tokens, no alternative) gets an
#               independent attempt (the arm's switches, a full budget, not shown the first answer), placed by tail_place
import importlib
from collections import Counter
from agi2_modal_harness.workloads.arc_eval.runtime import arc_vision as av
importlib.reload(av)


def picture(*grids, style='c32r', marks=None):
    """Grids side by side as (PNG data URI, visual tokens), in one of stage A's styles."""
    return av.picture(*grids, style=av.STYLES[style], marks=marks)


PIC_GUIDE = ('Each grid is drawn with its row and column numbers in the margins and a heavier line every 5 cells; '
             'the colours are ' + av.COLOURS + '. The digits above are exact; the pictures show shapes, objects and '
             'layout.')
SHOW_LINE = ('\nIn `python`, `show_image(*grids)` shows you up to 3 grids side by side as a picture (row and column '
             'numbers in the margins), which comes back with the call\'s output: use it to look at a grid, your '
             'program\'s output, or a comparison.')
FAIL_PIC = ('\n[harness] Example {n} as a picture: expected output left, your program\'s output right; the cells that '
            'differ are outlined in white in both: ' + IMG)
ACCEPT_PIC = '\nThe test input and your program\'s output on it as a picture (input left, output right): ' + IMG
NOTES_REQ = ('[harness] Pause here: {used:,} of the {total:,} tokens for this puzzle are used and no program reproduces '
             'every example yet. Write short notes for a fresh start, at most 250 words and no tool calls: (1) facts '
             'about the examples you checked in code; (2) each reading of the rule you tried, with the evidence that '
             'ruled it out; (3) ideas you have not tried. This conversation then starts again from the task with only '
             'your notes.')
NOTES_FACTS = ('[harness] Pause here: {used:,} of the {total:,} tokens for this puzzle are used and no program reproduces '
               'every example yet. Write notes for a fresh start, at most 250 words and no tool calls, with FACTS ONLY, '
               'no interpretations: (1) facts about the examples you checked in code (sizes, counts, positions, which '
               'cells change and which stay, what each output has that its input lacks); (2) each reading you tried and '
               'the specific example and cells that refuted it. Leave out your current explanation of the puzzle. This '
               'conversation then starts again from the task with only these notes.')
RESTART_NOTE = ('\n\n[harness] A fresh start of this attempt, with {left:,} tokens left. Your notes from the first part:\n'
                '{notes}\n\nThe readings in these notes gave no program that reproduces every example. Look at the '
                'examples again as if for the first time{pictures}: describe what changes from each input to its '
                'output, then look for a different reading. Your python session (its variables and functions) is '
                'still there.')
VERIFY_TASK = ('\n\nA solver wrote a program that reproduces every training example exactly. Its output on the test '
               'input is:\n{grid}\n{picline}Does this output follow the same rule as the training examples? Check it '
               'against each example: what the examples always keep, change, count or align, and what the test input '
               'shows that no training input does. Answer with VERDICT: CONSISTENT, or VERDICT: SUSPECT followed by '
               'the specific cells or features that look wrong and the reading of the rule that would change them.')
VERIFY_FORCE = ('Give your verdict now, in one line and nothing else: VERDICT: CONSISTENT, or VERDICT: SUSPECT followed by '
                'the cells or features that look wrong.')
VERIFY_NOTE = ('\n\n[harness] An independent check, made in a fresh conversation that saw only the task and this output, '
               'finds it SUSPECT:\n{why}\nWeigh this: if it names a reading that fits every training example and '
               'changes the test output, submit that program as your second answer.')


def night_pics_block(task, test_input, style, change=False):
    """Every training pair (input | output) and the test input as pictures, with their placeholders. With `change`, a
    pair whose input and output have one size gets the cells that change outlined in white in both."""
    pics, lines = [], []
    for i, p in enumerate(task['train']):
        cells = av.diff_cells(p['input'], p['output']) if change else set()
        marks = [{'cells': cells, 'colour': (255, 255, 255)}] * 2 if cells else None
        pics.append(picture(p['input'], p['output'], style=style, marks=marks))
        lines.append(f'Example {i} as a picture (input left, output right'
                     + ('; the cells that change are outlined in white' if cells else '') + f'): {IMG}')
    pics.append(picture(test_input, style=style))
    lines.append(f'The test input as a picture: {IMG}')
    return '--Pictures--\n' + PIC_GUIDE + '\n' + '\n'.join(lines) + '\n--End of Pictures--\n\n', pics


def night_conv(http, task, test_input, budget, arm, extra=''):
    """The attempt's conversation: submission 10's prompt, with the arm's pictures and tool line, and `extra` after."""
    tail = repl_tail(budget) + (SHOW_LINE if arm.get('show') else '') + extra
    pics = []
    if arm.get('pics'):
        block, pics = night_pics_block(task, test_input, arm['pics'], arm.get('pics_change', False))
        tail = block + tail
    if arm.get('open_note'):   # the second look's note, last; its picture (if any) is the last placeholder in it
        tail += arm['open_note']
        if arm.get('open_pic'):
            pics = pics + [picture(*arm['open_pic'], style=arm.get('pics') or 'c32r')]
    return SessionConv(http, task_parts(task, test_input, HEAD, tail), pics)


GESTALT_TASK = ('Here are the training examples of a puzzle as pictures (each input left, its output right) and its test '
                'input. ' + PIC_GUIDE.split('. The digits above')[0] + '. Do not solve it in full. Look carefully and '
                'describe, concretely (positions, counts, shapes, colours): the objects in the inputs; what stays the '
                'same and what changes from each input to its output; and two or three different candidate rules that '
                'could explain every example. Keep it under 300 words.\n')
GESTALT_NOTE = ('\n\n[harness] A first look at the pictures, written by a separate pass that saw only the pictures, not '
                'the digits (it may be wrong; the digits are exact):\n{desc}')


async def night_gestalt(http, task, test_input, arm, rec):
    """A fresh conversation that sees only ruled pictures (no digits) and describes the task: returns (text, tokens);
    its record goes into `rec` (a dict the attempt merges into its own)."""
    style = arm.get('gestalt_pics', 'c32r')
    pics, lines = [], []
    for i, p in enumerate(task['train']):
        pics.append(picture(p['input'], p['output'], style=style))
        lines.append(f'Example {i}: {IMG}')
    pics.append(picture(test_input, style=style))
    lines.append(f'Test input: {IMG}')
    conv = SessionConv(http, GESTALT_TASK + '\n'.join(lines), pics)
    desc, used = '', 0
    try:
        g = await conv.turn(arm.get('gestalt_tokens', 12288))
        used = g['tokens']
        desc = (g['content'] or '').strip()[:3000]
        if not desc:   # 1022_0454: a look can spend its whole turn thinking (as f1's checks did): one forced answer turn
            conv.user('Write your description now, in at most 300 words, from what you have seen.')
            g2 = await conv.turn(1536, thinking=False)
            used += g2['tokens']
            desc = (g2['content'] or '').strip()[:3000]
            rec['gestalt_forced'] = True
    except Exception as e:  # noqa: BLE001
        rec['gestalt_error'] = f'{type(e).__name__}: {e}'[:200]
    with contextlib.suppress(Exception):
        await asyncio.wait_for(conv.close(), 30)
    rec['gestalt_tokens'] = used
    rec['gestalt'] = desc[:1500]
    return desc, used


async def night_open(http, task, test_input, budget, arm):
    """The attempt's first conversation, after the optional picture-first look (g1): (conv, tokens the look used, what
    the attempt's record gets)."""
    info = {}
    if not arm.get('gestalt'):
        return night_conv(http, task, test_input, budget, arm), 0, info
    desc, used = await night_gestalt(http, task, test_input, arm, info)
    extra = GESTALT_NOTE.format(desc=desc) if desc else ''
    return night_conv(http, task, test_input, budget - used, arm, extra=extra), used, info


FLAGS_NOTE = ('\n\n[harness] Signs to weigh in this review, found by running your program (they do not mean it is wrong, '
              'but no training example checks these):\n{notes}\nAsk whether the training examples really decide what '
              'your program does there, and what another reading would do instead.')


async def night_submit(sb, code, arm):
    """Submission 10's Sandbox.submit, with the sandbox's own overfit checks on an accepted program when `flags`."""
    if not arm.get('flags'):
        return await sb.submit(code)
    r = await sb._call('submit', code, overfit=True)
    sb.pics = r.get('pictures') or []
    if r.get('error'):
        return clip(r.get('out', '') + r['error']), None
    text = (r['out'].strip() + '\n\n' if r['out'].strip() else '') + r['feedback']
    return clip(text), r


def night_flag_note(out, res, arm, rec):
    notes = (res or {}).get('overfit') or []
    if not arm.get('flags') or not notes:
        return out
    rec['flagged'] = notes[:6]
    return out + FLAGS_NOTE.format(notes='\n'.join(f'- {n}' for n in notes[:6]))


EXEC_REVIEW = ('\n\n[harness] Before you finish, test at least one alternative in code: write the strongest different '
               'reading you considered or set aside (the one that would change the test output most) as a second '
               'transform in `python`, run it on every training example, and print how many it reproduces and its output '
               'on the test input. A reading is ruled out only by a training example it fails, not by an argument. If it '
               'reproduces every example and changes the test output, submit it as your second answer.')


def night_exec_review(out, arm, rec):
    """The executed review (e1): the review note asks for the strongest set-aside reading as code, run on the examples
    (N1_r5parents: the review rescued 0 of 4 wrong acceptances; each dismissed the right alternative in prose)."""
    if not arm.get('exec_review'):
        return out
    rec['exec_review'] = True
    return out + EXEC_REVIEW


def night_shown(sb, out, arm, rec):
    if not arm.get('show') or not sb.pics:
        return out, []
    shown = [picture(*grids, style=arm['show']) for grids in sb.pics]
    rec['shown'] = rec.get('shown', 0) + len(shown)
    return out + ''.join(f'\n[picture {k + 1}] {IMG}' for k in range(len(shown))), shown


def night_fail_pic(out, shown, res, arm, rec):
    fp = res.get('fail_pair') if res else None
    if not arm.get('fail_pic') or not fp:
        return out, shown
    n, want, got = fp
    if not (kaggle_grid(want) and kaggle_grid(got)):
        return out, shown
    cells = av.diff_cells(want, got)
    marks = [{'cells': cells, 'colour': (255, 255, 255)}] * 2 if cells else None
    rec['fail_pics'] = rec.get('fail_pics', 0) + 1
    return out + FAIL_PIC.format(n=n), shown + [picture(want, got, style=arm['fail_pic'], marks=marks)]


def night_accept_pic(out, shown, test_input, grid, arm, rec):
    if not arm.get('accept_pic') or not kaggle_grid(grid):
        return out, shown
    rec['accept_pics'] = rec.get('accept_pics', 0) + 1
    return out + ACCEPT_PIC, shown + [picture(test_input, grid, style=arm['accept_pic'])]


async def night_restart(http, conv, task, test_input, budget, left, arm, rec, log):
    """Notes in the old conversation (thinking off), then a fresh conversation with the task and the notes."""
    conv.drop_notes()
    conv.user((NOTES_FACTS if arm.get('notes') == 'facts' else NOTES_REQ).format(used=budget - left, total=budget))
    used, notes = 0, ''
    try:
        g = await conv.turn(min(3072, max(256, left - 512)), thinking=False)
        used = g['tokens']
        notes = (g['content'] or g['reasoning']).strip()[:4000]
    except Exception as e:  # noqa: BLE001 - no notes: the fresh start still happens
        log(turn='notes', note=f'error: {type(e).__name__}: {e}'[:300])
    with contextlib.suppress(Exception):
        await asyncio.wait_for(conv.close(), 30)
    a2 = dict(arm, pics=arm.get('restart_pics') or arm.get('pics'))
    words = ', starting from the pictures' if a2.get('pics') else ''
    new = night_conv(http, task, test_input, left - used, a2,
                     extra=RESTART_NOTE.format(left=left - used, notes=notes or '(none)', pictures=words))
    rec['restarts'] = rec.get('restarts', 0) + 1
    rec['notes'] = notes[:2000]
    log(turn='restart', tokens=used, notes=notes[:1500])
    return new, 0, used


async def night_verify(http, task, test_input, grid, arm, rec, log):
    """A fresh conversation, thinking on with a bounded budget: is this test output consistent with the examples?"""
    picline, pics = '', []
    style = arm.get('verify_pics')
    if style:
        block, pics = night_pics_block(task, test_input, style)
        pics = pics + [picture(test_input, grid, style=style)]
        picline = block + 'The test input and this output as a picture (input left, output right): ' + IMG + '\n'
    rows = '\n'.join(json.dumps(r) for r in grid)
    conv = SessionConv(http, task_parts(task, test_input, HEAD, VERIFY_TASK.format(grid=rows, picline=picline)), pics)
    why, used, forced = '', 0, False
    try:
        g = await conv.turn(arm.get('verify_tokens', 16384))
        used = g['tokens']
        text = g['content'] or ''
        if 'VERDICT' not in text.upper() and arm.get('verify_force', True):
            # 1022_0301 (f1): 6 of 9 checks spent the whole turn thinking and wrote no verdict, read as consistent.
            # As submission 10's final answer turn: thinking off, one line.
            conv.user(VERIFY_FORCE)
            g2 = await conv.turn(1024, thinking=False)
            used += g2['tokens']
            text, forced = g2['content'] or '', True
        up = text.upper()
        if 'VERDICT: SUSPECT' in up or 'VERDICT:SUSPECT' in up:
            why = text[up.index('SUSPECT') + len('SUSPECT'):].strip()[:1500] or '(no reason given)'
    except Exception as e:  # noqa: BLE001
        log(turn='verify', note=f'error: {type(e).__name__}: {e}'[:300])
    with contextlib.suppress(Exception):
        await asyncio.wait_for(conv.close(), 30)
    rec['verify_tokens'] = rec.get('verify_tokens', 0) + used
    rec['verify'] = 'suspect' if why else 'consistent'
    rec['verify_forced'] = forced
    log(turn='verify', tokens=used, verdict=rec['verify'], forced=forced, why=why[:800])
    return why, used


# submission 10's attempt, patched: each (old, new) must match exactly once
_PATCHES = [
    ('async def repl_attempt(http,', 'async def night_attempt(http,'),
    ('siblings=(), on_shared=None):', 'siblings=(), on_shared=None, arm=None):'),
    ('    marks = sorted(BUDGET_AT)                # fractions of the budget still to announce\n',
     '    marks = sorted(BUDGET_AT)                # fractions of the budget still to announce\n'
     '    arm, restarted = arm or {}, False\n'),
    ('    conv = SessionConv(http, task_parts(task, test_input, HEAD, repl_tail(budget)))',
     '    conv, _g_used, _g_info = await night_open(http, task, test_input, budget, arm)\n'
     '    left -= _g_used'),
    ('               calls_in_reasoning=0, bare_calls=0, malformed=0, budget_notes=0, same_noted=False, final_from=None)\n',
     '               calls_in_reasoning=0, bare_calls=0, malformed=0, budget_notes=0, same_noted=False, final_from=None)\n'
     '    rec.update(_g_info)\n'),
    ('                room = CONTEXT - ctx - 1024          # how far the conversation can still grow\n',
     '                if (arm.get(\'restart_at\') and not restarted and pending is None\n'
     '                        and budget - left >= arm[\'restart_at\'] * budget and left > 8192):\n'
     '                    restarted = True\n'
     '                    conv, ctx, _used = await night_restart(http, conv, task, test_input, budget, left, arm, rec, log)\n'
     '                    left -= _used\n'
     '                room = CONTEXT - ctx - 1024          # how far the conversation can still grow\n'),
    ('                g = await conv.turn(min(left, room))\n',
     "                g = await conv.turn(min(left, room, arm.get('turn_cap') or left))\n"),
    ('                        name, code = call_code(c)\n',
     '                        name, code = call_code(c)\n'
     '                        shown = []\n'),
    ('                            out = await sb.run(code)\n',
     '                            out = await sb.run(code)\n'
     '                            out, shown = night_shown(sb, out, arm, rec)\n'),
    ('                            out, res = await sb.submit(code)\n',
     '                            out, res = await night_submit(sb, code, arm)\n'),
    ("                            rec['submits'].append(sub)\n",
     "                            rec['submits'].append(sub)\n"
     "                            if res and res['n_ok'] < res['n']:\n"
     "                                out, shown = night_fail_pic(out, shown, res, arm, rec)\n"),
    ("                                    out += review_note(test_input, sub['test'])\n",
     "                                    out += review_note(test_input, sub['test'])\n"
     "                                    out = night_flag_note(out, res, arm, rec)\n"
     "                                    out = night_exec_review(out, arm, rec)\n"
     "                                    out, shown = night_accept_pic(out, shown, test_input, sub['test'], arm, rec)\n"
     "                                    if arm.get('verify'):\n"
     "                                        _why, _used = await night_verify(http, task, test_input, sub['test'], arm, rec, log)\n"
     "                                        left -= _used\n"
     "                                        if _why:\n"
     "                                            out += VERIFY_NOTE.format(why=_why)\n"),
    ('                        conv.tool(c, out)\n'
     '                        ctx += len(out) + 16   # grids print about one token per character\n',
     '                        conv.tool(c, out, shown)\n'
     '                        ctx += len(out) + 16 + sum(n for _, n in shown)   # grids print about one token per character\n'),
]


def night_source(src):
    for old, new in _PATCHES:
        check(src.count(old) == 1, f'night patch does not match exactly once ({src.count(old)}): {old[:80]!r}')
        src = src.replace(old, new)
    return src


_NIGHT_SRC = night_source(SUB10['7a7b1aaa'])
night_attempt = None   # bound by the exec below (named here so the runner's preflight sees it defined)
exec(compile(_NIGHT_SRC, '<night_attempt: submission 10 patched>', 'exec'), globals())
check(callable(night_attempt), 'night_attempt was not built')
print('night_attempt built:', len(_NIGHT_SRC) - len(SUB10['7a7b1aaa']), 'characters of patches over submission 10')

ARMS = {
    's10': {},                                                                       # submission 10 as it is
    'v1': {'pics': 'c32r', 'show': 'c32r', 'fail_pic': 'c32r', 'accept_pic': 'c32r'},   # ruled pictures everywhere
    'r1': {'restart_at': 0.5},                                                      # notes, then a fresh conversation
    'r1v': {'restart_at': 0.5, 'restart_pics': 'c32r'},                              # the same, the restart pictured
    'f1': {'verify': True, 'verify_pics': 'c32r'},                                   # a fresh-context check, pictured
    'f1t': {'verify': True},                                                         # the same check, text only
    'c1': {'turn_cap': 24576},                                                       # a per-turn thinking cap
    'g1': {'gestalt': True, 'gestalt_pics': 'c32r'},                                 # a picture-only first look
    'o1': {'flags': True},                                                           # test-novelty notes at acceptance
    'e1': {'exec_review': True},                                                     # the review runs an alternative
    'r2v': {'restart_at': 0.5, 'restart_pics': 'c32r', 'notes': 'facts'},            # restart with facts-only notes
    'f2': {'verify': True, 'verify_pics': 'c32r', 'verify_tokens': 24576},           # f1 with a forced verdict
    'v2': {'pics': 'c32r', 'pics_change': True, 'show': 'c32r', 'fail_pic': 'c32r', 'accept_pic': 'c32r'},
    'a2': {'second_look': True, 'second_after': 40000, 'second_fixed': True},        # attempt_2 from a second look
    'a2c': {'second_look': True, 'second_after': 40000, 'second_fixed': True, 'second_turn_cap': 16384},  # capped turns
    'tail': {'tail': True},                                                          # end-of-run second attempts
}


def arm_of(name):
    """An arm by name; 'x+y' merges arms x and y (a later part wins a shared switch)."""
    merged = {}
    for part in name.split('+'):
        check(part in ARMS, f'no arm {part!r} in {name!r}')
        merged.update(ARMS[part])
    return merged


SECOND_NOTE = ('\n\n[harness] One answer for this test input is already recorded. It comes from a program that '
               'reproduces every training example:\n{grid}\n{picline}You are making the second of the two allowed '
               'attempts. It counts only if the recorded answer is wrong, so a program with the same test output adds '
               'nothing. Look for the most plausible different reading of the examples: one that also reproduces every '
               'training example but gives a different test output. Places to look: what the test input has that no '
               'training input shows, and choices the examples leave open. Submit it with `submit`; a program is '
               'accepted only if it reproduces every example.')
SECOND_PIC = 'The test input and the recorded answer as a picture (input left, answer right): ' + IMG + '\n'


async def night_second(http, task, i, rec, arm, a1, live, fill):
    """The second look (wave 3, 1022 05:30): wave 1 left attempt_2 empty in 89 % of s10's attempts, while accepted
    attempts ended with a median ~68k of 131k tokens unused. A fresh conversation gets that unused budget (capped), the
    task, ruled pictures and the recorded answer, and looks for a different reading that still reproduces every example.
    `fill(m, grid)` puts a grid into test m's attempt_2 only (when empty and different from its attempt_1), so attempt_1
    never changes. Returns the second look's record (with its texts), or None when skipped."""
    used = sum(q['tokens'] for q in rec.get('requests', [])) + rec.get('verify_tokens', 0) + rec.get('gestalt_tokens', 0)
    left, cap = HARNESS_TOKENS - used, arm.get('second_cap', 49152)
    # `second_fixed`: the cap whatever the attempt left (wave 3 measures the hit rate; on Kaggle the look would run in
    # the run's idle tail, not inside one attempt's budget). Tokens beyond `left` are extra cost, counted in the report.
    budget = cap if arm.get('second_fixed') else min(left, cap)
    info = rec['second_look'] = dict(budget=budget, used=used, left=left)
    if used < arm.get('second_after', 0):   # wave 1: s10's acceptances within 40k tokens were right 21 of 21
        info['skipped'] = 'early'
        return None
    if budget < arm.get('second_min', 16384):
        info['skipped'] = 'budget'
        return None
    style = arm.get('second_pics', 'c32r')
    note = SECOND_NOTE.format(grid='\n'.join(json.dumps(r) for r in a1), picline=SECOND_PIC if style else '')
    a2 = dict(open_note=note, **({'pics': style, 'open_pic': (task['test'][i]['input'], a1)} if style else {}))
    if arm.get('second_turn_cap'):   # 08:04: an uncapped look thought 49k tokens in one turn and left no answer
        a2['turn_cap'] = arm['second_turn_cap']
    sib = [m for m in range(len(task['test'])) if m != i]
    accs = []   # [test grid, sibling grids] per acceptance in the second look

    def on_acc(g):
        accs.append([g, None])

    def on_sh(grids):
        if accs:
            accs[-1][1] = grids

    r2 = await night_attempt(http, task, task['test'][i]['input'], budget, live=live, on_accept=on_acc, on_best=None,
                             siblings=[task['test'][m]['input'] for m in sib], on_shared=on_sh if sib else None, arm=a2)
    texts2 = r2.pop('texts', None)
    info.update(tokens=sum(q['tokens'] for q in r2.get('requests', [])), verified=bool(r2.get('verified')),
                turns=r2.get('turns'), source=r2.get('source'), error=r2.get('error'))
    pick = next((g for g in (r2.get('answer'), r2.get('second')) if g is not None and kaggle_grid(g) and g != a1), None)
    info['differs'] = pick is not None
    info['filled'] = bool(pick is not None and fill(i, pick))
    if pick is not None:
        grids = next((s for g, s in accs if g == pick and s), None)
        if grids:
            info['siblings_filled'] = sum(bool(fill(m, s)) for m, s in zip(sib, grids))
    return dict(r2, texts=texts2)


# The tail (10-22 midday; docs/reports/1022_0936_night_harness, "Morning follow-up"). Over 8 s10 samples, a second
# independent attempt rescued 18.3 % of the outputs whose first attempt had no accepted program (24 % of tasks) and
# 11.6 % of those accepted after >= 70k tokens (34 %), against 1.2 % of those accepted under 40k: the tail runs on the
# first two classes only, and only once no first attempt is waiting (on Kaggle a second attempt must never take a
# slot from a task with none). A look shown the first answer rescued ~10 %, so the tail is independent of it.
TAIL_LATE = 70000   # main-attempt tokens at or above which an accepted first attempt counts as late
ROUND_POLL_S = 20   # how often an idle worker of night_round looks for a job again


def tail_class(rec):
    """A first attempt's class for the tail: 'unaccepted' (no program reproduced every example), 'late' (accepted
    after TAIL_LATE main-attempt tokens or more, with no alternative program), else None (no tail)."""
    if not rec or rec.get('unfinished'):
        return None
    if not rec.get('verified'):
        return 'unaccepted'
    if (rec.get('tokens') or 0) >= TAIL_LATE and not rec.get('alternative'):
        return 'late'
    return None


def pair_of(grids):
    """The first two different Kaggle grids in `grids`, as [attempt_1, attempt_2] (None where there is none)."""
    out = []
    for g in grids:
        if g is not None and kaggle_grid(g) and g not in out:
            out.append(g)
    return (out + [None, None])[:2]


def tail_place(cls, cur, best, ta, ts, tv):
    """One output's [attempt_1, attempt_2] after a tail, and the rule used. `cls`: its first pass's tail_class; `cur`:
    its pair now; `best`: its first attempt's best failing submission; `ta`: the tail's answer for it; `ts`: the tail's
    accepted alternative for it (or None); `tv`: whether an accepted program gave `ta`."""
    if cls == 'unaccepted' and tv:   # the tail found a program: its answer first, the old first answer second
        return pair_of([ta, cur[0], ts, cur[1]]), 'swap'
    if cls == 'late':                # accepted late: the tail's answer, accepted or not, second if it differs
        return pair_of([cur[0], ta, ts, cur[1]]), 'second'
    if cls == 'unaccepted' and (cur[1] is None or cur[1] == best):   # neither accepted: only an empty or `best` slot
        return pair_of([cur[0], ta, ts, cur[1]]), 'fill'
    return list(cur), 'kept'


async def night_round(name, arms, samples=(0,), minutes=ROUND_MINUTES, slots=SLOTS):
    """Submission 10's solve (one conversation per task, its accepted program answering the task's other tests,
    set_answers' attempt_1 / attempt_2) for each (arm, sample) on this notebook's set, all in one pool of `slots`.

    Arms with `tail` then get second attempts: a task whose first pass left an output unaccepted or accepted late
    (tail_class) queues one tail job, which runs only when no first attempt is waiting and none of its task runs. A
    tail is a fresh attempt on the lowest such output, with the task's other such outputs as its siblings (one
    conversation per task, as in the first pass); tail_place places its answers. An output an accepted tail program
    answered is settled; if some are not, the next tail runs on one not yet tried as a tail's own test. Tail attempts
    go to tail_records.jsonl (and into `done` with tail=True); outputs rows keep the first pass (answers_first_pass)."""
    out = f'{OUT}/{name}'
    for d in ('attempts', 'live'):
        os.makedirs(f'{out}/{d}', exist_ok=True)
    reps = [(a, k) for k in samples for a in arms]
    tasks = list(dict.fromkeys(t for t, _ in OUTPUTS))
    ntest = {t: len(challenges[t]['test']) for t in tasks}
    queue = [(a, k, t, 0) for t in tasks for a, k in reps] + \
            [(a, k, t, i) for t in tasks for i in range(1, ntest[t]) for a, k in reps]
    cands, answers, shared, active, done, running = {}, {}, {}, Counter(), [], {}
    # the tail: its jobs (a, k, t, 'tail'), per task replica its state, each first attempt's record (by aid), each
    # output's pair before a tail changed it, and how many first attempts run (each may still queue a tail)
    tails, tail_state, first_rec, first_pass, nfirst = [], {}, {}, {}, [0]
    t_round = time.time()
    stop = t_round + minutes * 60
    print(f'{name}: arms {arms}, samples {list(samples)}: {len(queue)} outputs to answer, {slots} in flight, '
          f'safety stop at {time.strftime("%H:%M", time.gmtime(stop + 19800))} IST', flush=True)

    def set_answers(key):
        c = cands.get(key, {})
        acc = c.get('accepted', [])
        own = c.get('answer')
        first = own if own is not None and (c.get('verified') or not acc) else (acc[0] if acc else None)
        grids = []
        for g in [first, c.get('second')] + acc + [own, c.get('best')]:
            if g is not None and kaggle_grid(g) and g not in grids:
                grids.append(g)
        answers[key] = (grids + [None, None])[:2]

    def write(rec, texts=None):
        done.append(rec)
        with open(f'{out}/records.jsonl', 'a') as f:
            f.write(json.dumps(rec) + '\n')
        if texts is not None:
            with open(f"{out}/attempts/{rec['aid']}.json", 'w') as f:
                json.dump(dict(rec, texts=texts), f)

    def write_tail(rec, texts=None):
        done.append(rec)
        with open(f'{out}/tail_records.jsonl', 'a') as f:
            f.write(json.dumps(rec) + '\n')
        if texts is not None:
            with open(f"{out}/attempts/{rec['aid']}.json", 'w') as f:
                json.dump(dict(rec, texts=texts), f)

    def out_cls(a, k, t, m):
        """The tail class of output m of replica (a, k) of task t: that of the first attempt that set its first answer
        (the task's attempt whose accepted program answered it, else its own attempt); None if it never had one."""
        own = shared.get((a, k, t, m)) or f'{t}_{m}_{a}_s{k}'
        return tail_class(first_rec.get(own))

    def has_job():
        queue[:] = [j for j in queue if j not in shared]   # outputs their task's accepted program already answered
        if any(not active[j[:3]] for j in queue):
            return True
        return not queue and any(not active[j[:3]] for j in tails)

    def next_job():
        queue[:] = [j for j in queue if j not in shared]
        for n, j in enumerate(queue):
            if not active[j[:3]]:   # wait while the same replica's attempt on its task runs (it may answer it)
                return queue.pop(n)
        if queue:   # first attempts still wait (behind their task's running attempt): no tail before them
            return None
        for n, j in enumerate(tails):
            if not active[j[:3]]:   # a tail waits while any attempt of its task replica runs
                return tails.pop(n)
        return None

    async def run_one(http, j):
        a, k, t, i = j
        aid, task, t0 = f'{t}_{i}_{a}_s{k}', challenges[t], time.time()
        c = cands.setdefault(j, {})
        sib = [(a, k, t, m) for m in range(ntest[t]) if m != i and not cands.get((a, k, t, m), {}).get('accepted')]

        def on_shared(grids):
            for s, g in zip(sib, grids):
                if g is not None and kaggle_grid(g):
                    cs = cands.setdefault(s, {})
                    if g not in cs.setdefault('accepted', []):
                        cs['accepted'].append(g)
                    shared.setdefault(s, aid)
                    set_answers(s)

        def on_accept(g):
            if g not in c.setdefault('accepted', []):
                c['accepted'].append(g)
                set_answers(j)

        def on_best(g):
            c['best'] = g
            set_answers(j)

        running[aid] = t0
        active[(a, k, t)] += 1
        base = dict(aid=aid, arm=a, sample=k, task=t, test=i, kind=KIND[t], family=FAMILY[t], start_s=round(t0 - t_round))
        try:
            rec = await night_attempt(http, task, task['test'][i]['input'], HARNESS_TOKENS, live=f'{out}/live/{aid}.jsonl',
                                      on_accept=on_accept, on_best=on_best,
                                      siblings=[task['test'][m]['input'] for *_, m in sib], on_shared=on_shared,
                                      arm=arm_of(a))
        except asyncio.CancelledError:
            write(dict(base, unfinished=True, seconds=round(time.time() - t0)))
            raise
        except Exception as e:  # noqa: BLE001 - one failed attempt must not stop the round
            rec = dict(error=f'{type(e).__name__}: {e}'[:300], requests=[], submits=[])
        finally:
            running.pop(aid, None)
            active[(a, k, t)] -= 1
        texts = rec.pop('texts', None)
        c.update(answer=rec.get('answer'), second=rec.get('second'), verified=bool(rec.get('verified')),
                 source=rec.get('source'))
        set_answers(j)
        sol = solutions[t][i]
        arm, stopped = arm_of(a), None
        if (arm.get('second_look') and rec.get('verified') and not rec.get('error')
                and (answers.get(j) or [None, None])[1] is None):

            def fill(m, g):
                key = (a, k, t, m)
                cur = answers.get(key) or [None, None]
                if g is None or not kaggle_grid(g) or cur[1] is not None or g == cur[0]:
                    return False
                cands.setdefault(key, {})['second'] = g
                set_answers(key)
                return (answers.get(key) or [None, None])[1] == g

            running[aid + '_2nd'] = time.time()
            try:
                r2 = await night_second(http, task, i, rec, arm, answers[j][0], f'{out}/live/{aid}_2nd.jsonl', fill)
                if r2 is not None:
                    with open(f'{out}/attempts/{aid}_2nd.json', 'w') as f:
                        json.dump(r2, f)
            except asyncio.CancelledError as e:   # the round's stop: keep the main attempt's record
                rec.setdefault('second_look', {})['cancelled'] = True
                stopped = e
            except Exception as e:  # noqa: BLE001 - a failed second look leaves attempt_2 as it was
                rec.setdefault('second_look', {})['error'] = f'{type(e).__name__}: {e}'[:300]
            finally:
                running.pop(aid + '_2nd', None)
        rec.update(base, seconds=round(time.time() - t0), answers=answers.get(j),
                   solved=answers.get(j, [None])[0] == sol, pair_solved=sol in (answers.get(j) or []),
                   tokens=sum(q['tokens'] for q in rec.get('requests', [])),
                   prefill_new=sum(q['prompt'] - q['cached'] for q in rec.get('requests', [])),
                   n_submits=len(rec.get('submits', [])), first_accept_solved=rec.get('first_accept') == sol
                   if rec.get('first_accept') else None)
        write(rec, texts)
        first_rec[aid] = {k2: rec.get(k2) for k2 in ('verified', 'tokens', 'alternative', 'error')}
        if arm.get('tail') and stopped is None and tail_class(rec) and (a, k, t) not in tail_state:
            tail_state[(a, k, t)] = dict(tried=set(), settled=set())   # one tail job per task replica at a time
            tails.append((a, k, t, 'tail'))
        print(time.strftime('%H:%M:%S'), json.dumps({k2: rec.get(k2) for k2 in (
            'aid', 'kind', 'solved', 'pair_solved', 'verified', 'source', 'tokens', 'turns', 'n_submits', 'spent',
            'committed', 'final_from', 'restarts', 'verify', 'shown', 'fail_pics', 'second_look', 'error')
            if KIND[t] == 'parent' or k2 not in ('solved', 'pair_solved')}),   # children: the validation split
              f'| tail class {tail_class(rec)}' if arm.get('tail') else '', flush=True)
        if stopped is not None:
            raise stopped

    async def run_tail(http, j):
        """One tail job (a, k, t, 'tail'): a fresh attempt on the task's lowest output that needs one and was not yet a
        tail's own test, the task's other outputs that need one as its siblings; then tail_place on each."""
        a, k, t, _ = j
        st, task = tail_state[(a, k, t)], challenges[t]
        need = [m for m in range(ntest[t]) if out_cls(a, k, t, m) and m not in st['settled']]
        free = [m for m in need if m not in st['tried']]
        if not free:   # e.g. a later attempt's accepted program answered it early: nothing needs a tail any more
            return
        i = free[0]
        st['tried'].add(i)
        sib = [m for m in need if m != i]
        aid, t0 = f'{t}_{i}_{a}_s{k}_t', time.time()
        accs = []   # per acceptance in the tail: [its test output, its outputs on the siblings]

        def on_acc(g):
            accs.append([g, None])

        def on_sh(grids):
            if accs:
                accs[-1][1] = grids

        running[aid] = t0
        active[(a, k, t)] += 1
        base = dict(aid=aid, arm=a, sample=k, task=t, test=i, kind=KIND[t], family=FAMILY[t], tail=True, siblings=sib,
                    cls=out_cls(a, k, t, i), start_s=round(t0 - t_round))
        try:
            rec = await night_attempt(http, task, task['test'][i]['input'], HARNESS_TOKENS, live=f'{out}/live/{aid}.jsonl',
                                      on_accept=on_acc, on_best=None, siblings=[task['test'][m]['input'] for m in sib],
                                      on_shared=on_sh if sib else None, arm=arm_of(a))
        except asyncio.CancelledError:
            write_tail(dict(base, unfinished=True, seconds=round(time.time() - t0)))
            raise
        except Exception as e:  # noqa: BLE001 - a failed tail leaves the task's answers as they were
            rec = dict(error=f'{type(e).__name__}: {e}'[:300], requests=[], submits=[])
        finally:
            running.pop(aid, None)
            active[(a, k, t)] -= 1
        texts = rec.pop('texts', None)
        tv = bool(rec.get('verified'))
        alt = rec.get('second') if tv and rec.get('alternative') else None   # a second accepted program's answer

        def sib_grids(g):   # the siblings' outputs of the accepted program whose test output is g
            return next((s for x, s in accs if x == g and s), None) if g is not None else None

        ga, gs = sib_grids(rec.get('answer') if tv else None), sib_grids(alt)
        placed = []
        for n, m in enumerate([i] + sib):
            key = (a, k, t, m)
            if n == 0:
                ta, ts, tvm = rec.get('answer'), alt, tv
            else:
                ta, ts = (ga[n - 1] if ga else None), (gs[n - 1] if gs else None)
                tvm = ta is not None and kaggle_grid(ta)   # sibling grids come only from accepted programs
            cur = list(answers.get(key) or [None, None])
            first_pass.setdefault(key, cur)
            new, rule = tail_place(out_cls(a, k, t, m), cur, cands.get(key, {}).get('best'), ta, ts, tvm)
            answers[key] = new
            if tvm:
                st['settled'].add(m)
            sol = solutions[t][m]
            placed.append(dict(test=m, cls=out_cls(a, k, t, m), rule=rule, changed=new != cur, pair_before=sol in cur,
                               pair_after=sol in new, solved_before=cur[0] == sol, solved_after=new[0] == sol))
        if any(m not in st['settled'] and m not in st['tried'] for m in need):
            tails.append(j)   # an output no accepted tail program answered and not yet a tail's own test
        rec.update(base, seconds=round(time.time() - t0), placed=placed, answers=answers.get((a, k, t, i)),
                   tokens=sum(q['tokens'] for q in rec.get('requests', [])),
                   prefill_new=sum(q['prompt'] - q['cached'] for q in rec.get('requests', [])),
                   n_submits=len(rec.get('submits', [])))
        write_tail(rec, texts)
        shown = ('test', 'rule', 'changed') + (('pair_before', 'pair_after') if KIND[t] == 'parent' else ())
        print(time.strftime('%H:%M:%S'), 'tail', json.dumps({k2: rec.get(k2) for k2 in (
            'aid', 'kind', 'cls', 'siblings', 'verified', 'alternative', 'source', 'tokens', 'gestalt_tokens', 'turns',
            'spent', 'error')}), json.dumps([{k2: p[k2] for k2 in shown} for p in placed]), flush=True)

    async def status(every=300):
        while True:
            await asyncio.sleep(every)
            n_tail = sum(bool(r.get('tail')) for r in done)
            head = f"{time.strftime('%H:%M:%S')} {name}: {len(done) - n_tail} attempts done, {len(running)} running, " \
                   f"{len(queue)} waiting, {len(shared)} answered by their task, tails {n_tail} done / " \
                   f"{len(tails)} waiting, {(time.time() - t_round) / 60:.0f} min in"
            for a in arms:
                keys = [key for key in answers if key[0] == a]
                head += f" | {a}: {sum(solutions[key[2]][key[3]] in answers[key] for key in keys)}/{len(keys)} answered right"
            print(head + (f' | KV {GATE.usage:.0%}, parks {GATE.parks}' if GATE else ''), flush=True)

    async def worker(http):
        while time.time() < stop:
            if not (queue or tails or nfirst[0]):   # nothing waits, and no running first attempt can queue a tail
                return
            if not has_job():   # checked first: the gate counts each admission as a share of the pool
                await asyncio.sleep(ROUND_POLL_S)
                continue
            if GATE is not None:
                await GATE.admit_attempt()
            j = next_job()
            if j is None:
                continue
            if j[-1] == 'tail':
                await run_tail(http, j)
                continue
            nfirst[0] += 1
            try:
                await run_one(http, j)
            finally:
                nfirst[0] -= 1

    st = asyncio.create_task(status())
    async with aiohttp.ClientSession(timeout=aiohttp.ClientTimeout(total=None, sock_read=3600)) as http:
        async with aiohttp.ClientSession(timeout=aiohttp.ClientTimeout(total=None, sock_read=None)) as http_long:
            gate = asyncio.create_task(GATE.poll(http)) if GATE is not None else None
            workers = [asyncio.create_task(worker(http_long)) for _ in range(slots)]
            try:
                await asyncio.wait(workers, timeout=max(1, stop - time.time()))
            finally:
                st.cancel()
                if gate is not None:
                    gate.cancel()
                for w in workers:
                    w.cancel()
                await asyncio.gather(*workers, return_exceptions=True)
    # one row per output and replica: what it answered, and whether that is right
    rows = []
    for a, k in reps:
        for t in tasks:
            for i in range(ntest[t]):
                key = (a, k, t, i)
                ans = answers.get(key) or [None, None]
                fp = first_pass.get(key, ans)   # the pair before any tail changed it
                sol = solutions[t][i]
                rows.append(dict(arm=a, sample=k, task=t, test=i, kind=KIND[t], family=FAMILY[t], answers=ans,
                                 solved=ans[0] == sol, pair_solved=sol in ans, shared_by=shared.get(key),
                                 attempted=any(r['aid'] == f'{t}_{i}_{a}_s{k}' for r in done),
                                 tail_cls=out_cls(a, k, t, i), answers_first_pass=fp, solved_first_pass=fp[0] == sol,
                                 pair_first_pass=sol in fp))
    with open(f'{out}/outputs.jsonl', 'w') as f:
        f.write(''.join(json.dumps(r) + '\n' for r in rows))
    n_tail = sum(bool(r.get('tail')) for r in done)
    print(f'{name}: {len(done) - n_tail} attempts and {n_tail} tails in {(time.time() - t_round) / 60:.0f} min; '
          f'{len(queue)} outputs never started, {len(tails)} tail jobs never started')
    return rows, done


def night_summary(rows, done):
    for a in dict.fromkeys(r['arm'] for r in rows):
        for kind in ('parent', 'child', None):
            rs = [r for r in rows if r['arm'] == a and (kind is None or r['kind'] == kind)]
            at = [r for r in done if r.get('arm') == a and not r.get('tail') and (kind is None or r.get('kind') == kind)]
            fin = [r for r in at if not r.get('unfinished')]
            print(f"{a:5s} {kind or 'all':6s} outputs {len(rs)}: right by the pair {sum(r['pair_solved'] for r in rs)} "
                  f"({sum(r['pair_solved'] for r in rs) / max(1, len(rs)):.0%}), first answer {sum(r['solved'] for r in rs)}"
                  f" | attempts {len(at)}: accepted {sum(bool(r.get('verified')) for r in fin)}, spent "
                  f"{sum(bool(r.get('spent')) for r in fin)}, final turn {sum(bool(r.get('committed')) for r in fin)}, "
                  f"restarts {sum(r.get('restarts', 0) for r in fin)}, verify suspect "
                  f"{sum(r.get('verify') == 'suspect' for r in fin)}, mean tokens "
                  f"{sum(r.get('tokens', 0) for r in fin) / max(1, len(fin)):,.0f}")
            looks = [r['second_look'] for r in fin if r.get('second_look')]
            if looks:
                ran = [s for s in looks if 'tokens' in s]
                print(f"      second looks {len(looks)} (ran {len(ran)}): differs {sum(bool(s.get('differs')) for s in ran)}, "
                      f"verified {sum(bool(s.get('verified')) for s in ran)}, filled {sum(bool(s.get('filled')) for s in ran)}"
                      f" (+{sum(s.get('siblings_filled', 0) for s in ran)} siblings), mean tokens "
                      f"{sum(s['tokens'] for s in ran) / max(1, len(ran)):,.0f} | right only by attempt_2: "
                      f"{sum(r['pair_solved'] and not r['solved'] for r in rs)}")
            if arm_of(a).get('tail'):
                tr = [r for r in done if r.get('tail') and r.get('arm') == a and (kind is None or r.get('kind') == kind)]
                tf = [r for r in tr if not r.get('unfinished')]
                tok = lambda recs: sum((r.get('tokens') or 0) + (r.get('gestalt_tokens') or 0) for r in recs)
                cls = Counter(r.get('tail_cls') for r in rs if r.get('tail_cls'))
                fpp = [r.get('pair_first_pass', r['pair_solved']) for r in rs]
                fps = [r.get('solved_first_pass', r['solved']) for r in rs]
                print(f"      tail: outputs that need one {sum(cls.values())} {dict(cls)}; tail attempts {len(tr)} "
                      f"(unfinished {len(tr) - len(tf)}), accepted {sum(bool(r.get('verified')) for r in tf)}, tokens "
                      f"{tok(tf):,} = +{tok(tf) / max(1, tok(fin)):.0%} of the first pass's {tok(fin):,} | by the pair: "
                      f"first pass {sum(fpp)}, newly right {sum(r['pair_solved'] and not p for r, p in zip(rs, fpp))}, "
                      f"lost {sum(p and not r['pair_solved'] for r, p in zip(rs, fpp))} | first answer: first pass "
                      f"{sum(fps)}, newly right {sum(r['solved'] and not p for r, p in zip(rs, fps))}, lost "
                      f"{sum(p and not r['solved'] for r, p in zip(rs, fps))}")


async def night_fill(name, jobs, arm_name, slots=SLOTS, minutes=40):
    """Second looks only, on first answers recorded earlier in the night (08:04: a look on a wrong first answer thought
    49k tokens in one turn and left nothing). Each job is a dict with task, test, a1 (the recorded first answer), used
    (the tokens its attempt used) and an id; the look runs with `arm_name`'s switches and its grid is scored against
    the key. It never touches any round's answers: it measures how often a look rescues a wrong first answer."""
    out = f'{OUT}/{name}'
    for d in ('attempts', 'live'):
        os.makedirs(f'{out}/{d}', exist_ok=True)
    arm, queue, done, t0 = arm_of(arm_name), list(jobs), [], time.time()
    stop = t0 + minutes * 60
    print(f'{name}: {len(queue)} second looks ({arm_name}: {arm}), {slots} in flight, safety stop at '
          f'{time.strftime("%H:%M", time.gmtime(stop + 19800))} IST', flush=True)

    async def run_one(http, job):
        t, i = job['task'], job['test']
        task, rec, got = challenges[t], dict(requests=[{'tokens': job['used']}]), {}

        def fill(m, g):
            got.setdefault(m, g)
            return m == i

        ts = time.time()
        try:
            r2 = await night_second(http, task, i, rec, arm, job['a1'], f"{out}/live/{job['id']}.jsonl", fill)
            if r2 is not None:
                with open(f"{out}/attempts/{job['id']}.json", 'w') as f:
                    json.dump(r2, f)
        except asyncio.CancelledError:
            rec.setdefault('second_look', {})['cancelled'] = True
            raise
        except Exception as e:  # noqa: BLE001
            rec.setdefault('second_look', {})['error'] = f'{type(e).__name__}: {e}'[:300]
        finally:
            sol = solutions[t][i]
            row = dict({k: v for k, v in job.items() if k != 'a1'}, kind=KIND[t], look=rec.get('second_look'),
                       grid=got.get(i), right=got.get(i) == sol, a1_right=job['a1'] == sol,
                       seconds=round(time.time() - ts))
            done.append(row)
            with open(f'{out}/fills.jsonl', 'a') as f:
                f.write(json.dumps(row) + '\n')
            print(time.strftime('%H:%M:%S'), json.dumps({k: row.get(k) for k in ('id', 'kind', 'right', 'a1_right')}),
                  json.dumps({k: (row['look'] or {}).get(k) for k in ('tokens', 'turns', 'verified', 'differs', 'source',
                                                                      'error')}), flush=True)

    async def worker(http):
        while queue and time.time() < stop:
            await run_one(http, queue.pop(0))

    async with aiohttp.ClientSession(timeout=aiohttp.ClientTimeout(total=None, sock_read=None)) as http:
        workers = [asyncio.create_task(worker(http)) for _ in range(slots)]
        try:
            await asyncio.wait(workers, timeout=max(1, stop - time.time()))
        finally:
            for w in workers:
                w.cancel()
            await asyncio.gather(*workers, return_exceptions=True)
    for kind in ('parent', 'child', None):
        rs = [r for r in done if kind is None or r['kind'] == kind]
        ran = [r for r in rs if (r['look'] or {}).get('tokens')]
        print(f"{name} {kind or 'all':6s}: looks {len(rs)} (ran {len(ran)}), differs "
              f"{sum(bool(r['look'].get('differs')) for r in ran)}, verified {sum(bool(r['look'].get('verified')) for r in ran)}, "
              f"RIGHT {sum(r['right'] for r in rs)} of {len(rs)} wrong first answers, mean tokens "
              f"{sum(r['look']['tokens'] for r in ran) / max(1, len(ran)):,.0f}, mean turns "
              f"{sum(r['look'].get('turns') or 0 for r in ran) / max(1, len(ran)):.1f}", flush=True)
    print(f'{name}: {len(done)} looks in {(time.time() - t0) / 60:.0f} min; {len(queue)} never started')
    return done

### The 10-22 combined round: parents #31–#40 and their children, on one Modal 4 × L4 (written 2026-10-22 11:55 IST, before it runs)

The user, 2026-10-22 11:3x IST: "build the #31-40 set and the combined harness, then launch on one Modal 4xL4". Two
arms, one sample each, interleaved on one server, on set `v2b2` (cell `60d55098`: 30 tasks, 48 outputs):
- **`s10`**: submission 10's harness as it is (the control);
- **`g1+tail`**: g1's picture-only first look, plus **the tail**: once no first attempt is waiting, each task whose
  first pass left an output without an accepted program, or accepted it only after ≥ 70k tokens, gets one more
  independent attempt (a new look, a full budget, not shown the first answer). An accepted tail answer becomes
  attempt_1 of an unaccepted output (the old one moves to attempt_2); for a late acceptance it becomes attempt_2.
  Tail attempts go to `tail_records.jsonl`; `outputs.jsonl` keeps each output's first pass (`answers_first_pass`).

**The server** is submission 10's (`configs/l4x4_sub10_picks_harness.toml`, its launch line flag for flag) plus four
SGLang 0.5.22 picks: `LATCH_FIX`, `SKIP_FINISHING_DECODE`, `GENERATE_CONTRACT` (an unknown sampling key is refused
with HTTP 400) and `CHAT_FASTPATH`. The next cell checks the server's switches, the contract, and one picture request
exactly as a g1 look sends it, and stops the round if any of them fails.

**Prediction** (`runs/plans/1022_v2b2_combined/PREDICTION.md`): s10 20–30 of 48 by the pair; g1 −3 to +5 against
it on the first pass; the tail +1 to +5 outputs for +40–90 % of the arm's tokens, losing 0–1; no HTTP 400 from the
harness; ~3.3–3.8 h.

In [ ]:
# Before the 10-22 combined round (fail-closed): the server runs submission 10 plus the 0.5.22 picks, and with
# SGLANG_AGI2_GENERATE_CONTRACT a /generate body whose sampling_params hold an unknown key is refused with HTTP 400.
# (1) the server process carries the switches this round depends on; (2) the contract is on: a bogus key gets 400;
# (3) the harness's own picture request is answered: a g1 look's first turn (ruled pictures as data URIs, a streaming
# session, submission 10's sampling keys, `image_data`), cut at its cap, then the forced turn in the same session.
import subprocess
import aiohttp

PROBE_SWITCHES = {'SGLANG_AGI2_LATCH_FIX': '1', 'SGLANG_AGI2_SKIP_FINISHING_DECODE': '1',
                  'SGLANG_AGI2_GENERATE_CONTRACT': '1', 'SGLANG_AGI2_CHAT_FASTPATH': '1', 'SGLANG_AGI2_KV_V4': 'int4h',
                  'SGLANG_AGI2_HOST_EMBED': '1', 'SGLANG_AGI2_VERIFY_TAIL': '1', 'SGLANG_ENABLE_DFLASH_LENGTH_CAP': '1',
                  'SGLANG_ENABLE_DFLASH_STOP_CAP': '1', 'SGLANG_ENABLE_STREAMING_SESSION_FOLDED_CLAIM': '1',
                  'SGLANG_ENABLE_STREAMING_SESSION_PARK': '1', 'SGLANG_MM_ENCODE_MAX_TOKENS': '4096'}
_pids = subprocess.run(['pgrep', '-f', 'sglang.launch_server'], capture_output=True, text=True).stdout.split()
_env = None
for _p in _pids:
    try:
        _env = dict(kv.split('=', 1) for kv in open(f'/proc/{_p}/environ').read().split('\0') if '=' in kv)
        break
    except OSError:
        continue
if _env is None:
    print(f'(1) WARNING: no readable environ among the server processes {_pids}: switches not checked here')
else:
    _bad = {k: (v, _env.get(k)) for k, v in PROBE_SWITCHES.items() if _env.get(k) != v}
    check(not _bad, f'(1) the server process lacks switches (want, has): {_bad}')
    print(f'(1) the server process (pid {_p}) has all {len(PROBE_SWITCHES)} switches')


async def picks_probe():
    async with aiohttp.ClientSession(timeout=aiohttp.ClientTimeout(total=600)) as http:
        async with http.post(f'{SERVER}/generate', json={'input_ids': ids('Hello'), 'sampling_params': {
                'max_new_tokens': 1, 'temperature': 0.0, 'bogus_key': 1}}) as r:
            status, body = r.status, (await r.text())[:300]
        check(status == 400, f'(2) GENERATE_CONTRACT is not on: an unknown sampling key gave {status}: {body}')
        print(f'(2) the contract is on: an unknown sampling key gets {status} ({body[:120]})')
        t = challenges[PARENTS[0]]   # a parent's first pair and test input, as night_gestalt draws them
        pics = [picture(t['train'][0]['input'], t['train'][0]['output'], style='c32r'), picture(t['test'][0]['input'])]
        conv = SessionConv(http, GESTALT_TASK + f'Example 0: {IMG}\nTest input: {IMG}', pics)
        try:   # as night_gestalt: a thinking turn cut at its cap, then the forced turn with thinking off
            g1 = await conv.turn(64)
            conv.user('Write your description now, in at most 300 words, from what you have seen.')
            g2 = await conv.turn(32, thinking=False)
        finally:
            with contextlib.suppress(Exception):
                await asyncio.wait_for(conv.close(), 30)
        vis = sum(n for _, n in pics)
        check(g1['prompt'] >= vis and g1['tokens'] > 0 and g2['tokens'] > 0,
              f"(3) the picture turn: prompt {g1['prompt']} tokens for {vis} visual tokens, outputs {g1['tokens']}, "
              f"{g2['tokens']}")
        print(f"(3) picture request answered: prompt {g1['prompt']:,} tokens ({vis:,} visual), {g1['tokens']} out "
              f"({g1['finish']}); follow-up turn {g2['tokens']} out ({g2['finish']}): {g1['content'][:100]!r}")


await picks_probe()
PICKS_PROBE_OK = True

In [ ]:
# Before a g1 round on Flash-Next (G4, fail-closed): the harness's own picture request is answered inside a
# streaming session. A g1 look's first turn (ruled c32r pictures as data URIs, submission 10's sampling keys,
# `image_data`), cut at a small cap, then the forced turn with thinking off in the same session. This is part (3) of
# b37b0eb6 without its parts (1)-(2), which check the 0.5.22 picks build this server does not have. A look that
# fails gives no text and leaves the attempt as s10's, so a broken picture path must stop the round here.
import asyncio
import contextlib

import aiohttp


async def fn_picture_probe():
    async with aiohttp.ClientSession(timeout=aiohttp.ClientTimeout(total=600)) as http:
        t = challenges[PARENTS[0]]   # a parent's first pair and test input, as night_gestalt draws them
        pics = [picture(t['train'][0]['input'], t['train'][0]['output'], style='c32r'), picture(t['test'][0]['input'])]
        conv = SessionConv(http, GESTALT_TASK + f'Example 0: {IMG}\nTest input: {IMG}', pics)
        try:   # as night_gestalt: a thinking turn cut at its cap, then the forced turn with thinking off
            g1 = await conv.turn(256)
            conv.user('Write your description now, in at most 300 words, from what you have seen.')
            g2 = await conv.turn(256, thinking=False)
        finally:
            with contextlib.suppress(Exception):
                await asyncio.wait_for(conv.close(), 30)
        vis = sum(n for _, n in pics)
        if not (g1['prompt'] >= vis and g1['tokens'] > 0 and g2['tokens'] > 0):   # fail closed: no round runs
            raise RuntimeError(f"the picture turn: prompt {g1['prompt']} tokens for {vis} visual tokens, outputs "
                               f"{g1['tokens']}, {g2['tokens']} ({g1.get('finish')}, {g2.get('finish')})")
        print(f"picture request answered in a session: prompt {g1['prompt']:,} tokens ({vis:,} visual), "
              f"{g1['tokens']} out ({g1['finish']}); forced turn {g2['tokens']} out ({g2['finish']})")
        print(f"forced turn's text: {g2['content'][:400]!r}")


await fn_picture_probe()
FN_PICTURE_PROBE_OK = True

In [ ]:
# A round under its proportional stop that keeps the GPU busy to the end (G4 / Flash-Next work; additive: nothing
# above changes, and the round cell e03c3ecd runs night_round_sched in place of night_round only with FN_SCHED=1).
# The user, 2026-10-22: the objective is "effectively using gpu, as minimum bubbles as possible, so these things would
# on a larger run also scale dynamically automatically on tasks you havent seen" (~21:55 IST), and "lets do 1 3 4 but
# for 21-30 batch to test whether assumptions hold or not" (~22:00). So every decision below comes from what the round
# measures as it runs (the queue, what is running, the tokens generated, the clock, an answer's verification); the
# stop (342 min per 120 tasks) is the one input. No token thresholds and no right/wrong rates from any earlier set.
#   1. Work-conserving. First attempts always come first. When no first attempt can start (none left, or each waits
#      behind its own task's running attempt), a free slot takes an independent extra attempt (the arm's switches, a
#      full budget, not shown any earlier answer) on the output with the least verified evidence: no accepted program,
#      then one attempt's accepted program; an output on which two independent attempts' programs agree is done. Ties:
#      fewer attempts so far, then the test file's order. One attempt per task replica at a time.
#   3. Pace. From the round's own generation rate (last 10 min) and a Kaplan-Meier estimate of the tokens the first
#      pass still needs, project its end. If it falls past the wrap-up point, an accepted program ends its attempt
#      without the review and no extra attempt starts, until the projection has slack again (a tenth of the round).
#   4. Deadline. The wrap-up point is the stop minus the time a final answer turn needs at the measured per-request
#      rate. From there nothing starts; working turns are capped to end by it; an attempt without an accepted program
#      gives its final answer, thinking off, sized to the time left. Every answer is placed when it appears (accepted
#      programs at acceptance, a best failing submission when it is made), so the stop never loses one.
# Answers: an output's attempt_1 / attempt_2 are its verified grids ranked by how many independent attempts' programs
# gave each (ties: the earliest), then unverified grids (each attempt's final answer, then its best failing
# submission, attempts in start order). With one attempt this is submission 10's set_answers.
# Bubble account: sched_trace.jsonl, one line per tick (what runs, what waits, free slots, the clock's state).
import collections

FN_SCHED = os.environ.get('FN_SCHED') == '1'
FN_TIERS = os.environ.get('FN_TIERS') == '1'   # 2026-10-24: extras by tier (pick_extra), off by default
# 2026-10-25 (decode-3x baseline): first attempts only. No extra attempt starts, and a worker ends when no first
# attempt is left to start, so the round ends with its last first attempt instead of at the stop. Off by default.
FN_NO_EXTRAS = os.environ.get('FN_NO_EXTRAS') == '1'
if FN_NO_EXTRAS:
    print('FN_NO_EXTRAS: first attempts only; the round ends when the last one ends', flush=True)
if os.environ.get('FN_TEMP'):   # 2026-10-25: the round's sampling temperature (sub 10's SAMPLING is 1.0); unset: unchanged
    SAMPLING['temperature'] = float(os.environ['FN_TEMP'])
    print('FN_TEMP: every request of this round samples at temperature', SAMPLING['temperature'], flush=True)
SCHED_TICK_S = 30.0          # how often the clock measures
SCHED_WINDOW_S = 600.0       # the window of the measured rates
SCHED_FINAL_TOKENS = 2048    # the final answer turn the wrap-up leaves time for (thinking off)
SCHED_MARGIN_S = 30.0        # the wrap-up's allowance on top of that turn (its prefill, the sandbox call it may make)
SCHED_MIN_WRAP_S = 60.0      # the shortest wrap-up
COMMIT_WHY = globals()['COMMIT_WHY']   # submission 10's (its cell a5509211, run by e8c03140)
COMMIT_WHY.setdefault('deadline', 'The time for this puzzle is up.')

_SCHED_PATCHES = [
    ('async def night_attempt(http,', 'async def sched_attempt(http,'),
    ('siblings=(), on_shared=None, arm=None):', 'siblings=(), on_shared=None, arm=None, clock=None):'),
    ("                if left <= 512:\n                    why = 'spent'\n                    break\n",
     "                if left <= 512:\n                    why = 'spent'\n                    break\n"
     "                if clock is not None and clock.wrap_now():   # the round's wrap-up: the final answer turn now\n"
     "                    why = 'deadline'\n                    break\n"),
    ("                g = await conv.turn(min(left, room, arm.get('turn_cap') or left))\n",
     "                g = await conv.turn(min(left, room, arm.get('turn_cap') or left,\n"
     "                                        clock.turn_cap(left) if clock is not None else left))\n"),
    ("                left -= g['tokens']\n",
     "                left -= g['tokens']\n"
     "                if clock is not None:\n"
     "                    clock.progress(live, budget - left)\n"),
    ("                                    pending, rec['first_accept'] = dict(sub, code=code), sub['test']\n",
     "                                    pending, rec['first_accept'] = dict(sub, code=code), sub['test']\n"
     "                                    if clock is not None and clock.frugal:   # behind the pace: no review\n"
     "                                        accepted, rec['frugal'] = sub, True\n"),
    ("                g = await conv.turn(max(256, min(8192, CONTEXT - ctx - 512)), thinking=False)\n",
     "                g = await conv.turn(max(256, min(8192, CONTEXT - ctx - 512,\n"
     "                                                 clock.final_cap() if clock is not None else 8192)),\n"
     "                                    thinking=False)\n"),
]


def sched_source(src):
    for old, new in _SCHED_PATCHES:
        check(src.count(old) == 1, f'sched patch does not match exactly once ({src.count(old)}): {old[:80]!r}')
        src = src.replace(old, new)
    return src


_SCHED_SRC = sched_source(_NIGHT_SRC)
sched_attempt = None   # bound by the exec below
exec(compile(_SCHED_SRC, '<sched_attempt: the night attempt with the round clock>', 'exec'), globals())
check(callable(sched_attempt), 'sched_attempt was not built')
print('sched_attempt built:', len(_SCHED_SRC) - len(_NIGHT_SRC), 'characters of patches over the night attempt')


def km_curve(events, censored):
    """Kaplan-Meier survival of attempt lengths: [(x, S just after x)], from finished lengths (events) and the lengths so
    far of running attempts (censored)."""
    pts = sorted([(x, 1) for x in events] + [(x, 0) for x in censored])
    curve, s, at_risk, n = [(0.0, 1.0)], 1.0, len(pts), 0
    while n < len(pts):
        x, d, c = pts[n][0], 0, 0
        while n < len(pts) and pts[n][0] == x:
            d += pts[n][1]
            c += 1 - pts[n][1]
            n += 1
        if d and at_risk:
            s *= 1 - d / at_risk
            curve.append((float(x), s))
        at_risk -= d + c
    return curve


def km_rest(curve, c, cap):
    """E[T - c | T > c] under the curve, with S taken as 0 from `cap` on (no attempt outlasts its budget)."""
    def S(x):
        v = 1.0
        for xi, si in curve:
            if xi <= x:
                v = si
            else:
                break
        return v
    sc = S(c)
    if sc <= 0 or c >= cap:
        return 0.0
    area, x = 0.0, c
    for xi, si in curve:
        if xi <= c:
            continue
        if xi >= cap:
            break
        area += S(x) * (xi - x)
        x = xi
    area += S(x) * (cap - x)
    return area / sc


class RoundClock:
    """The round's clock: the stop, the rates the round measures, the wrap-up point and the pace projection."""

    def __init__(self, t0, stop, budget, slots, every=None, window=None, final_tokens=None):
        self.t0, self.stop, self.budget, self.slots = t0, stop, budget, slots
        self.every = SCHED_TICK_S if every is None else every
        self.window = SCHED_WINDOW_S if window is None else window
        self.final_tokens = SCHED_FINAL_TOKENS if final_tokens is None else final_tokens
        self.used = {}             # attempt (its live file) -> tokens generated so far, its look included
        self.running = {}          # attempt -> 'first' | 'extra'
        self.finished_first = []   # finished first attempts' tokens (look included)
        self.samples = collections.deque()   # (time, tokens generated by the round so far, attempts running)
        self.rate = self.per_req = self.projected_end = None
        self.frugal = self.wrapping = False
        self.frugal_s = 0.0
        self.unstarted_first = 0

    def progress(self, key, used):
        self.used[key] = used

    def wrap_at(self, now=None):
        span = self.stop - self.t0
        if self.per_req:
            need = 1.5 * self.final_tokens / self.per_req + SCHED_MARGIN_S
        else:
            need = 0.05 * span   # only until the first rate is measured (a minute or so in)
        return self.stop - min(max(need, SCHED_MIN_WRAP_S), 0.15 * span)

    def wrap_now(self):
        now = time.time()
        if not self.wrapping:
            w = self.wrap_at(now)
            if now >= w or (self.per_req and self.per_req * (w - now) < 512):
                self.wrapping = True
        return self.wrapping

    def turn_cap(self, left):
        if not self.per_req:
            return left
        return max(256, min(left, int(self.per_req * (self.wrap_at() - time.time()))))

    def final_cap(self):
        if not self.per_req:
            return 8192
        return max(256, int(self.per_req * (self.stop - time.time() - 15)))

    def extras_ok(self):
        return not self.frugal and not self.wrap_now()

    def tick(self, unstarted_first):
        now = time.time()
        self.unstarted_first = unstarted_first
        self.samples.append((now, sum(self.used.values()), len(self.running)))
        while self.samples and now - self.samples[0][0] > self.window:
            self.samples.popleft()
        (ta, ga, _), (tb, gb, _) = self.samples[0], self.samples[-1]
        if tb - ta >= self.window / 10:
            self.rate = max(0.0, (gb - ga) / (tb - ta))
            runs = sum(n for *_, n in self.samples) / len(self.samples)
            self.per_req = self.rate / runs if runs > 0 and self.rate > 0 else None
        # the pace: what the first pass still needs, at the share of the rate its attempts get
        first = [k for k, v in self.running.items() if v == 'first']
        if len(self.finished_first) >= 5 and self.rate:
            cap = self.budget + 8192
            curve = km_curve(self.finished_first, [self.used.get(k, 0) for k in first])
            need = unstarted_first * km_rest(curve, 0.0, cap) + sum(km_rest(curve, self.used.get(k, 0), cap)
                                                                    for k in first)
            share = len(first) / max(1, len(self.running))
            self.projected_end = now + need / max(self.rate * max(share, 1 / self.slots), 1e-6)
            span, w = self.stop - self.t0, self.wrap_at(now)
            was = self.frugal
            if not self.frugal and self.projected_end > w:
                self.frugal = True
            elif self.frugal and self.projected_end < w - 0.10 * span:
                self.frugal = False
            if was:
                self.frugal_s += self.every
        self.wrap_now()


async def night_round_sched(name, arms, samples=(0,), minutes=ROUND_MINUTES, slots=SLOTS):
    """night_round's first pass with mechanisms 1, 3 and 4 (above). Writes records.jsonl (first attempts),
    extra_records.jsonl, outputs.jsonl (the answers at the stop, the first pass's pair) and sched_trace.jsonl."""
    out = f'{OUT}/{name}'
    for d in ('attempts', 'live'):
        os.makedirs(f'{out}/{d}', exist_ok=True)
    reps = [(a, k) for k in samples for a in arms]
    tasks = list(dict.fromkeys(t for t, _ in OUTPUTS))
    ntest = {t: len(challenges[t]['test']) for t in tasks}
    order = [(t, i) for t in tasks for i in range(ntest[t])]   # the test file's order
    queue = [(a, k, t, 0) for t in tasks for a, k in reps] + \
            [(a, k, t, i) for t in tasks for i in range(1, ntest[t]) for a, k in reps]
    answers, shared, active, done, running = {}, {}, Counter(), [], {}
    ev = {}               # output -> its evidence (verified grids by attempt, unverified grids by attempt, tries)
    first_done = set()    # outputs whose first attempt (their own, or their task's that answered them) has ended
    first_pair = {}       # output -> its pair when an extra attempt first touched it
    busy_task = set()     # task replicas an extra attempt holds or reserved
    starts = []           # attempt start order (for the unverified ranking)
    t_round = time.time()
    stop = t_round + minutes * 60
    clock = RoundClock(t_round, stop, HARNESS_TOKENS, slots)
    print(f'{name}: arms {arms}, samples {list(samples)}: {len(queue)} outputs to answer, {slots} in flight, '
          f'stop at {time.strftime("%H:%M", time.gmtime(stop + 19800))} IST ({minutes:.1f} min, proportional); '
          f'mechanisms: work-conserving extra attempts, pace, deadline wrap-up', flush=True)

    def E(key):
        return ev.setdefault(key, dict(support={}, seen=[], unverified={}, tries=0))

    def place(key):
        e = E(key)
        ver = sorted(e['seen'], key=lambda s: (-len(e['support'][s]), e['seen'].index(s)))
        grids = [json.loads(s) for s in ver]
        for aid in [a for a in starts if a in e['unverified']]:
            for g in e['unverified'][aid]:
                if g is not None and kaggle_grid(g) and g not in grids:
                    grids.append(g)
        answers[key] = (grids + [None, None])[:2]

    def verified(key, g, aid):
        if g is None or not kaggle_grid(g):
            return
        e, s = E(key), json.dumps(g)
        if s not in e['support']:
            e['support'][s] = set()
            e['seen'].append(s)
        e['support'][s].add(aid)
        place(key)

    def unverified(key, aid, grids):
        E(key)['unverified'][aid] = [g for g in grids if g is not None]
        place(key)

    def support(key):
        return max((len(v) for v in E(key)['support'].values()), default=0)

    def write(path, rec, texts=None):
        done.append(rec)
        with open(f'{out}/{path}', 'a') as f:
            f.write(json.dumps(rec) + '\n')
        if texts is not None:
            with open(f"{out}/attempts/{rec['aid']}.json", 'w') as f:
                json.dump(dict(rec, texts=texts), f)

    def startable_first():
        queue[:] = [j for j in queue if j not in shared]
        return next((n for n, j in enumerate(queue) if not active[j[:3]] and j[:3] not in busy_task), None)

    acc_seen = {}         # FN_TIERS: attempt id -> generated tokens at its first accepted submit (None: none)
    pick_info = {}        # FN_TIERS: output -> its tier and acceptance tokens when an extra was picked for it

    def acc_tokens(aid):
        """FN_TIERS: the generated tokens at an ended attempt's first accepted submit, from its live log: the sum of
        its turns' tokens up to and including that turn (the picture look and the final answer turn excluded)."""
        if aid not in acc_seen:
            used, at = 0, None
            try:
                with open(f'{out}/live/{aid}.jsonl') as f:
                    for line in f:
                        t = json.loads(line)
                        if t.get('turn') in ('final', 'commit'):
                            continue
                        used += t.get('tokens') or 0
                        if any(c.get('name') == 'submit' and str(c.get('out', '')).startswith('Accepted:')
                               for c in t.get('calls') or []):
                            at = used
                            break
            except (OSError, ValueError):
                pass
            acc_seen[aid] = at
        return acc_seen[aid]

    def tier_of(key):
        """FN_TIERS: (tier, tokens) of an output with one accepted program. 1 doubtful: accepted after 96k tokens, or
        a different accepted grid exists; 2 plausible: 32-96k; 3 confident: within 32k and no other grid (2026-10-24,
        docs/reports/1024_1238_g4_vs_l4_gap/accept_signals.py: 141 of 142 confident first programs right, 117 of 138
        plausible, 23 of 54 doubtful). Unknown tokens count as plausible."""
        e = E(key)
        first = e['seen'][0]
        aids = sorted(e['support'][first], key=lambda a: starts.index(a) if a in starts else len(starts))
        acc = acc_tokens(aids[0]) if aids else None
        if len(e['seen']) >= 2 or (acc is not None and acc > 98304):
            return 1, acc or 0
        if acc is None:
            return 2, 0
        return (3 if acc <= 32768 else 2), acc

    def pick_extra():
        """The output an extra attempt should take now (and its task replica reserved), or None."""
        if FN_NO_EXTRAS or not clock.extras_ok():
            return None
        best = None
        for idx, (t, i) in enumerate(order):
            for a, k in reps:
                key = (a, k, t, i)
                if key not in first_done or active[(a, k, t)] or (a, k, t) in busy_task:
                    continue
                top = support(key)
                if top >= 2:
                    continue
                if FN_TIERS and top == 1:
                    tr, acc = tier_of(key)
                    pr = (top, tr, -acc, E(key)['tries'], idx)
                else:
                    tr, acc = 0, 0
                    pr = (top, 0, 0, E(key)['tries'], idx)
                if best is None or pr < best[0]:
                    best = (pr, key, tr, acc)
        if best is None:
            return None
        busy_task.add(best[1][:3])
        if FN_TIERS:
            pick_info[best[1]] = dict(tier=best[2], acc_tokens=best[3])
        return best[1]

    async def run_one(http, j):
        a, k, t, i = j
        aid, task, t0 = f'{t}_{i}_{a}_s{k}', challenges[t], time.time()
        live = f'{out}/live/{aid}.jsonl'
        sib = [(a, k, t, m) for m in range(ntest[t]) if m != i and not E((a, k, t, m))['support']]
        starts.append(aid)

        def on_shared(grids):
            for s, g in zip(sib, grids):
                if g is not None and kaggle_grid(g):
                    shared.setdefault(s, aid)
                    verified(s, g, aid)

        running[aid], clock.running[live] = t0, 'first'
        active[(a, k, t)] += 1
        base = dict(aid=aid, arm=a, sample=k, task=t, test=i, kind=KIND[t], family=FAMILY[t], start_s=round(t0 - t_round))
        try:
            rec = await sched_attempt(http, task, task['test'][i]['input'], HARNESS_TOKENS, live=live,
                                      on_accept=lambda g: verified(j, g, aid),
                                      on_best=lambda g: unverified(j, aid, [g]),
                                      siblings=[task['test'][m]['input'] for *_, m in sib], on_shared=on_shared,
                                      arm=arm_of(a), clock=clock)
        except asyncio.CancelledError:
            write('records.jsonl', dict(base, unfinished=True, seconds=round(time.time() - t0)))
            raise
        except Exception as e:  # noqa: BLE001 - one failed attempt must not stop the round
            rec = dict(error=f'{type(e).__name__}: {e}'[:300], requests=[], submits=[])
        finally:
            running.pop(aid, None)
            clock.running.pop(live, None)
            active[(a, k, t)] -= 1
        texts = rec.pop('texts', None)
        unverified(j, aid, [rec.get('answer'), rec.get('second')])   # verified grids among them rank as verified
        first_done.add(j)
        first_done.update(s for s in sib if s in shared)
        sol = solutions[t][i]
        tok = sum(q['tokens'] for q in rec.get('requests', []))
        clock.finished_first.append(tok + (rec.get('gestalt_tokens') or 0))
        rec.update(base, seconds=round(time.time() - t0), answers=answers.get(j),
                   solved=(answers.get(j) or [None])[0] == sol, pair_solved=sol in (answers.get(j) or []),
                   tokens=tok, prefill_new=sum(q['prompt'] - q['cached'] for q in rec.get('requests', [])),
                   n_submits=len(rec.get('submits', [])), first_accept_solved=rec.get('first_accept') == sol
                   if rec.get('first_accept') else None)
        write('records.jsonl', rec, texts)
        print(time.strftime('%H:%M:%S'), json.dumps({k2: rec.get(k2) for k2 in (
            'aid', 'kind', 'solved', 'pair_solved', 'verified', 'source', 'tokens', 'turns', 'n_submits', 'spent',
            'committed', 'final_from', 'frugal', 'error') if KIND[t] == 'parent' or k2 not in ('solved', 'pair_solved')}),
              flush=True)

    async def run_extra(http, key):
        a, k, t, i = key
        task = challenges[t]
        e = E(key)
        e['tries'] += 1
        n = sum(1 for r in done if r.get('extra') and r['task'] == t and r['arm'] == a and r['sample'] == k) + 1
        aid, t0 = f'{t}_{i}_{a}_s{k}_x{n}', time.time()
        live = f'{out}/live/{aid}.jsonl'
        sib = [(a, k, t, m) for m in range(ntest[t]) if m != i]
        for kk in [key] + sib:
            first_pair.setdefault(kk, list(answers.get(kk) or [None, None]))
        before = {kk: list(answers.get(kk) or [None, None]) for kk in [key] + sib}
        starts.append(aid)

        def on_shared(grids):
            for s, g in zip(sib, grids):
                verified(s, g, aid)

        running[aid], clock.running[live] = t0, 'extra'
        active[(a, k, t)] += 1
        base = dict(aid=aid, arm=a, sample=k, task=t, test=i, kind=KIND[t], family=FAMILY[t], extra=True, tail=True,
                    siblings=[m for *_, m in sib], support_before=support(key), start_s=round(t0 - t_round),
                    **pick_info.pop(key, {}))
        try:
            rec = await sched_attempt(http, task, task['test'][i]['input'], HARNESS_TOKENS, live=live,
                                      on_accept=lambda g: verified(key, g, aid),
                                      on_best=lambda g: unverified(key, aid, [g]),
                                      siblings=[task['test'][m]['input'] for *_, m in sib],
                                      on_shared=on_shared if sib else None, arm=arm_of(a), clock=clock)
        except asyncio.CancelledError:
            write('extra_records.jsonl', dict(base, unfinished=True, seconds=round(time.time() - t0)))
            raise
        except Exception as e2:  # noqa: BLE001 - a failed extra attempt leaves the answers as they were
            rec = dict(error=f'{type(e2).__name__}: {e2}'[:300], requests=[], submits=[])
        finally:
            running.pop(aid, None)
            clock.running.pop(live, None)
            active[(a, k, t)] -= 1
            busy_task.discard((a, k, t))
        texts = rec.pop('texts', None)
        unverified(key, aid, [rec.get('answer'), rec.get('second')])
        placed = []
        for kk in [key] + sib:
            sol, now = solutions[kk[2]][kk[3]], list(answers.get(kk) or [None, None])
            placed.append(dict(test=kk[3], changed=now != before[kk], support_after=support(kk),
                               **({} if KIND[t] != 'parent' else dict(pair_before=sol in before[kk],
                                                                      pair_after=sol in now))))
        rec.update(base, seconds=round(time.time() - t0), placed=placed, answers=answers.get(key),
                   tokens=sum(q['tokens'] for q in rec.get('requests', [])),
                   prefill_new=sum(q['prompt'] - q['cached'] for q in rec.get('requests', [])),
                   n_submits=len(rec.get('submits', [])))
        write('extra_records.jsonl', rec, texts)
        print(time.strftime('%H:%M:%S'), 'extra', json.dumps({k2: rec.get(k2) for k2 in (
            'aid', 'kind', 'support_before', 'siblings', 'verified', 'source', 'tokens', 'gestalt_tokens', 'turns',
            'frugal', 'error')}), json.dumps(placed), flush=True)

    async def ticker(http):
        trace = open(f'{out}/sched_trace.jsonl', 'a')
        last_status = 0.0
        try:
            while True:
                # The pace counts only queued outputs that could start a first attempt now. A sibling output of a task
                # whose attempt is running is usually answered by that attempt, so it is not a new attempt (the
                # sibling-count fix, 2026-10-23; before it, round 1023_1917 projected the tail 8-17 min too late).
                clock.tick(sum(1 for j in queue if not active[j[:3]] and j[:3] not in busy_task))
                now = time.time()
                nf = sum(1 for v in clock.running.values() if v == 'first')
                nx = len(clock.running) - nf
                row = dict(t_min=round((now - t_round) / 60, 2), first=nf, extra=nx, free=slots - nf - nx,
                           queue=len(queue), startable=startable_first() is not None,
                           rate=round(clock.rate or 0, 1), per_req=round(clock.per_req or 0, 1),
                           wrap_min=round((clock.wrap_at() - t_round) / 60, 2), wrapping=clock.wrapping,
                           frugal=clock.frugal, projected_end_min=round((clock.projected_end - t_round) / 60, 2)
                           if clock.projected_end else None,
                           kv=round(GATE.usage, 3) if GATE else None, parked=GATE.parked if GATE else None)
                trace.write(json.dumps(row) + '\n')
                trace.flush()
                if now - last_status >= 300:
                    last_status = now
                    nx_done = sum(1 for r in done if r.get('extra'))
                    head = (f"{time.strftime('%H:%M:%S')} {name}: {len(done) - nx_done} attempts done, {nf} first + {nx} "
                            f"extra running, {len(queue)} waiting, {len(shared)} answered by their task, extras {nx_done} "
                            f"done, {(now - t_round) / 60:.0f} min in | rate {clock.rate or 0:,.0f} tok/s, wrap at "
                            f"{(clock.wrap_at() - t_round) / 60:.0f} min{', FRUGAL' if clock.frugal else ''}"
                            f"{', WRAPPING' if clock.wrapping else ''}")
                    for a in arms:
                        keys = [key for key in answers if key[0] == a]
                        head += (f" | {a}: {sum(solutions[key[2]][key[3]] in answers[key] for key in keys)}/"
                                 f"{len(keys)} answered right")
                    print(head + (f' | KV {GATE.usage:.0%}, parks {GATE.parks}' if GATE else ''), flush=True)
                await asyncio.sleep(clock.every)
        finally:
            trace.close()

    async def worker(http):
        while time.time() < stop and not clock.wrap_now():
            n = startable_first()
            if n is not None:
                if GATE is not None:
                    await GATE.admit_attempt()
                n = startable_first()   # another worker may have taken it while the gate held this one
                if n is None or clock.wrap_now():
                    continue
                await run_one(http, queue.pop(n))
                continue
            key = pick_extra()
            if key is None:
                if not queue and (FN_NO_EXTRAS or (not running and not any(
                        support((a, k, t, i)) < 2 for t, i in order for a, k in reps))):
                    return   # every output has two agreeing programs and nothing runs: nothing useful is left
                             # (FN_NO_EXTRAS: no first attempt is left to start; the running ones end on their own)
                await asyncio.sleep(ROUND_POLL_S)
                continue
            try:
                if GATE is not None:
                    await GATE.admit_attempt()
                if clock.wrap_now() or startable_first() is not None:   # first attempts take a freed slot first
                    continue
                await run_extra(http, key)
            finally:
                busy_task.discard(key[:3])

    async with aiohttp.ClientSession(timeout=aiohttp.ClientTimeout(total=None, sock_read=3600)) as http:
        async with aiohttp.ClientSession(timeout=aiohttp.ClientTimeout(total=None, sock_read=None)) as http_long:
            gate = asyncio.create_task(GATE.poll(http)) if GATE is not None else None
            tick = asyncio.create_task(ticker(http))
            workers = [asyncio.create_task(worker(http_long)) for _ in range(slots)]
            try:
                await asyncio.wait(workers, timeout=max(1, stop - time.time()))
            finally:
                tick.cancel()
                if gate is not None:
                    gate.cancel()
                for w in workers:
                    w.cancel()
                await asyncio.gather(*workers, tick, return_exceptions=True)
    rows = []
    for a, k in reps:
        for t in tasks:
            for i in range(ntest[t]):
                key = (a, k, t, i)
                ans = answers.get(key) or [None, None]
                fp = first_pair.get(key, ans)
                sol = solutions[t][i]
                rows.append(dict(arm=a, sample=k, task=t, test=i, kind=KIND[t], family=FAMILY[t], answers=ans,
                                 solved=ans[0] == sol, pair_solved=sol in ans, shared_by=shared.get(key),
                                 attempted=any(r['aid'] == f'{t}_{i}_{a}_s{k}' for r in done),
                                 support=support(key), extra_tries=E(key)['tries'], answers_first_pass=fp,
                                 solved_first_pass=fp[0] == sol, pair_first_pass=sol in fp))
    with open(f'{out}/outputs.jsonl', 'w') as f:
        f.write(''.join(json.dumps(r) + '\n' for r in rows))
    nx_done = sum(1 for r in done if r.get('extra'))
    print(f'{name}: {len(done) - nx_done} attempts and {nx_done} extra attempts in {(time.time() - t_round) / 60:.0f} '
          f'min; {len(queue)} outputs never started; frugal {clock.frugal_s / 60:.1f} min')
    for a in arms:
        for kind in ('parent', 'child', None):
            rs = [r for r in rows if r['arm'] == a and (kind is None or r['kind'] == kind)]
            print(f"  {a} {kind or 'all':6s}: by the pair at the stop {sum(r['pair_solved'] for r in rs)} of {len(rs)} "
                  f"(first pass {sum(r['pair_first_pass'] for r in rs)}, newly right "
                  f"{sum(r['pair_solved'] and not r['pair_first_pass'] for r in rs)}, lost "
                  f"{sum(r['pair_first_pass'] and not r['pair_solved'] for r in rs)})")
    return rows, done


if FN_SCHED:
    night_round = night_round_sched
    print('FN_SCHED: the round cell runs night_round_sched (work-conserving extra attempts, pace, deadline wrap-up)'
          + (', extras by tier (FN_TIERS)' if FN_TIERS else ''))

In [ ]:
# One night round: the arms named in NIGHT_ARMS (ARMS above), samples NIGHT_SAMPLES, batch NIGHT_BATCH (round.sh passes them)
NIGHT_ARMS = [a for a in os.environ.get('NIGHT_ARMS', 's10').split(',') if a]
NIGHT_SAMPLES = tuple(int(x) for x in os.environ.get('NIGHT_SAMPLES', '0').split(','))
NIGHT_BATCH = os.environ.get('NIGHT_BATCH', 'night_' + '_'.join(NIGHT_ARMS))
for _a in NIGHT_ARMS:   # an arm may combine parts with '+', e.g. s10+a2
    arm_of(_a)
night_rows, night_done = await night_round(NIGHT_BATCH, NIGHT_ARMS, NIGHT_SAMPLES)
night_summary(night_rows, night_done)

In [ ]:
# Second looks only, on first answers recorded earlier in the night: NIGHT_FILL_JOBS is a JSON list of jobs (task, test,
# a1, used, id), NIGHT_FILL_ARM the look's switches (default a2c: turns capped), NIGHT_FILL_MINUTES the safety stop
FILL_JOBS = json.load(open(os.environ['NIGHT_FILL_JOBS']))
FILL_ARM = os.environ.get('NIGHT_FILL_ARM', 'a2c')
arm_of(FILL_ARM)
check(all(j['task'] in challenges and 0 <= j['test'] < len(challenges[j['task']]['test']) for j in FILL_JOBS),
      'a fill job names an output outside this set')
night_fills = await night_fill(os.environ.get('NIGHT_BATCH', 'fill_' + FILL_ARM), FILL_JOBS, FILL_ARM,
                               minutes=float(os.environ.get('NIGHT_FILL_MINUTES', '30')))